In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/spikie/capstone-split/train_tokens.bin
/kaggle/input/datasets/spikie/capstone-split/train_manifest.json
/kaggle/input/datasets/spikie/capstone-split/metadata.json
/kaggle/input/datasets/spikie/capstone-split/legal_m1.py
/kaggle/input/datasets/spikie/capstone-split/legal_bpe_32k.pkl
/kaggle/input/datasets/spikie/capstone-split/valid_manifest.json
/kaggle/input/datasets/spikie/capstone-split/latest.pt
/kaggle/input/datasets/spikie/capstone-split/legal_bpe_64k.pkl
/kaggle/input/datasets/spikie/capstone-split/valid_tokens.bin


In [1]:
import os
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF_TOKEN")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)

print("Hugging Face login successful.")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Hugging Face login successful.


In [7]:
import os
import json
import random
from huggingface_hub import hf_hub_download

HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"

TRAIN_FILES = {
    "acts": "Phase_1_Pretraining/Train/train_acts.json",
    "high_courts": "Phase_1_Pretraining/Train/train_high_courts.json",
    "supreme_court": "Phase_1_Pretraining/Train/train_supreme_court.json",
    "tribunals": "Phase_1_Pretraining/Train/train_tribunals.json"
}

SEED = 42
random.seed(SEED)

source_paths = {}

for source_name,file_name in TRAIN_FILES.items():
    print("Downloading:",source_name)

    path = hf_hub_download(
        repo_id=HF_DATASET_REPO,
        filename=file_name,
        repo_type="dataset",
        token=HF_TOKEN
    )

    source_paths[source_name] = path

    print("Path:",path)
    print()
    
print("All four files downloaded/accessed successfully.")

Downloading: acts


Phase_1_Pretraining/Train/train_acts.jso(…): reconstructing file:   0%|          |  0.00B /  729MB            

Phase_1_Pretraining/Train/train_acts.jso(…): downloading bytes:           |  0.00B            

Path: /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_1_Pretraining/Train/train_acts.json

Downloading: high_courts


Phase_1_Pretraining/Train/train_high_cou(…): reconstructing file:   0%|          |  0.00B / 10.7GB            

Phase_1_Pretraining/Train/train_high_cou(…): downloading bytes:           |  0.00B            

... (3 outputs left out)

Path: /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_1_Pretraining/Train/train_supreme_court.json

Downloading: tribunals


Phase_1_Pretraining/Train/train_tribunal(…): reconstructing file:   0%|          |  0.00B / 6.48GB            

Phase_1_Pretraining/Train/train_tribunal(…): downloading bytes:           |  0.00B            

Path: /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_1_Pretraining/Train/train_tribunals.json

All four files downloaded/accessed successfully.


In [4]:
!pip -q install ijson

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 149.9/149.9 kB 4.9 MB/s eta 0:00:00


In [5]:
import os
import ijson
import hashlib
import math
from collections import Counter

print("="*70)
print("RAW LEGAL CORPUS - STREAMING QUALITY AUDIT")
print("="*70)

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------
MIN_CHARS=100
MIN_WORDS=20
REPETITION_SAMPLE_SIZE=2000

# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------
def stream_documents(path):
    with open(path,"rb") as f:
        for doc in ijson.items(f,"item"):
            yield doc

def text_hash(text):
    return hashlib.blake2b(text.encode("utf-8"),digest_size=8).digest()

# ------------------------------------------------------------
# Audit one source
# ------------------------------------------------------------
def audit_source(source,path):
    print("\n"+"-"*70)
    print("SOURCE:",source)
    print("FILE SIZE:",f"{os.path.getsize(path)/1024**3:.3f} GB")
    print("-"*70)

    docs=0
    empty=0
    short=0
    missing=0
    bad_word_count=0
    bad_char_count=0

    duplicate_texts=0
    duplicate_urls=0

    hashes=set()
    urls=set()

    total_words=0
    total_chars=0

    min_words=10**18
    max_words=0
    min_chars=10**18
    max_chars=0

    word_lengths=[]
    char_lengths=[]

    repetitive_docs=0
    repetition_examples=[]

    missing_fields=Counter()

    required_fields=[
        "title",
        "category",
        "year",
        "url",
        "text",
        "word_count",
        "text_length_chars"
    ]

    # --------------------------------------------------------
    # Stream documents
    # --------------------------------------------------------
    for doc in stream_documents(path):
        docs+=1

        if not isinstance(doc,dict):
            missing+=1
            continue

        # Fields
        for field in required_fields:
            if field not in doc:
                missing_fields[field]+=1

        text=doc.get("text","")
        url=doc.get("url","")

        if not isinstance(text,str):
            text=""
        
        # Basic lengths
        chars=len(text)
        words=len(text.split())

        total_words+=words
        total_chars+=chars

        min_words=min(min_words,words)
        max_words=max(max_words,words)
        min_chars=min(min_chars,chars)
        max_chars=max(max_chars,chars)

        # Keep only small arrays of lengths for percentile estimation
        if len(word_lengths)<REPETITION_SAMPLE_SIZE:
            word_lengths.append(words)
            char_lengths.append(chars)

        # Empty/short
        if chars==0 or words==0:
            empty+=1

        if chars<MIN_CHARS or words<MIN_WORDS:
            short+=1

        # Stored metadata consistency
        stored_words=doc.get("word_count",None)
        stored_chars=doc.get("text_length_chars",None)

        if stored_words is not None and stored_words!=words:
            bad_word_count+=1

        if stored_chars is not None and stored_chars!=chars:
            bad_char_count+=1

        # Exact text duplicate
        h=text_hash(text)

        if h in hashes:
            duplicate_texts+=1
        else:
            hashes.add(h)

        # Duplicate URL
        if url:
            if url in urls:
                duplicate_urls+=1
            else:
                urls.add(url)

        # Simple repetition check
        # Only inspect documents with enough text
        if words>=50:
            tokens=text.split()
            unique_ratio=len(set(tokens))/len(tokens)

            if unique_ratio<0.15:
                repetitive_docs+=1

                if len(repetition_examples)<10:
                    repetition_examples.append({
                        "title":doc.get("title","")[:100],
                        "words":words,
                        "unique_ratio":round(unique_ratio,4)
                    })

        # Progress
        if docs%100000==0:
            print(f"Processed {docs:,} documents...",flush=True)

    # --------------------------------------------------------
    # Statistics
    # --------------------------------------------------------
    avg_words=total_words/docs if docs else 0
    avg_chars=total_chars/docs if docs else 0

    print("\nRESULTS")
    print("Documents:",f"{docs:,}")
    print("Total words:",f"{total_words:,}")
    print("Total characters:",f"{total_chars:,}")

    print("\nQUALITY")
    print("Empty documents:",f"{empty:,}",f"({100*empty/docs:.4f}%)")
    print("Very short documents:",f"{short:,}",f"({100*short/docs:.4f}%)")
    print("Exact duplicate texts:",f"{duplicate_texts:,}",f"({100*duplicate_texts/docs:.4f}%)")
    print("Duplicate URLs:",f"{duplicate_urls:,}",f"({100*duplicate_urls/docs:.4f}%)")
    print("Highly repetitive:",f"{repetitive_docs:,}",f"({100*repetitive_docs/docs:.4f}%)")

    print("\nLENGTHS")
    print("Words: min =",f"{min_words:,}",
          "max =",f"{max_words:,}",
          "mean =",f"{avg_words:,.2f}")

    print("Chars: min =",f"{min_chars:,}",
          "max =",f"{max_chars:,}",
          "mean =",f"{avg_chars:,.2f}")

    print("\nMETADATA CONSISTENCY")
    print("Wrong word_count:",f"{bad_word_count:,}")
    print("Wrong text_length_chars:",f"{bad_char_count:,}")

    print("\nMISSING FIELDS")
    if missing_fields:
        for k,v in missing_fields.items():
            print(k,":",f"{v:,}")
    else:
        print("None")

    if repetition_examples:
        print("\nREPETITION EXAMPLES")
        for x in repetition_examples:
            print(x)

    # Return compact statistics
    return {
        "documents":docs,
        "words":total_words,
        "chars":total_chars,
        "empty":empty,
        "short":short,
        "duplicate_texts":duplicate_texts,
        "duplicate_urls":duplicate_urls,
        "repetitive":repetitive_docs,
        "bad_word_count":bad_word_count,
        "bad_char_count":bad_char_count
    }

# ------------------------------------------------------------
# Run all four sources
# ------------------------------------------------------------
results={}

for source,path in source_paths.items():
    results[source]=audit_source(source,path)

# ------------------------------------------------------------
# Overall summary
# ------------------------------------------------------------
print("\n"+"="*70)
print("OVERALL SUMMARY")
print("="*70)

print(f"{'Source':20s}"
      f"{'Docs':>12s}"
      f"{'Words':>15s}"
      f"{'DupText':>12s}"
      f"{'DupURL':>12s}"
      f"{'Short':>12s}")

print("-"*83)

for source,r in results.items():
    print(f"{source:20s}"
          f"{r['documents']:12,d}"
          f"{r['words']:15,d}"
          f"{r['duplicate_texts']:12,d}"
          f"{r['duplicate_urls']:12,d}"
          f"{r['short']:12,d}")

print("="*70)
print("AUDIT COMPLETE")
print("="*70)

RAW LEGAL CORPUS - STREAMING QUALITY AUDIT

----------------------------------------------------------------------
SOURCE: acts
FILE SIZE: 0.679 GB
----------------------------------------------------------------------

RESULTS
Documents: 18,826
Total words: 119,638,348
Total characters: 718,326,398

QUALITY
Empty documents: 4 (0.0212%)
Very short documents: 8 (0.0425%)
Exact duplicate texts: 93 (0.4940%)
Duplicate URLs: 0 (0.0000%)
Highly repetitive: 1,411 (7.4950%)

LENGTHS
Words: min = 0 max = 503,777 mean = 6,354.95
Chars: min = 0 max = 2,960,781 mean = 38,156.08

METADATA CONSISTENCY
Wrong word_count: 0
Wrong text_length_chars: 0

MISSING FIELDS
None

REPETITION EXAMPLES
{'title': 'Bangalore Water Supply and Sewerage Act, 1964', 'words': 24873, 'unique_ratio': 0.1349}
{'title': 'THE BRUHAT BENGALURU MAHANAGARA PALIKE ACT, 2020', 'words': 65701, 'unique_ratio': 0.1415}
{'title': 'Bihar Entertainments Tax Rules, 1984', 'words': 16359, 'unique_ratio': 0.1481}
{'title': 'Estates Parti

In [8]:
import ijson,hashlib
from collections import defaultdict

examples={}
counts=defaultdict(int)

for source,path in source_paths.items():
    print("Scanning:",source)
    with open(path,"rb") as f:
        for doc in ijson.items(f,"item"):
            text=doc.get("text","")
            if not isinstance(text,str) or not text:
                continue
            
            h=hashlib.blake2b(text.encode("utf-8"),digest_size=8).digest()
            key=(source,h)
            counts[key]+=1
            
            if key not in examples:
                examples[key]={
                    "title":doc.get("title",""),
                    "url":doc.get("url",""),
                    "year":doc.get("year",""),
                    "category":doc.get("category",""),
                    "words":len(text.split()),
                    "text":text[:500]
                }

print("\nTOP 20 DUPLICATES")
for key,count in sorted(counts.items(),key=lambda x:x[1],reverse=True)[:20]:
    source,h=key
    e=examples[key]
    print("\n"+"-"*80)
    print("SOURCE:",source)
    print("COPIES:",count)
    print("WORDS:",e["words"])
    print("TITLE:",e["title"])
    print("YEAR:",e["year"])
    print("CATEGORY:",e["category"])
    print("URL:",e["url"])
    print("TEXT:",repr(e["text"][:300]))

Scanning: acts
Scanning: high_courts
Scanning: supreme_court
Scanning: tribunals

TOP 20 DUPLICATES

--------------------------------------------------------------------------------
SOURCE: high_courts
COPIES: 1081
WORDS: 10257
TITLE: 
YEAR: 2022
CATEGORY: 
URL: 
TEXT: "Present:- Hon'ble Mr. Justice Md. Nizamuddin W.P.O. No. 244 of 2021 Bagaria Properties and Investment Pvt. Ltd. & Anr.\n\nVs. U.O.I & Ors.\n\nWith W.P.O. No. 253 of 2021, W.P.O. No. 254 of 2021, W.P.O. No. 255 of 2021, W.P.O. No. 256 of 2021, W.P.O. No. 267 of 2021, W.P.O. No. 268 of 2021, W.P.O. No. 26"

--------------------------------------------------------------------------------
SOURCE: high_courts
COPIES: 944
WORDS: 8190
TITLE: 
YEAR: 2021
CATEGORY: 
URL: 
TEXT: 'THE HONOURABLE SRI JUSTICE BATTU DEVANAND\n\nWrit Petition Nos: 10038/2021, 4167/2021, 4427/2021, 6899/2021, 11511/2021, 11972/2021, 12126/2021, 12413/2021, 12831/2021, 12917/2021, 12946/2021, 13328/2021, 13381/2021, 13519/2021, 13593/2021, 13644/2021, 1

In [9]:
import ijson,json,hashlib,random,os
from collections import Counter

random.seed(42)

OUT_DIR="/kaggle/working/dedup_audit"
os.makedirs(OUT_DIR,exist_ok=True)

train_ratio=0.90

seen={}
source_stats=Counter()

for source,path in source_paths.items():
    print("\nScanning:",source)
    with open(path,"rb") as f:
        for doc in ijson.items(f,"item"):
            text=doc.get("text","")
            if not isinstance(text,str) or not text.strip():
                continue

            h=hashlib.blake2b(text.encode("utf-8"),digest_size=8).hexdigest()

            source_stats[source]+=1

            if h not in seen:
                seen[h]={
                    "source":source,
                    "text":text,
                    "words":len(text.split())
                }

print("\nUnique documents:",len(seen))

# Group unique documents by source
by_source={}
for h,item in seen.items():
    source=item["source"]
    if source not in by_source:
        by_source[source]=[]
    by_source[source].append((h,item))

train_manifest=[]
valid_manifest=[]

for source,items in by_source.items():
    random.shuffle(items)
    split=int(len(items)*train_ratio)

    train_items=items[:split]
    valid_items=items[split:]

    for i,(h,item) in enumerate(train_items):
        train_manifest.append({
            "document_id":f"{source}_{i:07d}",
            "source":source,
            "hash":h
        })

    for i,(h,item) in enumerate(valid_items):
        valid_manifest.append({
            "document_id":f"{source}_{i:07d}",
            "source":source,
            "hash":h
        })

# Save manifests
with open(f"{OUT_DIR}/train_manifest_dedup.json","w") as f:
    json.dump(train_manifest,f)

with open(f"{OUT_DIR}/valid_manifest_dedup.json","w") as f:
    json.dump(valid_manifest,f)

print("\n"+"="*70)
print("DEDUPLICATED CORPUS AUDIT")
print("="*70)

print("Total unique documents:",len(seen))
print("Train documents:",len(train_manifest))
print("Valid documents:",len(valid_manifest))

print("\nSOURCE BREAKDOWN")

for source in source_paths:
    original=source_stats[source]
    unique=len(by_source.get(source,[]))
    train=sum(1 for x in train_manifest if x["source"]==source)
    valid=sum(1 for x in valid_manifest if x["source"]==source)

    print(
        source,
        "original:",original,
        "unique:",unique,
        "removed:",original-unique,
        "train:",train,
        "valid:",valid
    )

# Word statistics
unique_words=0

for h,item in seen.items():
    unique_words+=item["words"]

original_words=0

# We already have original word counts from the previous audit,
# so recompute safely from raw files.
for source,path in source_paths.items():
    print("\nCounting original words:",source)
    with open(path,"rb") as f:
        for doc in ijson.items(f,"item"):
            text=doc.get("text","")
            if isinstance(text,str):
                original_words+=len(text.split())

print("\nOriginal words:",original_words)
print("Unique-document words:",unique_words)
print("Removed duplicate words:",original_words-unique_words)
print("Removed word %:",round(100*(original_words-unique_words)/original_words,2))

print("\nFiles written:")
print(f"{OUT_DIR}/train_manifest_dedup.json")
print(f"{OUT_DIR}/valid_manifest_dedup.json")


Scanning: acts

Scanning: high_courts

Scanning: supreme_court

Scanning: tribunals

Unique documents: 3356594

DEDUPLICATED CORPUS AUDIT
Total unique documents: 3356594
Train documents: 3020932
Valid documents: 335662

SOURCE BREAKDOWN
acts original: 18822 unique: 18732 removed: 90 train: 16858 valid: 1874
high_courts original: 2536738 unique: 2418301 removed: 118437 train: 2176470 valid: 241831
supreme_court original: 55528 unique: 55363 removed: 165 train: 49826 valid: 5537
tribunals original: 869132 unique: 864198 removed: 4934 train: 777778 valid: 86420

Counting original words: acts

Counting original words: high_courts

Counting original words: supreme_court

Counting original words: tribunals

Original words: 2900680187
Unique-document words: 2486313471
Removed duplicate words: 414366716
Removed word %: 14.29

Files written:
/kaggle/working/dedup_audit/train_manifest_dedup.json
/kaggle/working/dedup_audit/valid_manifest_dedup.json


In [4]:
#Custom BPE Tokenizer adapted from A-1
from collections import Counter,defaultdict
import heapq
import re
from tqdm.auto import tqdm

class ByteLevelBPE:
    SPECIALS=["<eos>"]

    def __init__(self,vocab_size=32000):
        self.target_vocab_size=vocab_size
        self._merge_id_pairs=[]
        self.merges=[]
        self._token_bytes=[]
        self._bytes_to_id={}
        self._merge_ranks={}
        self._merge_to_new_id={}
        self.itos=[]
        self.stoi={}
        self.vocab_size=0
        self.eos_id=0

    @staticmethod
    def _get_word_pairs(symbols):
        return Counter(zip(symbols,symbols[1:]))

    @staticmethod
    def _merge_word(symbols,pair,new_id):
        left,right=pair
        merged=[]
        i=0

        while i<len(symbols):
            if i+1<len(symbols) and symbols[i]==left and symbols[i+1]==right:
                merged.append(new_id)
                i+=2
            else:
                merged.append(symbols[i])
                i+=1

        return merged

    def train_from_word_counts(self,word_freqs):
        self._token_bytes=[(byte,) for byte in range(256)]
        words=[]
        freqs=[]
        for word,freq in tqdm(word_freqs.items(),desc="Preparing BPE corpus"):
            symbols=list(word.encode("utf-8"))
            if len(symbols)>=2:
                words.append(symbols)
                freqs.append(freq)

        pair_counts=Counter()
        pair_to_words=defaultdict(set)
        for word_idx,symbols in tqdm(enumerate(words),total=len(words),desc="Building pair statistics"):
            word_freq=freqs[word_idx]
            for pair,occurrences in self._get_word_pairs(symbols).items():
                pair_counts[pair]+=occurrences*word_freq
                pair_to_words[pair].add(word_idx)

        heap=[]
        for pair,count in pair_counts.items():
            left,right=pair
            heap.append((-count,self._token_bytes[left],self._token_bytes[right],pair))
        heapq.heapify(heap)
        max_merges=self.target_vocab_size-256-len(self.SPECIALS)
        for _ in tqdm(range(max_merges),desc="Learning BPE merges"):
            best_pair=None
            while heap:
                neg_count,_,_,pair=heapq.heappop(heap)
                count=pair_counts.get(pair,0)
                if count==-neg_count and count>1:
                    best_pair=pair
                    break

            if best_pair is None:
                break

            left,right=best_pair
            new_id=len(self._token_bytes)
            self._token_bytes.append(self._token_bytes[left]+self._token_bytes[right])
            self._merge_id_pairs.append(best_pair)
            self.merges.append((self._token_bytes[left],self._token_bytes[right]))
            affected_words=list(pair_to_words.get(best_pair,set()))
            changed_pairs=set()
            for word_idx in affected_words:
                symbols=words[word_idx]
                word_freq=freqs[word_idx]
                new_symbols=[]
                i=0
                while i<len(symbols):
                    if i+1<len(symbols) and symbols[i]==left and symbols[i+1]==right:
                        new_symbols.append(new_id)
                        i+=2
                    else:
                        new_symbols.append(symbols[i])
                        i+=1

                if new_symbols==symbols:
                    continue
                old_pairs=self._get_word_pairs(symbols)
                new_pairs=self._get_word_pairs(new_symbols)
                for pair,occurrences in old_pairs.items():
                    pair_counts[pair]-=occurrences*word_freq
                    changed_pairs.add(pair)

                for pair,occurrences in new_pairs.items():
                    pair_counts[pair]+=occurrences*word_freq
                    changed_pairs.add(pair)

                for pair in old_pairs:
                    if pair not in new_pairs:
                        pair_to_words[pair].discard(word_idx)

                for pair in new_pairs:
                    pair_to_words[pair].add(word_idx)

                words[word_idx]=new_symbols
            pair_counts.pop(best_pair,None)
            pair_to_words.pop(best_pair,None)
            for pair in changed_pairs:
                count=pair_counts.get(pair,0)

                if count<=0:
                    pair_counts.pop(pair,None)
                    pair_to_words.pop(pair,None)
                else:
                    left_pair,right_pair=pair
                    heapq.heappush(heap,(-count,self._token_bytes[left_pair],self._token_bytes[right_pair],pair))

        self._bytes_to_id={token_bytes:idx for idx,token_bytes in enumerate(self._token_bytes)}
        self._merge_ranks={pair:rank for rank,pair in enumerate(self._merge_id_pairs)}
        self._merge_to_new_id={pair:256+rank for rank,pair in enumerate(self._merge_id_pairs)}
        self.itos=self.SPECIALS.copy()
        self.itos.extend(self._token_bytes)
        self.stoi={token:i for i,token in enumerate(self.itos)}
        self.eos_id=0
        self.vocab_size=len(self.itos)
        print("Learned merges:",len(self.merges))
        print("Final vocabulary size:",self.vocab_size)
        return self

    def encode_word(self,word):
        symbols=[byte for byte in word.encode("utf-8")]
        while len(symbols)>=2:
            best_rank=None
            best_pair=None
            for i in range(len(symbols)-1):
                pair=(symbols[i],symbols[i+1])
                rank=self._merge_ranks.get(pair)
                if rank is not None and (best_rank is None or rank<best_rank):
                    best_rank=rank
                    best_pair=pair

            if best_pair is None:
                break

            symbols=self._merge_word(symbols,best_pair,self._merge_to_new_id[best_pair])
        return [token_id+1 for token_id in symbols]

    def encode_text(self,text,add_eos=True):
        tokens=[]

        for part in re.findall(r"\s+|\S+",text):
            tokens.extend(self.encode_word(part))

        if add_eos:
            tokens.append(self.eos_id)

        return tokens

In [21]:
import os
import json
import pickle
import hashlib
import ijson
import numpy as np
from tqdm.auto import tqdm

# =========================
# Load tokenizer
# =========================

with open("/kaggle/input/datasets/spikie/capstone-split/legal_bpe_32k.pkl","rb") as f:
    bpe=pickle.load(f)

print("BPE vocab size:",bpe.vocab_size)
print("EOS id:",bpe.eos_id)

def get_hash(text):
    return hashlib.blake2b(text.encode("utf-8"),digest_size=8).hexdigest()

def encode_text(text):
    return bpe.encode_text(text,add_eos=True)

# =========================
# Test tokenizer
# =========================

print("Test 1:",encode_text("hello")[-5:])
print("Test 2:",encode_text("The Supreme Court of India")[-5:])

# =========================
# Paths
# =========================

dedup_dir="/kaggle/working/dedup_audit"
out_dir="/kaggle/working/dedup_corpus"
os.makedirs(out_dir,exist_ok=True)

source_names=[
    "acts",
    "high_courts",
    "supreme_court",
    "tribunals"
]

source_paths={
    "acts":"/root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_1_Pretraining/Train/train_acts.json",
    "high_courts":"/root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_1_Pretraining/Train/train_high_courts.json",
    "supreme_court":"/root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_1_Pretraining/Train/train_supreme_court.json",
    "tribunals":"/root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_1_Pretraining/Train/train_tribunals.json"
}

BPE vocab size: 32000
EOS id: 0
Test 1: [289, 336, 112, 0]
Test 2: [33, 266, 33, 841, 0]


In [ ]:
# =========================
# Load dedup manifests
# =========================

with open(f"{dedup_dir}/train_manifest_dedup.json","r") as f:
    train_manifest=json.load(f)

with open(f"{dedup_dir}/valid_manifest_dedup.json","r") as f:
    valid_manifest=json.load(f)

train_hashes=set(x["hash"] for x in train_manifest)
valid_hashes=set(x["hash"] for x in valid_manifest)

print("Train documents:",len(train_hashes))
print("Valid documents:",len(valid_hashes))

# =========================
# Output files
# =========================

train_bin=f"{out_dir}/train_tokens.bin"
valid_bin=f"{out_dir}/valid_tokens.bin"

train_f=open(train_bin,"wb")
valid_f=open(valid_bin,"wb")

BLOCK_SIZE=2048

train_buffer=[]
valid_buffer=[]

train_docs=0
valid_docs=0
train_tokens=0
valid_tokens=0
train_blocks=0
valid_blocks=0

# =========================
# Streaming tokenization
# =========================

for source in source_names:
    path=source_paths[source]

    print(f"\nProcessing {source}...")
    
    with open(path,"rb") as f:
        for item in tqdm(
            ijson.items(f,"item"),
            desc=source,
            unit="doc"
        ):
            text=item["text"]
            h=get_hash(text)

            if h in train_hashes:
                tokens=encode_text(text)
                train_buffer.extend(tokens)
                train_docs+=1
                train_tokens+=len(tokens)

            elif h in valid_hashes:
                tokens=encode_text(text)
                valid_buffer.extend(tokens)
                valid_docs+=1
                valid_tokens+=len(tokens)

            # Write complete train blocks immediately
            while len(train_buffer)>=BLOCK_SIZE:
                arr=np.asarray(
                    train_buffer[:BLOCK_SIZE],
                    dtype=np.uint16
                )
                train_f.write(arr.tobytes())
                train_buffer=train_buffer[BLOCK_SIZE:]
                train_blocks+=1

            # Write complete validation blocks immediately
            while len(valid_buffer)>=BLOCK_SIZE:
                arr=np.asarray(
                    valid_buffer[:BLOCK_SIZE],
                    dtype=np.uint16
                )
                valid_f.write(arr.tobytes())
                valid_buffer=valid_buffer[BLOCK_SIZE:]
                valid_blocks+=1

            # Periodic flush
            if train_docs%10000==0:
                train_f.flush()
                valid_f.flush()

    print(
        f"{source} done | "
        f"train docs={train_docs:,} | "
        f"valid docs={valid_docs:,} | "
        f"train blocks={train_blocks:,} | "
        f"valid blocks={valid_blocks:,}"
    )

# =========================
# Write remaining complete blocks
# =========================

while len(train_buffer)>=BLOCK_SIZE:
    arr=np.asarray(train_buffer[:BLOCK_SIZE],dtype=np.uint16)
    train_f.write(arr.tobytes())
    train_buffer=train_buffer[BLOCK_SIZE:]
    train_blocks+=1

while len(valid_buffer)>=BLOCK_SIZE:
    arr=np.asarray(valid_buffer[:BLOCK_SIZE],dtype=np.uint16)
    valid_f.write(arr.tobytes())
    valid_buffer=valid_buffer[BLOCK_SIZE:]
    valid_blocks+=1

train_f.flush()
valid_f.flush()
train_f.close()
valid_f.close()

# =========================
# Metadata
# =========================

metadata={
    "vocab_size":bpe.vocab_size,
    "block_size":BLOCK_SIZE,
    "dtype":"uint16",
    "train_docs":train_docs,
    "valid_docs":valid_docs,
    "train_tokens":train_tokens,
    "valid_tokens":valid_tokens,
    "train_blocks":train_blocks,
    "valid_blocks":valid_blocks,
    "train_leftover_tokens":len(train_buffer),
    "valid_leftover_tokens":len(valid_buffer),
    "tokenizer":"legal_bpe_32k",
    "deduplicated":True
}

with open(f"{out_dir}/metadata.json","w") as f:
    json.dump(metadata,f,indent=2)

with open(f"{out_dir}/train_manifest.json","w") as f:
    json.dump(train_manifest,f)

with open(f"{out_dir}/valid_manifest.json","w") as f:
    json.dump(valid_manifest,f)

print("\n========== DONE ==========")
print("Train docs:",f"{train_docs:,}")
print("Valid docs:",f"{valid_docs:,}")
print("Train tokens:",f"{train_tokens:,}")
print("Valid tokens:",f"{valid_tokens:,}")
print("Train blocks:",f"{train_blocks:,}")
print("Valid blocks:",f"{valid_blocks:,}")
print("Train leftover:",len(train_buffer))
print("Valid leftover:",len(valid_buffer))
print("Output:",out_dir)

Train documents: 3020932
Valid documents: 335662

Processing acts...


acts: 0doc [00:00, ?doc/s]

acts done | train docs=16,941 | valid docs=1,881 | train blocks=111,740 | valid blocks=13,253

Processing high_courts...


high_courts: 0doc [00:00, ?doc/s]

high_courts done | train docs=2,300,166 | valid docs=255,394 | train blocks=1,627,677 | valid blocks=190,668

Processing supreme_court...


supreme_court: 0doc [00:00, ?doc/s]

supreme_court done | train docs=2,350,137 | valid docs=260,951 | train blocks=1,823,698 | valid blocks=212,482

Processing tribunals...


tribunals: 0doc [00:00, ?doc/s]

In [7]:
import os

for root,dirs,files in os.walk("/kaggle/input"):
    if "train_tokens.bin" in files:
        print("Dataset path:",root)
        print("train_tokens.bin:",os.path.join(root,"train_tokens.bin"))
        print("valid_tokens.bin:",os.path.join(root,"valid_tokens.bin"))

Dataset path: /kaggle/input/datasets/spikie/capstone-split
train_tokens.bin: /kaggle/input/datasets/spikie/capstone-split/train_tokens.bin
valid_tokens.bin: /kaggle/input/datasets/spikie/capstone-split/valid_tokens.bin


In [3]:
import torch
print("PyTorch:",torch.__version__)
print("CUDA:",torch.cuda.is_available())
print("GPU count:",torch.cuda.device_count())

if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(i,torch.cuda.get_device_name(i))

PyTorch: 2.11.0+cu128
CUDA: True
GPU count: 2
0 Tesla T4
1 Tesla T4


In [6]:
import os,math,time
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.distributed as dist
from torch.utils.data import Dataset,DataLoader
from torch.utils.data.distributed import DistributedSampler
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.amp import autocast,GradScaler
from tqdm.auto import tqdm

VOCAB_SIZE=32000
BLOCK_SIZE=2048

D_MODEL=768
N_LAYERS=16
N_HEADS=12

Q_LORA_RANK=384
KV_LORA_RANK=192
QK_NOPE_HEAD_DIM=48
QK_ROPE_HEAD_DIM=24
V_HEAD_DIM=64

QK_HEAD_DIM=QK_NOPE_HEAD_DIM+QK_ROPE_HEAD_DIM
FFN_DIM=2304

DROPOUT=0.05


class RMSNorm(nn.Module):
    def __init__(self,dim,eps=1e-6):
        super().__init__()
        self.weight=nn.Parameter(torch.ones(dim))
        self.eps=eps

    def forward(self,x):
        rms=x.pow(2).mean(dim=-1,keepdim=True)
        return self.weight*x*torch.rsqrt(rms+self.eps)


class RoPE(nn.Module):
    def __init__(self,dim,max_seq_len=BLOCK_SIZE,base=10000):
        super().__init__()

        inv_freq=1.0/(base**(torch.arange(0,dim,2).float()/dim))
        positions=torch.arange(max_seq_len).float()
        freqs=torch.outer(positions,inv_freq)

        self.register_buffer(
            "cos",
            freqs.cos()[None,None,:,:],
            persistent=False
        )
        self.register_buffer(
            "sin",
            freqs.sin()[None,None,:,:],
            persistent=False
        )

    def forward(self,x,start_pos=0):
        T=x.size(-2)

        cos=self.cos[:,:,start_pos:start_pos+T,:]
        sin=self.sin[:,:,start_pos:start_pos+T,:]

        x1=x[...,::2]
        x2=x[...,1::2]

        out=torch.stack(
            (x1*cos-x2*sin,x1*sin+x2*cos),
            dim=-1
        ).flatten(-2)

        return out


class MLA(nn.Module):
    def __init__(self):
        super().__init__()

        # Query compression
        self.q_a_proj=nn.Linear(
            D_MODEL,
            Q_LORA_RANK,
            bias=False
        )
        self.q_norm=RMSNorm(Q_LORA_RANK)
        self.q_b_proj=nn.Linear(
            Q_LORA_RANK,
            N_HEADS*QK_HEAD_DIM,
            bias=False
        )

        # Compressed KV
        self.kv_a_proj=nn.Linear(
            D_MODEL,
            KV_LORA_RANK+QK_ROPE_HEAD_DIM,
            bias=False
        )
        self.kv_norm=RMSNorm(KV_LORA_RANK)

        self.kv_b_proj=nn.Linear(
            KV_LORA_RANK,
            N_HEADS*(QK_NOPE_HEAD_DIM+V_HEAD_DIM),
            bias=False
        )

        self.rope=RoPE(QK_ROPE_HEAD_DIM)

        self.out_proj=nn.Linear(
            N_HEADS*V_HEAD_DIM,
            D_MODEL,
            bias=False
        )

    def forward(self,x):
        B,T,C=x.shape

        # -------------------------
        # Query
        # -------------------------
        q=self.q_a_proj(x)
        q=self.q_norm(q)
        q=self.q_b_proj(q)

        q=q.view(
            B,T,N_HEADS,QK_HEAD_DIM
        ).transpose(1,2)

        q_nope=q[...,:QK_NOPE_HEAD_DIM]
        q_rope=q[...,QK_NOPE_HEAD_DIM:]

        q_rope=self.rope(q_rope)

        # -------------------------
        # Compressed KV
        # -------------------------
        kv=self.kv_a_proj(x)

        kv_latent=kv[...,:KV_LORA_RANK]
        k_rope=kv[...,KV_LORA_RANK:]

        kv_latent=self.kv_norm(kv_latent)

        kv_full=self.kv_b_proj(kv_latent)

        kv_full=kv_full.view(
            B,T,N_HEADS,
            QK_NOPE_HEAD_DIM+V_HEAD_DIM
        ).transpose(1,2)

        k_nope=kv_full[...,:QK_NOPE_HEAD_DIM]
        v=kv_full[...,QK_NOPE_HEAD_DIM:]

        k_rope=k_rope.unsqueeze(1)
        k_rope=self.rope(k_rope)
        k_rope=k_rope.expand(-1,N_HEADS,-1,-1)

        k=torch.cat((k_nope,k_rope),dim=-1)
        q=torch.cat((q_nope,q_rope),dim=-1)

        out=F.scaled_dot_product_attention(
            q,k,v,
            dropout_p=DROPOUT if self.training else 0.0,
            is_causal=True
        )

        out=out.transpose(1,2).contiguous()
        out=out.view(B,T,N_HEADS*V_HEAD_DIM)

        return self.out_proj(out)

    def forward_cached(self,x,past_kv=None,start_pos=0):
        B,T,C=x.shape

        # -------------------------
        # Query
        # -------------------------
        q=self.q_a_proj(x)
        q=self.q_norm(q)
        q=self.q_b_proj(q)

        q=q.view(
            B,T,N_HEADS,QK_HEAD_DIM
        ).transpose(1,2)

        q_nope=q[...,:QK_NOPE_HEAD_DIM]
        q_rope=q[...,QK_NOPE_HEAD_DIM:]

        q_rope=self.rope(q_rope,start_pos)

        # -------------------------
        # Compressed KV
        # -------------------------
        kv=self.kv_a_proj(x)

        kv_latent=kv[...,:KV_LORA_RANK]
        k_rope=kv[...,KV_LORA_RANK:]

        kv_latent=self.kv_norm(kv_latent)

        if past_kv is not None:
            past_latent,past_k_rope=past_kv

            kv_latent=torch.cat(
                (past_latent,kv_latent),
                dim=1
            )

            k_rope=k_rope.unsqueeze(1)
            k_rope=self.rope(k_rope,start_pos)
            k_rope=k_rope[:,0]

            k_rope=torch.cat(
                (past_k_rope,k_rope),
                dim=1
            )

            causal=False

        else:
            k_rope=k_rope.unsqueeze(1)
            k_rope=self.rope(k_rope,0)
            k_rope=k_rope[:,0]

            causal=True

        kv_full=self.kv_b_proj(kv_latent)

        total_T=kv_latent.size(1)

        kv_full=kv_full.view(
            B,total_T,N_HEADS,
            QK_NOPE_HEAD_DIM+V_HEAD_DIM
        ).transpose(1,2)

        k_nope=kv_full[...,:QK_NOPE_HEAD_DIM]
        v=kv_full[...,QK_NOPE_HEAD_DIM:]

        k_rope=k_rope.unsqueeze(1)
        k_rope=k_rope.expand(-1,N_HEADS,-1,-1)
        k=torch.cat((k_nope,k_rope),dim=-1)
        q=torch.cat(
            (q_nope,q_rope),
            dim=-1
        )

        out=F.scaled_dot_product_attention(
            q,k,v,
            dropout_p=0.0,
            is_causal=causal
        )

        out=out.transpose(1,2).contiguous()
        out=out.view(
            B,T,N_HEADS*V_HEAD_DIM
        )

        new_kv=(
            kv_latent,
            k_rope[:,0]
        )

        return self.out_proj(out),new_kv


class SwiGLU(nn.Module):
    def __init__(self):
        super().__init__()

        self.gate=nn.Linear(
            D_MODEL,
            FFN_DIM,
            bias=False
        )
        self.up=nn.Linear(
            D_MODEL,
            FFN_DIM,
            bias=False
        )
        self.down=nn.Linear(
            FFN_DIM,
            D_MODEL,
            bias=False
        )

    def forward(self,x):
        return self.down(
            F.silu(self.gate(x))*self.up(x)
        )


class M4Block(nn.Module):
    def __init__(self):
        super().__init__()

        self.norm1=RMSNorm(D_MODEL)
        self.attn=MLA()

        self.norm2=RMSNorm(D_MODEL)
        self.ffn=SwiGLU()

    def forward(self,x):
        x=x+self.attn(self.norm1(x))
        x=x+self.ffn(self.norm2(x))
        return x

    def forward_cached(self,x,past_kv=None,start_pos=0):
        attn_out,new_kv=self.attn.forward_cached(
            self.norm1(x),
            past_kv,
            start_pos
        )

        x=x+attn_out
        x=x+self.ffn(self.norm2(x))

        return x,new_kv


class M4Model(nn.Module):
    def __init__(self):
        super().__init__()

        self.token_embedding=nn.Embedding(
            VOCAB_SIZE,
            D_MODEL
        )

        nn.init.normal_(
            self.token_embedding.weight,
            mean=0.0,
            std=0.02
        )

        self.blocks=nn.ModuleList(
            [M4Block() for _ in range(N_LAYERS)]
        )

        self.norm=RMSNorm(D_MODEL)

        self.lm_head=nn.Linear(
            D_MODEL,
            VOCAB_SIZE,
            bias=False
        )

        self.lm_head.weight=self.token_embedding.weight

        self.dropout=nn.Dropout(DROPOUT)

        self.apply(self._init_weights)

    def _init_weights(self,module):
        if isinstance(module,nn.Linear):
            nn.init.normal_(
                module.weight,
                mean=0.0,
                std=0.02
            )

    def forward(self,x):
        x=self.token_embedding(x)
        x=self.dropout(x)

        for block in self.blocks:
            x=block(x)

        x=self.norm(x)

        return self.lm_head(x)

    @torch.no_grad()
    def forward_cached(self,x,past_kvs=None,start_pos=0):
        x=self.token_embedding(x)

        if past_kvs is None:
            past_kvs=[None]*N_LAYERS

        new_kvs=[]

        for i,block in enumerate(self.blocks):
            x,kv=block.forward_cached(
                x,
                past_kvs[i],
                start_pos
            )
            new_kvs.append(kv)

        x=self.norm(x)

        logits=self.lm_head(x)

        return logits,new_kvs

In [7]:
model=M4Model().cuda()
model.eval()

x=torch.randint(0,VOCAB_SIZE,(1,128),device="cuda")

with torch.no_grad():
    normal=model(x)
    cached,past=model.forward_cached(x,None,0)

diff=(normal-cached).abs().max().item()

print("Normal shape:",normal.shape)
print("Cached shape:",cached.shape)
print("Max difference:",diff)
print("Cache layers:",len(past))
print("Cache latent shape:",past[0][0].shape)
print("Cache RoPE shape:",past[0][1].shape)

Normal shape: torch.Size([1, 128, 32000])
Cached shape: torch.Size([1, 128, 32000])
Max difference: 0.0
Cache layers: 16
Cache latent shape: torch.Size([1, 128, 192])
Cache RoPE shape: torch.Size([1, 128, 24])


In [23]:
model.eval()

x=torch.randint(0,VOCAB_SIZE,(1,64),device="cuda")

with torch.no_grad():
    normal_logits=model(x)

    cached_logits,past=model.forward_cached(x,None,0)

    normal_next=normal_logits[:,-1,:]

    cached_next=cached_logits[:,-1,:]

    diff=(normal_next-cached_next).abs().max().item()

    next_token=torch.argmax(cached_next,dim=-1,keepdim=True)

    cached_logits2,past=model.forward_cached(
        next_token,
        past,
        start_pos=x.size(1)
    )

    x2=torch.cat((x,next_token),dim=1)

    normal_logits2=model(x2)

    normal_next2=normal_logits2[:,-1,:]
    cached_next2=cached_logits2[:,-1,:]

    diff2=(normal_next2-cached_next2).abs().max().item()

print("Prefill difference:",diff)
print("1-token decode difference:",diff2)
print("Generated token:",int(next_token.item()))

Prefill difference: 0.0
1-token decode difference: 4.5299530029296875e-06
Generated token: 6570


In [10]:
TRAIN_BIN="/kaggle/input/datasets/spikie/capstone-split/train_tokens.bin"
VALID_BIN="/kaggle/input/datasets/spikie/capstone-split/valid_tokens.bin"

CKPT_DIR="/kaggle/working/m4_checkpoints"
os.makedirs(CKPT_DIR,exist_ok=True)

BATCH_SIZE=5
NUM_WORKERS=2

TARGET_TOKENS=250_000_000

LR=3e-4
MIN_LR=3e-5
WEIGHT_DECAY=0.1
GRAD_CLIP=1.0
WARMUP_FRAC=0.03

VAL_EVERY=2500
VAL_BATCHES=100
SAVE_EVERY=2500

In [11]:
class TokenBlockDataset(Dataset):
    def __init__(self,path,block_size=2048):
        self.data=torch.from_file(
            path,
            dtype=torch.int16,
            size=os.path.getsize(path)//2
        )
        self.block_size=block_size
        self.n_blocks=(len(self.data)-1)//block_size

    def __len__(self):
        return self.n_blocks

    def __getitem__(self,idx):
        start=idx*self.block_size

        block=self.data[
            start:start+self.block_size+1
        ].to(torch.long)

        return block[:-1],block[1:]

In [24]:
%%writefile /kaggle/working/train_m4.py

import os,math,time
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.distributed as dist
from torch.utils.data import Dataset,DataLoader
from torch.utils.data.distributed import DistributedSampler
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.amp import autocast,GradScaler

# ============================================================
# CONFIG
# ============================================================

VOCAB_SIZE=32000
BLOCK_SIZE=2048

D_MODEL=768
N_LAYERS=16
N_HEADS=12

Q_LORA_RANK=384
KV_LORA_RANK=192
QK_NOPE_HEAD_DIM=48
QK_ROPE_HEAD_DIM=24
V_HEAD_DIM=64

QK_HEAD_DIM=QK_NOPE_HEAD_DIM+QK_ROPE_HEAD_DIM
FFN_DIM=2304

DROPOUT=0.05

TRAIN_TOKENS=250_000_000
BATCH_SIZE=5
GRAD_ACCUM=1

LR=3e-4
MIN_LR=3e-5
WEIGHT_DECAY=0.1
GRAD_CLIP=1.0

WARMUP_RATIO=0.03

VAL_EVERY=2500
VAL_BATCHES=100
SAVE_EVERY=2500

TRAIN_BIN="/kaggle/input/datasets/spikie/capstone-split/train_tokens.bin"
VALID_BIN="/kaggle/input/datasets/spikie/capstone-split/valid_tokens.bin"

OUT_DIR="/kaggle/working/m4_checkpoints"

# Set this to True only for the first smoke test.
SMOKE_TEST=False
SMOKE_STEPS=10


# ============================================================
# DATASET
# ============================================================

class TokenBlockDataset(Dataset):
    def __init__(self,path,block_size):
        self.data=torch.from_file(
            path,
            dtype=torch.int16,
            size=os.path.getsize(path)//2
        )
        self.block_size=block_size
        self.n_blocks=(len(self.data)-1)//block_size

    def __len__(self):
        return self.n_blocks

    def __getitem__(self,idx):
        start=idx*self.block_size
        block=self.data[start:start+self.block_size+1].to(torch.int64)
        return block[:-1],block[1:]


# ============================================================
# MODEL
# ============================================================

class RMSNorm(nn.Module):
    def __init__(self,dim,eps=1e-6):
        super().__init__()
        self.weight=nn.Parameter(torch.ones(dim))
        self.eps=eps

    def forward(self,x):
        rms=x.pow(2).mean(dim=-1,keepdim=True)
        return self.weight*x*torch.rsqrt(rms+self.eps)


class RoPE(nn.Module):
    def __init__(self,dim,max_seq_len=BLOCK_SIZE,base=10000):
        super().__init__()

        inv_freq=1.0/(base**(torch.arange(0,dim,2).float()/dim))
        positions=torch.arange(max_seq_len).float()
        freqs=torch.outer(positions,inv_freq)

        self.register_buffer(
            "cos",
            freqs.cos()[None,None,:,:],
            persistent=False
        )
        self.register_buffer(
            "sin",
            freqs.sin()[None,None,:,:],
            persistent=False
        )

    def forward(self,x,start_pos=0):
        T=x.size(-2)

        cos=self.cos[:,:,start_pos:start_pos+T,:]
        sin=self.sin[:,:,start_pos:start_pos+T,:]

        x1=x[...,::2]
        x2=x[...,1::2]

        return torch.stack(
            (x1*cos-x2*sin,x1*sin+x2*cos),
            dim=-1
        ).flatten(-2)


class MLA(nn.Module):
    def __init__(self):
        super().__init__()

        self.q_a_proj=nn.Linear(
            D_MODEL,Q_LORA_RANK,bias=False
        )
        self.q_norm=RMSNorm(Q_LORA_RANK)
        self.q_b_proj=nn.Linear(
            Q_LORA_RANK,N_HEADS*QK_HEAD_DIM,bias=False
        )

        self.kv_a_proj=nn.Linear(
            D_MODEL,
            KV_LORA_RANK+QK_ROPE_HEAD_DIM,
            bias=False
        )
        self.kv_norm=RMSNorm(KV_LORA_RANK)

        self.kv_b_proj=nn.Linear(
            KV_LORA_RANK,
            N_HEADS*(QK_NOPE_HEAD_DIM+V_HEAD_DIM),
            bias=False
        )

        self.rope=RoPE(QK_ROPE_HEAD_DIM)

        self.out_proj=nn.Linear(
            N_HEADS*V_HEAD_DIM,
            D_MODEL,
            bias=False
        )

    def forward(self,x):
        B,T,C=x.shape

        q=self.q_a_proj(x)
        q=self.q_norm(q)
        q=self.q_b_proj(q)

        q=q.view(
            B,T,N_HEADS,QK_HEAD_DIM
        ).transpose(1,2)

        q_nope=q[...,:QK_NOPE_HEAD_DIM]
        q_rope=q[...,QK_NOPE_HEAD_DIM:]
        q_rope=self.rope(q_rope)

        kv=self.kv_a_proj(x)

        kv_latent=kv[...,:KV_LORA_RANK]
        k_rope=kv[...,KV_LORA_RANK:]

        kv_latent=self.kv_norm(kv_latent)

        kv_full=self.kv_b_proj(kv_latent)

        kv_full=kv_full.view(
            B,T,N_HEADS,
            QK_NOPE_HEAD_DIM+V_HEAD_DIM
        ).transpose(1,2)

        k_nope=kv_full[...,:QK_NOPE_HEAD_DIM]
        v=kv_full[...,QK_NOPE_HEAD_DIM:]

        k_rope=k_rope.unsqueeze(1)
        k_rope=self.rope(k_rope)
        k_rope=k_rope.expand(-1,N_HEADS,-1,-1)

        k=torch.cat((k_nope,k_rope),dim=-1)
        q=torch.cat((q_nope,q_rope),dim=-1)

        out=F.scaled_dot_product_attention(
            q,k,v,
            dropout_p=DROPOUT if self.training else 0.0,
            is_causal=True
        )

        out=out.transpose(1,2).contiguous()
        out=out.view(B,T,N_HEADS*V_HEAD_DIM)

        return self.out_proj(out)

    def forward_cached(self,x,past_kv=None,start_pos=0):
        B,T,C=x.shape

        q=self.q_a_proj(x)
        q=self.q_norm(q)
        q=self.q_b_proj(q)

        q=q.view(
            B,T,N_HEADS,QK_HEAD_DIM
        ).transpose(1,2)

        q_nope=q[...,:QK_NOPE_HEAD_DIM]
        q_rope=q[...,QK_NOPE_HEAD_DIM:]
        q_rope=self.rope(q_rope,start_pos)

        kv=self.kv_a_proj(x)

        kv_latent=kv[...,:KV_LORA_RANK]
        k_rope=kv[...,KV_LORA_RANK:]

        kv_latent=self.kv_norm(kv_latent)

        if past_kv is not None:
            past_latent,past_k_rope=past_kv

            kv_latent=torch.cat(
                (past_latent,kv_latent),dim=1
            )

            k_rope=k_rope.unsqueeze(1)
            k_rope=self.rope(k_rope,start_pos)
            k_rope=k_rope[:,0]

            k_rope=torch.cat(
                (past_k_rope,k_rope),dim=1
            )

            causal=False

        else:
            k_rope=k_rope.unsqueeze(1)
            k_rope=self.rope(k_rope,0)
            k_rope=k_rope[:,0]

            causal=True

        kv_full=self.kv_b_proj(kv_latent)

        total_T=kv_latent.size(1)

        kv_full=kv_full.view(
            B,total_T,N_HEADS,
            QK_NOPE_HEAD_DIM+V_HEAD_DIM
        ).transpose(1,2)

        k_nope=kv_full[...,:QK_NOPE_HEAD_DIM]
        v=kv_full[...,QK_NOPE_HEAD_DIM:]

        k_rope=k_rope.unsqueeze(1)
        k_rope=k_rope.expand(-1,N_HEADS,-1,-1)

        k=torch.cat((k_nope,k_rope),dim=-1)
        q=torch.cat((q_nope,q_rope),dim=-1)

        out=F.scaled_dot_product_attention(
            q,k,v,
            dropout_p=0.0,
            is_causal=causal
        )

        out=out.transpose(1,2).contiguous()
        out=out.view(B,T,N_HEADS*V_HEAD_DIM)

        new_kv=(kv_latent,k_rope[:,0])

        return self.out_proj(out),new_kv


class SwiGLU(nn.Module):
    def __init__(self):
        super().__init__()

        self.gate=nn.Linear(
            D_MODEL,FFN_DIM,bias=False
        )
        self.up=nn.Linear(
            D_MODEL,FFN_DIM,bias=False
        )
        self.down=nn.Linear(
            FFN_DIM,D_MODEL,bias=False
        )

    def forward(self,x):
        return self.down(
            F.silu(self.gate(x))*self.up(x)
        )


class M4Block(nn.Module):
    def __init__(self):
        super().__init__()

        self.norm1=RMSNorm(D_MODEL)
        self.attn=MLA()

        self.norm2=RMSNorm(D_MODEL)
        self.ffn=SwiGLU()

    def forward(self,x):
        x=x+self.attn(self.norm1(x))
        x=x+self.ffn(self.norm2(x))
        return x

    def forward_cached(self,x,past_kv=None,start_pos=0):
        attn_out,new_kv=self.attn.forward_cached(
            self.norm1(x),past_kv,start_pos
        )

        x=x+attn_out
        x=x+self.ffn(self.norm2(x))

        return x,new_kv


class M4Model(nn.Module):
    def __init__(self):
        super().__init__()

        self.token_embedding=nn.Embedding(
            VOCAB_SIZE,D_MODEL
        )

        nn.init.normal_(
            self.token_embedding.weight,
            mean=0.0,
            std=0.02
        )

        self.blocks=nn.ModuleList(
            [M4Block() for _ in range(N_LAYERS)]
        )

        self.norm=RMSNorm(D_MODEL)

        self.lm_head=nn.Linear(
            D_MODEL,VOCAB_SIZE,bias=False
        )

        self.lm_head.weight=self.token_embedding.weight

        self.dropout=nn.Dropout(DROPOUT)

        self.apply(self._init_weights)

    def _init_weights(self,module):
        if isinstance(module,nn.Linear):
            nn.init.normal_(
                module.weight,
                mean=0.0,
                std=0.02
            )

    def forward(self,x):
        x=self.token_embedding(x)
        x=self.dropout(x)

        for block in self.blocks:
            x=block(x)

        x=self.norm(x)

        return self.lm_head(x)

    @torch.no_grad()
    def forward_cached(self,x,past_kvs=None,start_pos=0):
        x=self.token_embedding(x)

        if past_kvs is None:
            past_kvs=[None]*N_LAYERS

        new_kvs=[]

        for i,block in enumerate(self.blocks):
            x,kv=block.forward_cached(
                x,past_kvs[i],start_pos
            )
            new_kvs.append(kv)

        x=self.norm(x)

        return self.lm_head(x),new_kvs


# ============================================================
# DDP
# ============================================================

def setup_ddp():
    dist.init_process_group("nccl")

    rank=int(os.environ["LOCAL_RANK"])
    torch.cuda.set_device(rank)

    return rank


def cleanup_ddp():
    dist.destroy_process_group()


# ============================================================
# LR SCHEDULE
# ============================================================

def get_lr(step,total_steps):
    warmup_steps=max(1,int(total_steps*WARMUP_RATIO))

    if step<warmup_steps:
        return LR*step/warmup_steps

    progress=(step-warmup_steps)/max(
        1,total_steps-warmup_steps
    )

    progress=min(1.0,progress)

    return MIN_LR+0.5*(LR-MIN_LR)*(1.0+math.cos(math.pi*progress))


# ============================================================
# VALIDATION
# ============================================================

@torch.no_grad()
def validate(model,val_loader,device,rank):
    model.eval()

    total_loss=0.0
    count=0

    for i,(x,y) in enumerate(val_loader):
        if i>=VAL_BATCHES:
            break

        x=x.to(device,non_blocking=True)
        y=y.to(device,non_blocking=True)

        with autocast("cuda",dtype=torch.float16):
            logits=model(x)
            loss=F.cross_entropy(
                logits.reshape(-1,VOCAB_SIZE),
                y.reshape(-1)
            )

        total_loss+=loss.item()
        count+=1

    value=torch.tensor(
        [total_loss,count],
        dtype=torch.float64,
        device=device
    )

    dist.all_reduce(value,op=dist.ReduceOp.SUM)

    loss=value[0]/value[1]

    model.train()

    if rank==0:
        print(
            f"Validation loss: {loss.item():.4f} "
            f"PPL: {math.exp(min(loss.item(),20)):.2f}"
        )

    return loss.item()


# ============================================================
# CHECKPOINT
# ============================================================

def save_checkpoint(
    model,optimizer,scaler,step,tokens_seen,
    best_val,rank,path
):
    if rank!=0:
        return

    state=model.module

    if hasattr(state,"_orig_mod"):
        state=state._orig_mod

    os.makedirs(os.path.dirname(path),exist_ok=True)

    torch.save(
        {
            "model":state.state_dict(),
            "optimizer":optimizer.state_dict(),
            "scaler":scaler.state_dict(),
            "step":step,
            "tokens_seen":tokens_seen,
            "best_val_loss":best_val,
            "config":{
                "vocab_size":VOCAB_SIZE,
                "block_size":BLOCK_SIZE,
                "d_model":D_MODEL,
                "n_layers":N_LAYERS,
                "n_heads":N_HEADS,
                "q_lora_rank":Q_LORA_RANK,
                "kv_lora_rank":KV_LORA_RANK,
                "qk_nope_head_dim":QK_NOPE_HEAD_DIM,
                "qk_rope_head_dim":QK_ROPE_HEAD_DIM,
                "v_head_dim":V_HEAD_DIM,
                "ffn_dim":FFN_DIM,
                "dropout":DROPOUT
            }
        },
        path
    )

    print("Saved:",path)


# ============================================================
# TRAIN
# ============================================================

def train():
    rank=setup_ddp()
    world_size=dist.get_world_size()

    device=torch.device("cuda",rank)

    torch.backends.cuda.matmul.allow_tf32=True
    torch.backends.cudnn.allow_tf32=True

    if rank==0:
        print("="*60)
        print("M4 TRAINING")
        print("="*60)
        print("GPUs:",world_size)
        print("Batch/GPU:",BATCH_SIZE)
        print("Global batch:",BATCH_SIZE*world_size)
        print("Context:",BLOCK_SIZE)
        print("Training tokens:",TRAIN_TOKENS)
        print("="*60)

    train_dataset=TokenBlockDataset(
        TRAIN_BIN,BLOCK_SIZE
    )

    val_dataset=TokenBlockDataset(
        VALID_BIN,BLOCK_SIZE
    )

    train_sampler=DistributedSampler(
        train_dataset,
        num_replicas=world_size,
        rank=rank,
        shuffle=True,
        drop_last=True
    )

    val_sampler=DistributedSampler(
        val_dataset,
        num_replicas=world_size,
        rank=rank,
        shuffle=False,
        drop_last=True
    )

    train_loader=DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        sampler=train_sampler,
        num_workers=2,
        pin_memory=True,
        persistent_workers=True,
        prefetch_factor=2
    )

    val_loader=DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        sampler=val_sampler,
        num_workers=2,
        pin_memory=True,
        persistent_workers=True,
        prefetch_factor=2
    )

    model=M4Model().to(device)

    if rank==0:
        params=sum(
            p.numel() for p in model.parameters()
        )
        print("Parameters:",params)
        print("Parameters(M):",params/1e6)

    model=torch.compile(
        model,
        mode="default"
    )

    model=DDP(
        model,
        device_ids=[rank],
        broadcast_buffers=False
    )

    optimizer=torch.optim.AdamW(
        model.parameters(),
        lr=LR,
        betas=(0.9,0.95),
        weight_decay=WEIGHT_DECAY,
        fused=True
    )

    scaler=GradScaler("cuda")

    tokens_per_step=(
        BATCH_SIZE*
        world_size*
        (BLOCK_SIZE-1)*
        GRAD_ACCUM
    )

    total_steps=math.ceil(
        TRAIN_TOKENS/tokens_per_step
    )

    if SMOKE_TEST:
        total_steps=SMOKE_STEPS

    if rank==0:
        print("Tokens/step:",tokens_per_step)
        print("Total steps:",total_steps)

    best_val=float("inf")
    tokens_seen=0

    model.train()

    step=0
    data_iter=iter(train_loader)

    start_time=time.time()

    while step<total_steps:
        optimizer.zero_grad(set_to_none=True)

        for _ in range(GRAD_ACCUM):
            try:
                x,y=next(data_iter)
            except StopIteration:
                train_sampler.set_epoch(
                    step+1
                )
                data_iter=iter(train_loader)
                x,y=next(data_iter)

            x=x.to(
                device,
                non_blocking=True
            )
            y=y.to(
                device,
                non_blocking=True
            )

            with autocast(
                "cuda",
                dtype=torch.float16
            ):
                logits=model(x)

                loss=F.cross_entropy(
                    logits.reshape(-1,VOCAB_SIZE),
                    y.reshape(-1)
                )

                loss=loss/GRAD_ACCUM

            scaler.scale(loss).backward()

        scaler.unscale_(optimizer)

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            GRAD_CLIP
        )

        lr=get_lr(step,total_steps)

        for group in optimizer.param_groups:
            group["lr"]=lr

        scaler.step(optimizer)
        scaler.update()

        step+=1

        tokens_seen+=tokens_per_step

        if rank==0 and (
            step==1 or step%100==0
        ):
            elapsed=time.time()-start_time

            tok_per_sec=tokens_seen/max(
                elapsed,1e-6
            )

            print(
                f"step {step}/{total_steps} "
                f"loss {loss.item()*GRAD_ACCUM:.4f} "
                f"lr {lr:.2e} "
                f"tokens {tokens_seen:,} "
                f"tok/s {tok_per_sec:,.0f}"
            )

        if step%VAL_EVERY==0 or step==total_steps:
            val_loss=validate(
                model,
                val_loader,
                device,
                rank
            )

            if val_loss<best_val:
                best_val=val_loss

                save_checkpoint(
                    model,
                    optimizer,
                    scaler,
                    step,
                    tokens_seen,
                    best_val,
                    rank,
                    OUT_DIR+"/best.pt"
                )

            save_checkpoint(
                model,
                optimizer,
                scaler,
                step,
                tokens_seen,
                best_val,
                rank,
                OUT_DIR+"/latest.pt"
            )

            model.train()

    save_checkpoint(
        model,
        optimizer,
        scaler,
        step,
        tokens_seen,
        best_val,
        rank,
        OUT_DIR+"/final.pt"
    )

    dist.barrier()

    if rank==0:
        print("="*60)
        print("TRAINING COMPLETE")
        print("Steps:",step)
        print("Tokens:",tokens_seen)
        print("Best validation loss:",best_val)
        print("="*60)

    cleanup_ddp()


if __name__=="__main__":
    train()

Writing /kaggle/working/train_m4.py


In [25]:
!torchrun --nproc_per_node=2 /kaggle/working/train_m4.py

W1006 11:43:22.880000 187 torch/distributed/run.py:851] 
W1006 11:43:22.880000 187 torch/distributed/run.py:851] *****************************************
W1006 11:43:22.880000 187 torch/distributed/run.py:851] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W1006 11:43:22.880000 187 torch/distributed/run.py:851] *****************************************
M4 TRAINING
GPUs: 2
Batch/GPU: 5
Global batch: 10
Context: 2048
Training tokens: 250000000
Parameters: 135792384
Parameters(M): 135.792384
Tokens/step: 20470
Total steps: 12213
[rank0]:W1006 11:43:44.679000 193 torch/_inductor/utils.py:1731] [0/0] Not enough SMs to use max_autotune_gemm mode
[rank1]:W1006 11:43:44.682000 194 torch/_inductor/utils.py:1731] [0/0] Not enough SMs to use max_autotune_gemm mode
step 1/12213 loss 10.4630 lr 0.00e+00 tokens 20,470 tok/s 284
step 100/1

In [60]:
from huggingface_hub import hf_hub_download
import json

test_file_path=hf_hub_download(
    repo_id="Exploration-Lab/CS-781-Capstone",
    filename="Phase_1_Pretraining/Test/test.json",
    repo_type="dataset"
)

with open(test_file_path,"r",encoding="utf-8") as f:
    test_data=json.load(f)

print("Number of test examples:",len(test_data))
print("First ID:",list(test_data.keys())[0])
print("First prompt words:",len(test_data["1"]["text"].split()))

Number of test examples: 1000
First ID: 1
First prompt words: 512


In [58]:
import sys,os,torch,importlib.util

spec=importlib.util.spec_from_file_location(
    "m4_train",
    "/kaggle/working/train_m4.py"
)

m4_train=importlib.util.module_from_spec(spec)
spec.loader.exec_module(m4_train)

M4Model=m4_train.M4Model

print("M4 model loaded from train_m4.py")
print("Parameters:",sum(p.numel() for p in M4Model().parameters()))

DEVICE=torch.device("cuda")

model=M4Model().to(DEVICE)

checkpoint=torch.load(
    "/kaggle/working/m4_checkpoints/best.pt",
    map_location=DEVICE,
    weights_only=False
)

model.load_state_dict(checkpoint["model"])
model.eval()

print("Model loaded")
print("Step:",checkpoint["step"])
print("Tokens:",checkpoint["tokens_seen"])
print("Best validation loss:",checkpoint["best_val_loss"])

M4 model loaded from train_m4.py
Parameters: 135792384
Model loaded
Step: 12213
Tokens: 250000110
Best validation loss: 1.5915145874023438


In [59]:
import pickle

with open(
    "/kaggle/input/datasets/spikie/capstone-split/legal_bpe_32k.pkl",
    "rb"
) as f:
    bpe=pickle.load(f)

print("BPE vocab size:",bpe.vocab_size)
print("EOS ID:",bpe.eos_id)

BPE vocab size: 32000
EOS ID: 0


In [61]:
def decode_tokens(tokens):
    data=bytearray()

    for token in tokens:
        token=int(token)

        if token==bpe.eos_id:
            break

        data.extend(bpe._token_bytes[token-1])

    return data.decode("utf-8",errors="ignore")

In [62]:
import torch.nn.functional as F

@torch.no_grad()
def m4_forward_cached_fast(model,x,past_kvs=None,start_pos=0):
    B,T=x.shape
    h=model.token_embedding(x)

    if past_kvs is None:
        past_kvs=[None]*len(model.blocks)

    new_kvs=[]

    for i,block in enumerate(model.blocks):
        h_norm=block.norm1(h)
        attn=block.attn

        q=attn.q_a_proj(h_norm)
        q=attn.q_norm(q)
        q=attn.q_b_proj(q)

        q=q.view(
            B,T,N_HEADS,QK_HEAD_DIM
        ).transpose(1,2)

        q_nope=q[...,:QK_NOPE_HEAD_DIM]
        q_rope=q[...,QK_NOPE_HEAD_DIM:]

        q_rope=attn.rope(q_rope,start_pos)

        kv=attn.kv_a_proj(h_norm)

        kv_latent=kv[...,:KV_LORA_RANK]
        k_rope=kv[...,KV_LORA_RANK:]

        kv_latent=attn.kv_norm(kv_latent)

        kv_full=attn.kv_b_proj(kv_latent)

        kv_full=kv_full.view(
            B,T,N_HEADS,
            QK_NOPE_HEAD_DIM+V_HEAD_DIM
        ).transpose(1,2)

        k_nope=kv_full[...,:QK_NOPE_HEAD_DIM]
        v=kv_full[...,QK_NOPE_HEAD_DIM:]

        k_rope=attn.rope(
            k_rope.unsqueeze(1),
            start_pos
        )[:,0]

        k_rope=k_rope.unsqueeze(1).expand(
            -1,N_HEADS,-1,-1
        )

        k=torch.cat(
            (k_nope,k_rope),
            dim=-1
        )

        q=torch.cat(
            (q_nope,q_rope),
            dim=-1
        )

        if past_kvs[i] is not None:
            past_k,past_v=past_kvs[i]

            k=torch.cat(
                (past_k,k),
                dim=2
            )

            v=torch.cat(
                (past_v,v),
                dim=2
            )

            causal=False
        else:
            causal=True

        out=F.scaled_dot_product_attention(
            q,
            k,
            v,
            dropout_p=0.0,
            is_causal=causal
        )

        out=out.transpose(1,2).contiguous()

        out=out.view(
            B,T,N_HEADS*V_HEAD_DIM
        )

        h=h+attn.out_proj(out)

        h=h+block.ffn(
            block.norm2(h)
        )

        new_kvs.append((k,v))

    h=model.norm(h)

    return model.lm_head(h),new_kvs

In [64]:
next_token=torch.argmax(logits[:,-1,:],dim=-1,keepdim=True)

with torch.inference_mode():
    next_logits,new_past_kvs=m4_forward_cached_fast(
        model,
        next_token,
        past_kvs,
        len(prompt_tokens)
    )

print("Next token:",next_token.item())
print("Next logits shape:",next_logits.shape)
print("Old K shape:",past_kvs[0][0].shape)
print("New K shape:",new_past_kvs[0][0].shape)
print("Old V shape:",past_kvs[0][1].shape)
print("New V shape:",new_past_kvs[0][1].shape)

Next token: 33
Next logits shape: torch.Size([1, 1, 32000])
Old K shape: torch.Size([1, 12, 1074, 72])
New K shape: torch.Size([1, 12, 1075, 72])
Old V shape: torch.Size([1, 12, 1074, 64])
New V shape: torch.Size([1, 12, 1075, 64])


In [71]:
import re,time,torch

def top_p_sample_fast(logits,temperature=0.7,top_p=0.95):
    logits=logits/temperature
    k=128

    top_probs,top_indices=torch.topk(
        torch.softmax(logits,dim=-1),
        k,
        dim=-1
    )

    cumulative=torch.cumsum(top_probs,dim=-1)

    remove=cumulative>top_p
    remove[...,1:]=remove[...,:-1].clone()
    remove[...,0]=False

    top_probs[remove]=0
    top_probs/=top_probs.sum(
        dim=-1,
        keepdim=True
    )

    sample_idx=torch.multinomial(
        top_probs,
        1
    )

    return top_indices.gather(
        1,
        sample_idx
    )

def generate_128_words_fast(prompt,max_new_tokens=450):
    TEMPERATURE=0.7
    TOP_P=0.95
    REP_PENALTY=1.15
    REP_WINDOW=50

    prompt_tokens=bpe.encode_text(
        prompt,
        add_eos=False
    )

    x=torch.tensor(
        prompt_tokens,
        dtype=torch.long,
        device=DEVICE
    ).unsqueeze(0)

    generated=[]

    with torch.inference_mode():
        logits,past_kvs=m4_forward_cached_fast(
            model,
            x,
            None,
            0
        )

        for _ in range(max_new_tokens):
            logits=logits[:,-1,:].float()

            for token_id in set(generated[-REP_WINDOW:]):
                if logits[0,token_id]>0:
                    logits[0,token_id]/=REP_PENALTY
                else:
                    logits[0,token_id]*=REP_PENALTY

            logits[0,bpe.eos_id]=-float("inf")

            next_token=top_p_sample_fast(
                logits,
                TEMPERATURE,
                TOP_P
            )

            token_id=int(next_token.item())

            if token_id==bpe.eos_id:
                continue

            generated.append(token_id)

            if len(generated)%4==0:
                text=decode_tokens(generated)
                if len(text.split())>=128:
                    break

            logits,past_kvs=m4_forward_cached_fast(
                model,
                next_token,
                past_kvs,
                len(prompt_tokens)+len(generated)-1
            )

    text=decode_tokens(generated)
    return " ".join(text.split()[:128])

start=time.time()

prediction=generate_128_words_fast(
    test_data["1"]["text"]
)

elapsed=time.time()-start

print("Time:",round(elapsed,2),"seconds")
print("Words:",len(prediction.split()))
print("\nPrediction:\n")
print(prediction)

Time: 5.31 seconds
Words: 128

Prediction:

not in dispute that the Government has taken a decision to acquire land for public purpose. The Government cannot be permitted to run the area without any delay or laches on the part of the State. The decision making process may not amount to an abuse of the power and that too in exercise of its power under Article 226 of the Constitution. The object of the Constitution is to ensure that public interest is not affected unless it is shown by reason of its existence or in violation of any law nor a breach of the constitutional scheme or statutory provisions may render such right unaffected by the act or omission." 10 11. In the present case, we are of the view that the State is not


In [68]:
from tqdm.auto import tqdm
import time

predictions={}
start=time.time()

for example_id,record in tqdm(
    test_data.items(),
    total=len(test_data),
    desc="Generating M4 predictions"
):
    predictions[str(example_id)]={
        "text":generate_128_words_fast(record["text"])
    }

elapsed=time.time()-start

print("\nGeneration complete")
print("Predictions:",len(predictions))
print("Time:",round(elapsed/60,2),"minutes")
print("Average:",round(elapsed/len(predictions),2),"sec/example")

Generating M4 predictions:   0%|          | 0/1000 [00:00<?, ?it/s]


Generation complete
Predictions: 1000
Time: 92.9 minutes
Average: 5.57 sec/example


In [72]:
prediction=generate_128_words_fast(test_data["17"]["text"])

print("Words:",len(prediction.split()))
print("\n",prediction)

Words: 128

 is made out for granting the benefit of doubt to the Appellants. 19. However, we are constrained to observe that in a case where there was no demand for dowry death or harassment of a woman as alleged by the prosecution, the court should have granted benefit of doubt to the Appellants. It is also an admitted fact that both were never informed about the demand for dowry but it was made clear by them that they had been subjected to cruelty or harassment on account of their failure to perform such demand. The Trial Court has not even referred to the relevant facts and circumstances which were found to be wholly unsatisfactory. It is not a case where the Trial Court was required to have considered the


In [70]:
bad_info=[]

for example_id,record in predictions.items():
    words=len(record["text"].split())
    if words!=128:
        bad_info.append((example_id,words,record["text"]))

print("Bad:",len(bad_info))

for example_id,words,text in bad_info[:10]:
    print("\nID:",example_id,"Words:",words)
    print(text)

Bad: 108

ID: 8 Words: 83
accused guilty of the offence under Section 302 read with Section 34 IPC are not applicable to the facts and circumstances of this case. The fact that the accused was charged for the offence under Section 302 read with Section 34 IPC is an undisputed fact. In view of the above discussion I do not find any illegality or perversity in the order of the trial court and consequently it is accordingly set aside. Consequently, Criminal Appeal No. 39 of 2006 is dismissed.

ID: 11 Words: 117
contents of the information which the appellant has made out clearly indicate that it is used for captive consumption, cannot be treated as information that is provided under Section 2(f) of the Act. 22. In view of the foregoing, we hold that the conditions mentioned in Section 11A (1) and Explanation (2) to Section 11A are applicable to the present case as in terms of this proviso, the same is required to be treated as inputs and it is not covered by Section 2(f) of the Act. 23. I

In [74]:
for idx in tqdm(bad):
    predictions[idx]=generate_128_words_fast(test_data[str(idx)]["text"])

  0%|          | 0/108 [00:00<?, ?it/s]

In [85]:
bad=[]
dict_count=0
str_count=0

for idx,data in predictions.items():
    if isinstance(data,dict):
        text=data["text"]
        dict_count+=1
    else:
        text=data
        str_count+=1

    wc=len(text.split())

    if wc!=128:
        bad.append((idx,wc))

print("Total predictions:",len(predictions))
print("Dictionary entries:",dict_count)
print("String entries:",str_count)
print("Exactly 128 words:",len(predictions)-len(bad))
print("Not 128 words:",len(bad))

if bad:
    print("\nBad predictions:")
    for idx,wc in bad[:20]:
        print(idx,wc)
else:
    print("\nAll predictions have exactly 128 words.")

Total predictions: 1000
Dictionary entries: 892
String entries: 108
Exactly 128 words: 1000
Not 128 words: 0

All predictions have exactly 128 words.


In [86]:
for idx in predictions:
    if isinstance(predictions[idx],str):
        predictions[idx]={"text":predictions[idx]}

print(type(predictions["1"]))
print(type(predictions["8"]))
print(predictions["1"].keys())
print(predictions["8"].keys())

<class 'dict'>
<class 'dict'>
dict_keys(['text'])
dict_keys(['text'])


In [87]:
import json

PREDICTION_PATH="/kaggle/working/prediction.json"

with open(PREDICTION_PATH,"w",encoding="utf-8") as f:
    json.dump(predictions,f,ensure_ascii=False,indent=2)

print("Saved:",PREDICTION_PATH)

Saved: /kaggle/working/prediction.json


In [88]:
import zipfile

SUBMISSION_ZIP="/kaggle/working/submission_M4.zip"

with zipfile.ZipFile(SUBMISSION_ZIP,"w",zipfile.ZIP_DEFLATED) as zipf:
    zipf.write(PREDICTION_PATH,arcname="prediction.json")

with zipfile.ZipFile(SUBMISSION_ZIP,"r") as z:
    print("Contents:",z.namelist())

print("Upload this:",SUBMISSION_ZIP)

Contents: ['prediction.json']
Upload this: /kaggle/working/submission_M4.zip


In [1]:
import os
import json
from huggingface_hub import hf_hub_download

HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"

TRAIN_FILES = {
    "fold1_text": "Phase_2_LNER/Train/fold1-text.json",
    "fold1_gold": "Phase_2_LNER/Train/fold1-gold.json",
    "fold2_text": "Phase_2_LNER/Train/fold2-text.json",
    "fold2_gold": "Phase_2_LNER/Train/fold2-gold.json",
    "labels": "Phase_2_LNER/Train/labels.txt"
}

TEST_FILES = {
    "fold3_text": "Phase_2_LNER/Test/fold3-text.json"
}

source_paths = {}

for source_name, file_name in {**TRAIN_FILES, **TEST_FILES}.items():
    print("Downloading:", source_name)

    path = hf_hub_download(
        repo_id=HF_DATASET_REPO,
        filename=file_name,
        repo_type="dataset",
        token=HF_TOKEN
    )

    source_paths[source_name] = path

    print("Path:", path)
    print()

print("All Phase 2 files downloaded/accessed successfully.")

Downloading: fold1_text


fold1-text.json:   0%|          | 0.00/1.11M [00:00<?, ?B/s]

Path: /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_2_LNER/Train/fold1-text.json

Downloading: fold1_gold


fold1-gold.json:   0%|          | 0.00/461k [00:00<?, ?B/s]

Path: /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_2_LNER/Train/fold1-gold.json

Downloading: fold2_text


fold2-text.json:   0%|          | 0.00/1.12M [00:00<?, ?B/s]

... (3 outputs left out)

labels.txt:   0%|          | 0.00/71.0 [00:00<?, ?B/s]

Path: /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_2_LNER/Train/labels.txt

Downloading: fold3_text


fold3-text.json:   0%|          | 0.00/1.19M [00:00<?, ?B/s]

Path: /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_2_LNER/Test/fold3-text.json

All Phase 2 files downloaded/accessed successfully.


In [9]:
import pickle

tokenizer_path = "/kaggle/input/datasets/spikie/capstone-split/legal_bpe_32k.pkl"

with open(tokenizer_path, "rb") as f:
    tokenizer = pickle.load(f)

print("Tokenizer:", type(tokenizer))
print("Vocabulary size:", tokenizer.vocab_size)
print("EOS ID:", tokenizer.eos_id)
print("Test encoding:", tokenizer.encode_text("The Supreme Court of India"))

Tokenizer: <class '__main__.ByteLevelBPE'>
Vocabulary size: 32000
EOS ID: 0
Test encoding: [323, 33, 1413, 33, 399, 33, 266, 33, 841, 0]


In [10]:
import os
import json
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from collections import defaultdict
from tqdm.auto import tqdm

BASE = "/root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_2_LNER"

TRAIN_DIR = os.path.join(BASE, "Train")
TEST_DIR = os.path.join(BASE, "Test")

def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

with open(os.path.join(TRAIN_DIR, "labels.txt"), "r") as f:
    labels = [line.strip() for line in f if line.strip()]

fold1_text = load_json(os.path.join(TRAIN_DIR, "fold1-text.json"))
fold1_gold = load_json(os.path.join(TRAIN_DIR, "fold1-gold.json"))
fold2_text = load_json(os.path.join(TRAIN_DIR, "fold2-text.json"))
fold2_gold = load_json(os.path.join(TRAIN_DIR, "fold2-gold.json"))
fold3_text = load_json(os.path.join(TEST_DIR, "fold3-text.json"))

print("Labels:", labels)
print("Fold 1:", len(fold1_text), "documents")
print("Fold 2:", len(fold2_text), "documents")
print("Test:", len(fold3_text), "documents")

print("\nText example:")
print(next(iter(fold1_text.items())))

print("\nGold example:")
print(next(iter(fold1_gold.items())))

Labels: ['APP', 'RESP', 'A.COUNSEL', 'R.COUNSEL', 'JUDGE', 'WIT', 'AUTH', 'COURT', 'STAT', 'PREC', 'DATE', 'CASENO']
Fold 1: 35 documents
Fold 2: 35 documents
Test: 35 documents

Text example:
('115651329', 'REPORTABLE IN THE SUPREME COURT OF INDIA CRIMINAL APPELLATE JURISDICTION CRIMINAL APPEAL NO. 92/2015 JAGE RAM & ORS. ..Appellants Versus STATE OF HARYANA ..Respondent J U D G M E N T R. BANUMATHI, J. This appeal is preferred against the judgment dated 19.8.2011 passed by the High Court of Punjab and Haryana in Criminal Appeal No.181 SB of 2000, whereby the High Court partly allowed the appeal filed by the appellants thereby confirming the conviction of the appellants with certain modifications. 2. Briefly stated, case of the prosecution is that on the fateful day i.e. 18.11.1994, at about 8.00 A.M. in the morning the complainant Jagdish (PW-5) along with his two sons namely Sukhbir and Mange Ram (PW-6) were busy in cutting pullas (reeds) from the dola of their field. At that time, 

In [12]:
import os
import sys
import torch

sys.path.insert(0, "/kaggle/working")
from train_m4 import M4Model

checkpoint_path = "/kaggle/working/m4_checkpoints/best.pt"

if not os.path.exists(checkpoint_path):
    raise FileNotFoundError(checkpoint_path)

checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=False
)

print("Checkpoint keys:", list(checkpoint.keys()))

if "model" in checkpoint:
    state_dict = checkpoint["model"]
else:
    state_dict = checkpoint

print("Number of parameter tensors:", len(state_dict))
print("Checkpoint config:", checkpoint.get("config", "Not stored"))

m4_model = M4Model()
missing, unexpected = m4_model.load_state_dict(state_dict, strict=False)

print("Missing keys:", len(missing))
print("Unexpected keys:", len(unexpected))

Checkpoint keys: ['model', 'optimizer', 'scaler', 'step', 'tokens_seen', 'best_val_loss', 'config']
Number of parameter tensors: 195
Checkpoint config: {'vocab_size': 32000, 'block_size': 2048, 'd_model': 768, 'n_layers': 16, 'n_heads': 12, 'q_lora_rank': 384, 'kv_lora_rank': 192, 'qk_nope_head_dim': 48, 'qk_rope_head_dim': 24, 'v_head_dim': 64, 'ffn_dim': 2304, 'dropout': 0.05}
Missing keys: 0
Unexpected keys: 0


In [13]:
import types
import torch

def get_m4_hidden_states(self, input_ids):
    x = self.token_embedding(input_ids)
    x = self.dropout(x)

    for block in self.blocks:
        x = block(x)

    return self.norm(x)

m4_model.get_hidden_states = types.MethodType(
    get_m4_hidden_states, m4_model
)

m4_model = m4_model.to(device)
m4_model.eval()

print("M4 hidden-state extraction is ready.")

M4 hidden-state extraction is ready.


In [18]:
import bisect

def tokenize_with_offsets(text, tokenizer):
    token_ids = tokenizer.encode_text(text, add_eos=False)

    # Map every character boundary to its UTF-8 byte position.
    byte_boundaries = [0]
    for char in text:
        byte_boundaries.append(
            byte_boundaries[-1] + len(char.encode("utf-8"))
        )

    offsets = []
    byte_pos = 0

    for token_id in token_ids:
        token_bytes = tokenizer._token_bytes[token_id - 1]
        token_end = byte_pos + len(token_bytes)

        # Find characters overlapping this token's byte span.
        char_start = max(
            0, bisect.bisect_right(byte_boundaries, byte_pos) - 1
        )
        char_end = bisect.bisect_left(byte_boundaries, token_end)

        offsets.append((char_start, char_end))
        byte_pos = token_end

    return token_ids, offsets


# Verify on one real document.
token_ids, offsets = tokenize_with_offsets(text, tokenizer)

print("Characters:", len(text))
print("Tokens:", len(token_ids))
print("Offsets:", len(offsets))
print("\nFirst 15 token alignments:")

for i in range(min(15, len(token_ids))):
    start, end = offsets[i]
    print(i, repr(text[start:end]), (start, end))

print("\nAnnotation alignment checks:")
for start, end, label, entity in gold[:10]:
    print(
        repr(entity),
        "matches text:", text[start:end] == entity,
        "| overlapping tokens:",
        sum(1 for s, e in offsets if s < end and e > start)
    )

Characters: 18616
Tokens: 6640
Offsets: 6640

First 15 token alignments:
0 'REPORTABLE' (0, 10)
1 ' ' (10, 11)
2 'IN' (11, 13)
3 ' ' (13, 14)
4 'THE' (14, 17)
5 ' ' (17, 18)
6 'SUPREME' (18, 25)
7 ' ' (25, 26)
8 'COURT' (26, 31)
9 ' ' (31, 32)
10 'OF' (32, 34)
11 ' ' (34, 35)
12 'INDIA' (35, 40)
13 ' ' (40, 41)
14 'CRIMINAL' (41, 49)

Annotation alignment checks:
'STATE OF HARYANA' matches text: True | overlapping tokens: 5
'19.8.2011' matches text: True | overlapping tokens: 2
'High Court of Punjab and Haryana' matches text: True | overlapping tokens: 11
'High Court' matches text: True | overlapping tokens: 3
'18.11.1994' matches text: True | overlapping tokens: 2
'Jagdish' matches text: True | overlapping tokens: 1
'Mange Ram' matches text: True | overlapping tokens: 4
'Jage Ram' matches text: True | overlapping tokens: 4
'Jage Ram' matches text: True | overlapping tokens: 4
'Jage Ram' matches text: True | overlapping tokens: 4


In [21]:
import torch
import torch.nn as nn

num_labels = len(labels)

def make_boundary_targets(text, gold):
    start_y = torch.zeros(len(text), num_labels)
    end_y = torch.zeros(len(text), num_labels)
    for start, end, label, entity in gold:
        label_id = label2id[label]
        start_y[start, label_id] = 1
        end_y[end - 1, label_id] = 1
    return start_y, end_y

start_y, end_y = make_boundary_targets(text, gold)
print("Text length:", len(text))
print("Start targets:", int(start_y.sum().item()))
print("End targets:", int(end_y.sum().item()))
print("Expected entities:", len(gold))

Text length: 18616
Start targets: 104
End targets: 104
Expected entities: 104


In [22]:
class BoundaryHead(nn.Module):
    def __init__(self, hidden_size=768, num_labels=12):
        super().__init__()
        self.start = nn.Linear(hidden_size, num_labels)
        self.end = nn.Linear(hidden_size, num_labels)

    def forward(self, hidden):
        return self.start(hidden), self.end(hidden)

boundary_head = BoundaryHead().to(device)
print("Parameters:", sum(p.numel() for p in boundary_head.parameters()))

Parameters: 18456


In [23]:
import random
random.seed(42)

docs = []
for fold_text, fold_gold in [(fold1_text, fold1_gold), (fold2_text, fold2_gold)]:
    for doc_id in fold_text:
        docs.append((doc_id, fold_text[doc_id], fold_gold[doc_id]))

random.shuffle(docs)
val_docs = docs[:7]
train_docs = docs[7:]

def make_chunks(text, gold, chunk_size=1536, overlap=256):
    ids, offsets = tokenize_with_offsets(text, tokenizer)
    chunks = []
    step = chunk_size - overlap
    for a in range(0, len(ids), step):
        b = min(a + chunk_size, len(ids))
        if a >= b:
            break
        cs, ce = offsets[a][0], offsets[b-1][1]
        if ce <= cs:
            continue
        chunk_text = text[cs:ce]
        chunk_gold = [(s-cs, e-cs, lab, ent) for s,e,lab,ent in gold if cs <= s and e <= ce]
        chunks.append({"ids": ids[a:b], "text": chunk_text, "gold": chunk_gold})
        if b == len(ids):
            break
    return chunks

train_chunks = [c for _,t,g in train_docs for c in make_chunks(t,g)]
val_chunks = [c for _,t,g in val_docs for c in make_chunks(t,g)]
print("Train documents/chunks:", len(train_docs), len(train_chunks))
print("Validation documents/chunks:", len(val_docs), len(val_chunks))
print("Max chunk tokens:", max(len(c["ids"]) for c in train_chunks + val_chunks))

Train documents/chunks: 63 583
Validation documents/chunks: 7 52
Max chunk tokens: 1536


In [25]:
def prepare_chunks(chunks):
    data = []
    for c in chunks:
        ids, offsets = tokenize_with_offsets(c["text"], tokenizer)
        start_y, end_y = make_boundary_targets(c["text"], c["gold"])
        data.append({"ids": torch.tensor(ids, dtype=torch.long), "offsets": offsets, "start_y": start_y, "end_y": end_y, "text": c["text"], "gold": c["gold"]})
    return data

train_data = prepare_chunks(train_chunks)
val_data = prepare_chunks(val_chunks)
print("Prepared:", len(train_data), len(val_data))
print("Token/character shapes:", train_data[0]["ids"].shape, train_data[0]["start_y"].shape)

Prepared: 583 52
Token/character shapes: torch.Size([1536]) torch.Size([4526, 12])


In [26]:
def token_hidden_to_chars(hidden, text, offsets):
    char_idx = torch.zeros(len(text), dtype=torch.long, device=hidden.device)
    for i, (s, e) in enumerate(offsets):
        char_idx[s:e] = i
    return hidden[0, char_idx]

sample = train_data[0]
with torch.no_grad():
    ids = sample["ids"].unsqueeze(0).to(device)
    hidden = m4_model.get_hidden_states(ids)
    char_hidden = token_hidden_to_chars(hidden, sample["text"], sample["offsets"])

print("Token hidden:", tuple(hidden.shape))
print("Character hidden:", tuple(char_hidden.shape))
print("Target shape:", tuple(sample["start_y"].shape))

Token hidden: (1, 1536, 768)
Character hidden: (4526, 768)
Target shape: (4526, 12)


In [29]:
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(boundary_head.parameters(), lr=3e-4, weight_decay=1e-4)

In [30]:
from tqdm.auto import tqdm

m4_model.eval()
boundary_head.train()

for p in m4_model.parameters():
    p.requires_grad = False

for epoch in range(1):
    total_loss = 0
    progress = tqdm(train_data, desc=f"Epoch {epoch+1}")
    for sample in progress:
        ids = sample["ids"].unsqueeze(0).to(device)
        sy = sample["start_y"].to(device)
        ey = sample["end_y"].to(device)

        with torch.no_grad():
            hidden = m4_model.get_hidden_states(ids)
            char_hidden = token_hidden_to_chars(hidden, sample["text"], sample["offsets"])

        start_logits, end_logits = boundary_head(char_hidden)
        loss = (criterion(start_logits, sy) + criterion(end_logits, ey)) / 2

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        progress.set_postfix(loss=f"{loss.item():.4f}")

    print(f"Epoch {epoch+1} average loss: {total_loss / len(train_data):.4f}")

Epoch 1:   0%|          | 0/583 [00:00<?, ?it/s]

Epoch 1 average loss: 0.0605


In [32]:
import torch
from tqdm.auto import tqdm

m4_model.eval()
boundary_head.eval()

val_loss = 0.0

with torch.no_grad():
    for sample in tqdm(val_data, desc="M4 validation"):
        ids = sample["ids"].unsqueeze(0).to(device)
        sy = sample["start_y"].to(device)
        ey = sample["end_y"].to(device)

        hidden = m4_model.get_hidden_states(ids)
        char_hidden = token_hidden_to_chars(
            hidden, sample["text"], sample["offsets"]
        )

        start_logits, end_logits = boundary_head(char_hidden)
        loss = (
            criterion(start_logits, sy) +
            criterion(end_logits, ey)
        ) / 2

        val_loss += loss.item()

print("M4 validation loss:", round(val_loss / len(val_data), 4))

M4 validation:   0%|          | 0/52 [00:00<?, ?it/s]

M4 validation loss: 0.0263


In [44]:
start_pos_weight = start_pos_weight.clamp(max=20)
end_pos_weight = end_pos_weight.clamp(max=20)

start_criterion = torch.nn.BCEWithLogitsLoss(
    pos_weight=start_pos_weight
)
end_criterion = torch.nn.BCEWithLogitsLoss(
    pos_weight=end_pos_weight
)

optimizer = torch.optim.AdamW(
    boundary_head.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)

print("Start weights:", start_pos_weight.tolist())
print("End weights:", end_pos_weight.tolist())
print("Optimizer reset with learning rate 1e-4")

Start weights: [20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0]
End weights: [20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0]
Optimizer reset with learning rate 1e-4


In [45]:
m4_model.eval()
boundary_head.train()
for p in m4_model.parameters():
    p.requires_grad = False

total_loss = 0.0
progress = tqdm(train_data, desc="M4 weighted epoch")

for sample in progress:
    ids = sample["ids"].unsqueeze(0).to(device)
    sy = sample["start_y"].to(device)
    ey = sample["end_y"].to(device)

    with torch.no_grad():
        hidden = m4_model.get_hidden_states(ids)
        char_hidden = token_hidden_to_chars(
            hidden, sample["text"], sample["offsets"]
        )

    start_logits, end_logits = boundary_head(char_hidden)
    loss = (
        start_criterion(start_logits, sy) +
        end_criterion(end_logits, ey)
    ) / 2

    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(boundary_head.parameters(), 1.0)
    optimizer.step()

    total_loss += loss.item()
    progress.set_postfix(loss=f"{loss.item():.4f}")

print("M4 weighted training loss:", round(total_loss / len(train_data), 4))

M4 weighted epoch:   0%|          | 0/583 [00:00<?, ?it/s]

M4 weighted training loss: 0.0363


In [46]:
m4_model.eval()
boundary_head.eval()
val_loss = 0.0

with torch.no_grad():
    for sample in tqdm(val_data, desc="M4 weighted validation"):
        ids = sample["ids"].unsqueeze(0).to(device)
        sy = sample["start_y"].to(device)
        ey = sample["end_y"].to(device)

        hidden = m4_model.get_hidden_states(ids)
        char_hidden = token_hidden_to_chars(
            hidden, sample["text"], sample["offsets"])
        start_logits, end_logits = boundary_head(char_hidden)

        loss = (
            start_criterion(start_logits, sy) +
            end_criterion(end_logits, ey)
        ) / 2
        val_loss += loss.item()

print("M4 weighted validation loss:", round(val_loss / len(val_data), 4))

M4 weighted validation:   0%|          | 0/52 [00:00<?, ?it/s]

M4 weighted validation loss: 0.0347


In [47]:
sample = val_data[0]
m4_model.eval()
boundary_head.eval()

with torch.no_grad():
    ids = sample["ids"].unsqueeze(0).to(device)
    hidden = m4_model.get_hidden_states(ids)
    char_hidden = token_hidden_to_chars(hidden, sample["text"], sample["offsets"])
    start_logits, end_logits = boundary_head(char_hidden)
    start_probs = torch.sigmoid(start_logits)
    end_probs = torch.sigmoid(end_logits)

for threshold in [0.3, 0.5, 0.7]:
    start_counts = (start_probs >= threshold).sum(dim=0).tolist()
    end_counts = (end_probs >= threshold).sum(dim=0).tolist()
    print(f"\nThreshold {threshold}")
    for i, label in enumerate(labels):
        print(f"{label:10s} starts={start_counts[i]:4d} ends={end_counts[i]:4d}")


Threshold 0.3
APP        starts= 203 ends= 291
RESP       starts= 211 ends= 176
A.COUNSEL  starts= 186 ends= 151
R.COUNSEL  starts= 193 ends= 175
JUDGE      starts= 166 ends= 146
WIT        starts= 174 ends= 202
AUTH       starts= 173 ends= 180
COURT      starts= 167 ends= 233
STAT       starts= 201 ends= 184
PREC       starts= 265 ends= 256
DATE       starts= 226 ends= 291
CASENO     starts= 209 ends= 228

Threshold 0.5
APP        starts= 125 ends= 179
RESP       starts= 101 ends=  84
A.COUNSEL  starts= 106 ends=  56
R.COUNSEL  starts= 106 ends=  50
JUDGE      starts= 102 ends=  76
WIT        starts= 118 ends= 113
AUTH       starts=  80 ends=  74
COURT      starts=  83 ends=  90
STAT       starts= 115 ends= 104
PREC       starts= 130 ends= 120
DATE       starts= 167 ends= 151
CASENO     starts= 101 ends= 107

Threshold 0.7
APP        starts=  72 ends=  68
RESP       starts=  57 ends=  45
A.COUNSEL  starts=  52 ends=  31
R.COUNSEL  starts=  49 ends=  32
JUDGE      starts=  48 ends=  3

In [49]:
def decode_ordered(start_logits, end_logits, labels, threshold=0.5, max_span=500):
    start_probs = torch.sigmoid(start_logits)
    end_probs = torch.sigmoid(end_logits)
    pred = set()

    for i, label in enumerate(labels):
        starts = torch.where(start_probs[:, i] >= threshold)[0].tolist()
        ends = torch.where(end_probs[:, i] >= threshold)[0].tolist()
        used_ends = set()

        for start in starts:
            candidates = [e for e in ends if e >= start and e - start < max_span and e not in used_ends]
            if candidates:
                end = min(candidates)
                pred.add((start, end + 1, label))
                used_ends.add(end)

    return pred

def evaluate_ordered(threshold=0.5):
    tp = total_pred = total_gold = 0
    m4_model.eval()
    boundary_head.eval()

    with torch.no_grad():
        for sample in tqdm(val_data, desc="M4 exact-span evaluation"):
            ids = sample["ids"].unsqueeze(0).to(device)
            hidden = m4_model.get_hidden_states(ids)
            char_hidden = token_hidden_to_chars(hidden, sample["text"], sample["offsets"])
            sl, el = boundary_head(char_hidden)

            pred = decode_ordered(sl, el, labels, threshold=threshold)
            gold = set((s, e, label) for s, e, label, _ in sample["gold"])
            tp += len(pred & gold)
            total_pred += len(pred)
            total_gold += len(gold)

    p = tp / total_pred if total_pred else 0
    r = tp / total_gold if total_gold else 0
    f1 = 2 *p*r/(p+r) if p + r else 0
    print(f"Threshold={threshold} | P={p:.4f} R={r:.4f} F1={f1:.4f} | Pred={total_pred} Gold={total_gold} TP={tp}")

for threshold in [0.5, 0.7, 0.9]:
    evaluate_ordered(threshold)

M4 exact-span evaluation:   0%|          | 0/52 [00:00<?, ?it/s]

Threshold=0.5 | P=0.0008 R=0.0099 F1=0.0014 | Pred=10560 Gold=809 TP=8


M4 exact-span evaluation:   0%|          | 0/52 [00:00<?, ?it/s]

Threshold=0.7 | P=0.0006 R=0.0025 F1=0.0010 | Pred=3328 Gold=809 TP=2


M4 exact-span evaluation:   0%|          | 0/52 [00:00<?, ?it/s]

Threshold=0.9 | P=0.0000 R=0.0000 F1=0.0000 | Pred=409 Gold=809 TP=0


In [54]:
torch.save({
    "boundary_head": boundary_head.state_dict(),
    "labels": labels,
    "epoch": 2,
}, "/kaggle/working/m4_ner_head.pt")
print("Saved fallback M4 NER head")

Saved fallback M4 NER head


In [55]:
m4_model.eval()
boundary_head.train()

total_loss = 0.0
progress = tqdm(train_data, desc="M4 weighted epoch 2")

for sample in progress:
    ids = sample["ids"].unsqueeze(0).to(device)
    sy = sample["start_y"].to(device)
    ey = sample["end_y"].to(device)

    with torch.no_grad():
        hidden = m4_model.get_hidden_states(ids)
        char_hidden = token_hidden_to_chars(
            hidden, sample["text"], sample["offsets"]
        )

    start_logits, end_logits = boundary_head(char_hidden)
    loss = (
        start_criterion(start_logits, sy) +
        end_criterion(end_logits, ey)
    ) / 2

    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(boundary_head.parameters(), 1.0)
    optimizer.step()

    total_loss += loss.item()
    progress.set_postfix(loss=f"{loss.item():.4f}")

print("M4 weighted training loss:", round(total_loss / len(train_data), 4))

M4 weighted epoch 2:   0%|          | 0/583 [00:00<?, ?it/s]

M4 weighted training loss: 0.0256


In [56]:
m4_model.eval()
boundary_head.eval()
val_loss = 0.0

with torch.no_grad():
    for sample in tqdm(val_data, desc="M4 validation"):
        ids = sample["ids"].unsqueeze(0).to(device)
        sy = sample["start_y"].to(device)
        ey = sample["end_y"].to(device)

        hidden = m4_model.get_hidden_states(ids)
        char_hidden = token_hidden_to_chars(
            hidden, sample["text"], sample["offsets"]
        )
        sl, el = boundary_head(char_hidden)

        loss = (start_criterion(sl, sy) + end_criterion(el, ey)) / 2
        val_loss += loss.item()

print("M4 weighted validation loss:", round(val_loss / len(val_data), 4))

M4 validation:   0%|          | 0/52 [00:00<?, ?it/s]

M4 weighted validation loss: 0.0293


In [57]:
torch.save({
    "boundary_head": boundary_head.state_dict(),
    "labels": labels,
    "epoch": 3,
    "val_loss": 0.0293
}, "/kaggle/working/m4_ner_head.pt")
print("Saved M4 NER head")

Saved M4 NER head


In [58]:
def decode_local_maxima(sl, el, text, labels, threshold=0.5, max_span=500):
    sp = torch.sigmoid(sl)
    ep = torch.sigmoid(el)
    pred = set()

    for i, label in enumerate(labels):
        s_scores = sp[:, i]
        e_scores = ep[:, i]

        s_mask = (s_scores >= threshold)
        e_mask = (e_scores >= threshold)

        s_mask[1:-1] &= (s_scores[1:-1] >= s_scores[:-2]) & (s_scores[1:-1] >= s_scores[2:])
        e_mask[1:-1] &= (e_scores[1:-1] >= e_scores[:-2]) & (e_scores[1:-1] >= e_scores[2:])

        starts = torch.where(s_mask)[0].tolist()
        ends = torch.where(e_mask)[0].tolist()

        for s in starts:
            candidates = [e for e in ends if s <= e < min(len(text), s + max_span)]
            if candidates:
                e = max(candidates, key=lambda x: e_scores[x].item())
                pred.add((s, e + 1, label))

    return pred

def evaluate_local_maxima(threshold=0.5):
    tp = total_pred = total_gold = 0
    m4_model.eval()
    boundary_head.eval()

    with torch.no_grad():
        for sample in tqdm(val_data, desc="M4 local-max validation"):
            ids = sample["ids"].unsqueeze(0).to(device)
            hidden = m4_model.get_hidden_states(ids)
            char_hidden = token_hidden_to_chars(hidden, sample["text"], sample["offsets"])
            sl, el = boundary_head(char_hidden)

            pred = decode_local_maxima(sl, el, sample["text"], labels, threshold)
            gold = set((s, e, label) for s, e, label, _ in sample["gold"])

            tp += len(pred & gold)
            total_pred += len(pred)
            total_gold += len(gold)

    p = tp / total_pred if total_pred else 0
    r = tp / total_gold if total_gold else 0
    f1 = 2 * p * r / (p + r) if p + r else 0
    print(f"Threshold={threshold} | P={p:.4f} R={r:.4f} F1={f1:.4f} | Pred={total_pred} Gold={total_gold} TP={tp}")

for t in [0.5, 0.7, 0.9]:
    evaluate_local_maxima(t)

M4 local-max validation:   0%|          | 0/52 [00:00<?, ?it/s]

Threshold=0.5 | P=0.0000 R=0.0000 F1=0.0000 | Pred=10615 Gold=809 TP=0


M4 local-max validation:   0%|          | 0/52 [00:00<?, ?it/s]

Threshold=0.7 | P=0.0000 R=0.0000 F1=0.0000 | Pred=3792 Gold=809 TP=0


M4 local-max validation:   0%|          | 0/52 [00:00<?, ?it/s]

Threshold=0.9 | P=0.0000 R=0.0000 F1=0.0000 | Pred=494 Gold=809 TP=0


In [60]:
sample = val_data[0]
s, e, label, entity_text = sample["gold"][0]
ids = sample["ids"].unsqueeze(0).to(device)

with torch.no_grad():
    hidden = m4_model.get_hidden_states(ids)
    char_hidden = token_hidden_to_chars(hidden, sample["text"], sample["offsets"])

print("Text length:", len(sample["text"]))
print("Token count:", len(sample["ids"]))
print("Hidden shape:", tuple(hidden.shape))
print("Character hidden shape:", tuple(char_hidden.shape))
print("Gold:", repr(sample["text"][s:e]), (s, e))
print("Gold start target:", sample["start_y"][s, labels.index(label)].item())
print("Gold end target:", sample["end_y"][e-1, labels.index(label)].item())

Text length: 4171
Token count: 1536
Hidden shape: (1, 1536, 768)
Character hidden shape: (4171, 768)
Gold: 'STATE (NCT OF DELHI)' (157, 177)
Gold start target: 1.0
Gold end target: 1.0


In [61]:
boundary_head = BoundaryHead(hidden_size=768, num_labels=len(labels)).to(device)

start_criterion = torch.nn.BCEWithLogitsLoss(
    pos_weight=torch.full((len(labels),), 20.0, device=device)
)
end_criterion = torch.nn.BCEWithLogitsLoss(
    pos_weight=torch.full((len(labels),), 20.0, device=device)
)

optimizer = torch.optim.AdamW(
    boundary_head.parameters(), lr=3e-4, weight_decay=1e-4
)

print("M4 NER head reinitialized; ready to train.")

M4 NER head reinitialized; ready to train.


In [62]:
m4_model.eval()
boundary_head.train()
for p in m4_model.parameters():
    p.requires_grad = False

total_loss = 0.0
progress = tqdm(train_data, desc="M4 fresh head")

for sample in progress:
    ids = sample["ids"].unsqueeze(0).to(device)
    sy = sample["start_y"].to(device)
    ey = sample["end_y"].to(device)

    with torch.no_grad():
        hidden = m4_model.get_hidden_states(ids)
        char_hidden = token_hidden_to_chars(
            hidden, sample["text"], sample["offsets"]
        )

    sl, el = boundary_head(char_hidden)
    loss = (start_criterion(sl, sy) + end_criterion(el, ey)) / 2

    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(boundary_head.parameters(), 1.0)
    optimizer.step()

    total_loss += loss.item()
    progress.set_postfix(loss=f"{loss.item():.4f}")

print("Fresh-head training loss:", round(total_loss / len(train_data), 4))

M4 fresh head:   0%|          | 0/583 [00:00<?, ?it/s]

Fresh-head training loss: 0.0716


In [63]:
m4_model.eval()
boundary_head.eval()
val_loss = 0.0

with torch.no_grad():
    for sample in tqdm(val_data, desc="M4 fresh-head validation"):
        ids = sample["ids"].unsqueeze(0).to(device)
        sy = sample["start_y"].to(device)
        ey = sample["end_y"].to(device)

        hidden = m4_model.get_hidden_states(ids)
        char_hidden = token_hidden_to_chars(
            hidden, sample["text"], sample["offsets"]
        )
        sl, el = boundary_head(char_hidden)
        loss = (start_criterion(sl, sy) + end_criterion(el, ey)) / 2
        val_loss += loss.item()

print("Fresh-head weighted validation loss:", round(val_loss / len(val_data), 4))

M4 fresh-head validation:   0%|          | 0/52 [00:00<?, ?it/s]

Fresh-head weighted validation loss: 0.0386


In [65]:
import torch, json

m4_model.eval()
boundary_head.eval()

tp = pred_total = gold_total = 0
start_gold_sum = start_neg_sum = end_gold_sum = end_neg_sum = 0.0
start_gold_n = start_neg_n = end_gold_n = end_neg_n = 0
per_label = {label: [0, 0, 0] for label in labels}

with torch.no_grad():
    for sample in tqdm(val_data, desc="M4 final validation"):
        ids = sample["ids"].unsqueeze(0).to(device)
        hidden = m4_model.get_hidden_states(ids)
        char_hidden = token_hidden_to_chars(hidden, sample["text"], sample["offsets"])
        sl, el = boundary_head(char_hidden)
        sp, ep = torch.sigmoid(sl), torch.sigmoid(el)
        sy = sample["start_y"].to(device).bool()
        ey = sample["end_y"].to(device).bool()

        start_gold_sum += sp[sy].sum().item()
        start_neg_sum += sp[~sy].sum().item()
        end_gold_sum += ep[ey].sum().item()
        end_neg_sum += ep[~ey].sum().item()
        start_gold_n += sy.sum().item()
        start_neg_n += (~sy).sum().item()
        end_gold_n += ey.sum().item()
        end_neg_n += (~ey).sum().item()

        # Candidate spans use local maxima, with one-to-one ordered pairing.
        pred = set()
        for i, label in enumerate(labels):
            ss, es = sp[:, i], ep[:, i]
            sm = ss >= 0.5
            em = es >= 0.5
            if len(ss) > 2:
                sm[1:-1] &= (ss[1:-1] >= ss[:-2]) & (ss[1:-1] >= ss[2:])
                em[1:-1] &= (es[1:-1] >= es[:-2]) & (es[1:-1] >= es[2:])
            starts = torch.where(sm)[0].tolist()
            ends = torch.where(em)[0].tolist()
            used = set()
            for s in starts:
                candidates = [e for e in ends if s <= e < min(len(sample["text"]), s + 500) and e not in used]
                if candidates:
                    e = max(candidates, key=lambda x: (ss[s].item() + es[x].item()))
                    pred.add((s, e + 1, label))
                    used.add(e)

        gold = set((s, e, lab) for s, e, lab, _ in sample["gold"])
        tp += len(pred & gold)
        pred_total += len(pred)
        gold_total += len(gold)
        for lab in labels:
            per_label[lab][0] += len({x for x in pred & gold if x[2] == lab})
            per_label[lab][1] += len({x for x in pred if x[2] == lab})
            per_label[lab][2] += len({x for x in gold if x[2] == lab})

p = tp / pred_total if pred_total else 0.0
r = tp / gold_total if gold_total else 0.0
f1 = 2*p*r/(p+r) if p+r else 0.0

print(f"Exact-span P={p:.4f} R={r:.4f} F1={f1:.4f}")
print(f"Predictions={pred_total} Gold={gold_total} Exact matches={tp}")
print("Mean boundary probabilities:")
print("Start gold/non-gold:", round(start_gold_sum/max(start_gold_n,1),4),
      round(start_neg_sum/max(start_neg_n,1),4))
print("End gold/non-gold:", round(end_gold_sum/max(end_gold_n,1),4),
      round(end_neg_sum/max(end_neg_n,1),4))
print("\nPer-label P/R/F1:")
for lab in labels:
    t, npred, ngold = per_label[lab]
    lp = t/npred if npred else 0.0
    lr = t/ngold if ngold else 0.0
    lf = 2*lp*lr/(lp+lr) if lp+lr else 0.0
    print(f"{lab:10s} P={lp:.3f} R={lr:.3f} F1={lf:.3f} (TP={t}, pred={npred}, gold={ngold})")

save_path = "/kaggle/working/m4_ner_head.pt"
torch.save({
    "boundary_head": boundary_head.state_dict(),
    "labels": labels,
    "val_precision": p,
    "val_recall": r,
    "val_f1": f1,
    "val_loss": 0.0386
}, save_path)
print("\nSaved current head:", save_path)

M4 final validation:   0%|          | 0/52 [00:00<?, ?it/s]

Exact-span P=0.0005 R=0.0074 F1=0.0010
Predictions=11453 Gold=809 Exact matches=6
Mean boundary probabilities:
Start gold/non-gold: 0.382 0.0203
End gold/non-gold: 0.3254 0.0214

Per-label P/R/F1:
APP        P=0.000 R=0.000 F1=0.000 (TP=0, pred=1249, gold=68)
RESP       P=0.000 R=0.000 F1=0.000 (TP=0, pred=809, gold=17)
A.COUNSEL  P=0.000 R=0.000 F1=0.000 (TP=0, pred=866, gold=21)
R.COUNSEL  P=0.000 R=0.000 F1=0.000 (TP=0, pred=785, gold=20)
JUDGE      P=0.000 R=0.000 F1=0.000 (TP=0, pred=827, gold=17)
WIT        P=0.000 R=0.000 F1=0.000 (TP=0, pred=809, gold=31)
AUTH       P=0.000 R=0.000 F1=0.000 (TP=0, pred=854, gold=55)
COURT      P=0.002 R=0.022 F1=0.004 (TP=2, pred=935, gold=91)
STAT       P=0.003 R=0.017 F1=0.005 (TP=3, pred=998, gold=173)
PREC       P=0.000 R=0.000 F1=0.000 (TP=0, pred=1202, gold=141)
DATE       P=0.001 R=0.010 F1=0.002 (TP=1, pred=1146, gold=96)
CASENO     P=0.000 R=0.000 F1=0.000 (TP=0, pred=973, gold=79)

Saved current head: /kaggle/working/m4_ner_head.pt


In [66]:
import torch
import torch.nn as nn

class CharacterBoundaryHead(nn.Module):
    def __init__(self, hidden_size=768, num_labels=12,
                 char_embed_size=32, relative_embed_size=32,
                 max_relative_position=256):
        super().__init__()
        self.char_embedding = nn.Embedding(256, char_embed_size)
        self.relative_embedding = nn.Embedding(
            max_relative_position, relative_embed_size
        )
        self.max_relative_position = max_relative_position
        combined_size = hidden_size + char_embed_size + relative_embed_size
        self.start_layer = nn.Linear(combined_size, num_labels)
        self.end_layer = nn.Linear(combined_size, num_labels)

    def forward(self, hidden_states, text, offsets):
        dev = hidden_states.device
        n = len(text)
        token_idx = torch.zeros(n, dtype=torch.long, device=dev)
        rel_pos = torch.zeros(n, dtype=torch.long, device=dev)

        for j, (s, e) in enumerate(offsets):
            token_idx[s:e] = j
            rel_pos[s:e] = torch.arange(e-s, device=dev).clamp(
                max=self.max_relative_position - 1
            )

        byte_lists = [list(c.encode("utf-8")) for c in text]
        max_bytes = max((len(x) for x in byte_lists), default=1)
        byte_ids = torch.zeros(n, max_bytes, dtype=torch.long, device=dev)
        byte_mask = torch.zeros(n, max_bytes, device=dev)

        for j, values in enumerate(byte_lists):
            byte_ids[j, :len(values)] = torch.tensor(values, device=dev)
            byte_mask[j, :len(values)] = 1.0

        byte_emb = self.char_embedding(byte_ids)
        char_emb = (
            (byte_emb * byte_mask.unsqueeze(-1)).sum(dim=1)
            / byte_mask.sum(dim=1, keepdim=True).clamp_min(1.0)
        )
        rel_emb = self.relative_embedding(rel_pos)
        token_hidden = hidden_states[0, token_idx, :]
        features = torch.cat([token_hidden, char_emb, rel_emb], dim=-1)[None]

        return self.start_layer(features), self.end_layer(features)

boundary_head = CharacterBoundaryHead(
    hidden_size=768, num_labels=len(labels)
).to(device)

print("Initialized M1-style CharacterBoundaryHead for M4.")

Initialized M1-style CharacterBoundaryHead for M4.


In [69]:
import torch
import torch.nn as nn
from torch.optim import AdamW
from tqdm.auto import tqdm

# Freeze M4 encoder; train only the character-aware NER head
m4_model.eval()
for p in m4_model.parameters():
    p.requires_grad = False

boundary_head = CharacterBoundaryHead(
    hidden_size=768, num_labels=len(labels)
).to(device)

optimizer = AdamW(boundary_head.parameters(), lr=1e-3, weight_decay=1e-4)
pos_weight = torch.full((len(labels),), 20.0, device=device)
criterion = nn.BCEWithLogitsLoss(
    pos_weight=pos_weight, reduction="none"
)

def run_head_epoch(data, training):
    boundary_head.train(training)
    total_loss = 0.0
    total_chars = 0
    gold_spans = set()
    pred_spans = set()

    context = torch.enable_grad() if training else torch.no_grad()
    with context:
        for sample in tqdm(data, desc="Training" if training else "Validation"):
            ids = sample["ids"].to(device).long().reshape(1, -1)
            text = sample["text"]
            offsets = sample["offsets"]

            with torch.no_grad():
                hidden = m4_model.get_hidden_states(ids)

            start_logits, end_logits = boundary_head(hidden, text, offsets)
            start_logits = start_logits.squeeze(0)
            end_logits = end_logits.squeeze(0)

            start_y = sample["start_y"].to(device).float()
            end_y = sample["end_y"].to(device).float()

            loss_start = criterion(start_logits, start_y).mean()
            loss_end = criterion(end_logits, end_y).mean()
            loss = loss_start + loss_end

            if training:
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(boundary_head.parameters(), 1.0)
                optimizer.step()

            total_loss += loss.item()
            total_chars += 1

            if not training:
                # Exact-span evaluation using M1's boundary decoding strategy
                start_probs = torch.sigmoid(start_logits).cpu()
                end_probs = torch.sigmoid(end_logits).cpu()

                for label_id, label in enumerate(labels):
                    starts = torch.where(start_probs[:, label_id] >= 0.7)[0]
                    ends = torch.where(end_probs[:, label_id] >= 0.7)[0]
                    candidates = []

                    for s in starts.tolist():
                        valid = ends[
                            (ends >= s) &
                            (ends < min(len(text), s + 200))
                        ]
                        for e in valid.tolist():
                            score = (
                                start_probs[s, label_id].item() *
                                end_probs[e, label_id].item()
                            )
                            candidates.append((score, s, e))

                    candidates.sort(reverse=True)
                    used_starts = set()
                    used_ends = set()

                    for score, s, e in candidates[:10000]:
                        if s in used_starts or e in used_ends:
                            continue
                        pred_spans.add((sample.get("doc_id", ""), 
                                        sample.get("chunk_idx", -1),
                                        s, e + 1, label))
                        used_starts.add(s)
                        used_ends.add(e)

                for s, e, label, *rest in sample["gold"]:
                    gold_spans.add((
                        sample.get("doc_id", ""),
                        sample.get("chunk_idx", -1),
                        int(s), int(e), label
                    ))

    if training:
        return total_loss / max(total_chars, 1), None, None, None

    tp = len(pred_spans & gold_spans)
    precision = tp / max(len(pred_spans), 1)
    recall = tp / max(len(gold_spans), 1)
    f1 = 2 * precision * recall / max(precision + recall, 1e-12)
    return total_loss / max(total_chars, 1), precision, recall, f1


best_f1 = -1.0
save_path = "/kaggle/working/m4_character_head_best.pt"

for epoch in range(1):
    train_loss, _, _, _ = run_head_epoch(train_data, True)
    val_loss, precision, recall, f1 = run_head_epoch(val_data, False)

    print(
        f"Epoch {epoch}/5 | train_loss={train_loss:.4f} "
        f"| val_loss={val_loss:.4f} | P={precision:.4f} "
        f"| R={recall:.4f} | F1={f1:.4f}"
    )

    if f1 > best_f1:
        best_f1 = f1
        torch.save({
            "boundary_head": boundary_head.state_dict(),
            "labels": labels,
            "val_precision": precision,
            "val_recall": recall,
            "val_f1": f1,
            "epoch": epoch
        }, save_path)
        print("Saved best checkpoint:", save_path)

print("Best validation F1:", best_f1)

Training:   0%|          | 0/583 [00:00<?, ?it/s]

Validation:   0%|          | 0/52 [00:00<?, ?it/s]

Epoch 0/5 | train_loss=0.0476 | val_loss=0.0305 | P=0.0842 | R=0.2410 | F1=0.1248
Saved best checkpoint: /kaggle/working/m4_character_head_best.pt
Best validation F1: 0.12476007677543187


In [ ]:
def make_document_chunks(text, tokenizer, chunk_size=1536, overlap=256):
    result = tokenizer.encode_text(text, add_eos=False)

    if isinstance(result, dict):
        ids = result.get("input_ids", result.get("ids"))
        offsets = result.get("offsets")
    elif isinstance(result, (tuple, list)):
        ids = result[0]
        offsets = result[1]
    else:
        raise TypeError(f"Unexpected encode_text result: {type(result)}")

    if ids is None or offsets is None:
        raise ValueError("Could not find token IDs and offsets in encode_text result.")

    if len(ids) != len(offsets):
        raise ValueError(f"Token/offset lengths differ: {len(ids)} vs {len(offsets)}")

    chunks = []
    step = chunk_size - overlap

    for token_start in range(0, len(ids), step):
        token_end = min(token_start + chunk_size, len(ids))
        chunk_ids = ids[token_start:token_end]
        chunk_offsets = offsets[token_start:token_end]

        valid = [(int(s), int(e)) for s, e in chunk_offsets if int(e) > int(s)]
        if not valid:
            continue

        char_start = min(s for s, e in valid)
        char_end = max(e for s, e in valid)

        chunks.append({
            "input_ids": chunk_ids,
            "offsets": [(s - char_start, e - char_start) for s, e in valid],
            "text": text[char_start:char_end],
            "char_start": char_start,
            "char_end": char_end
        })

        if token_end == len(ids):
            break

    return chunks

print("Chunking function updated.")
print("Test:", len(make_document_chunks(
    next(iter(fold3_text.values())), tokenizer, CHUNK_SIZE, CHUNK_OVERLAP
)), "chunks")

In [86]:
word = "PETITION"
symbols = list(word.encode("utf-8"))

while len(symbols) >= 2:
    best_rank = None
    best_pair = None

    for i in range(len(symbols) - 1):
        pair = (symbols[i], symbols[i + 1])
        rank = tokenizer._merge_ranks.get(pair)
        if rank is not None and (best_rank is None or rank < best_rank):
            best_rank = rank
            best_pair = pair

    if best_pair is None:
        break

    merged_id = tokenizer._merge_to_new_id[best_pair]
    print("Before:", symbols, "| merging:", best_pair, "->", merged_id)
    symbols = tokenizer._merge_word(symbols, best_pair, merged_id)
    print("After: ", symbols)

print("Final symbols:", symbols)
print("encode_word:", tokenizer.encode_word(word))

Before: [80, 69, 84, 73, 84, 73, 79, 78] | merging: (79, 78) -> 434
After:  [80, 69, 84, 73, 84, 73, 434]
Before: [80, 69, 84, 73, 84, 73, 434] | merging: (84, 73) -> 550
After:  [80, 69, 550, 550, 434]
Before: [80, 69, 550, 550, 434] | merging: (80, 69) -> 818
After:  [818, 550, 550, 434]
Before: [818, 550, 550, 434] | merging: (550, 434) -> 1092
After:  [818, 550, 1092]
Before: [818, 550, 1092] | merging: (818, 550) -> 1264
After:  [1264, 1092]
Before: [1264, 1092] | merging: (1264, 1092) -> 1274
After:  [1274]
Final symbols: [1274]
encode_word: [1275]


In [89]:
import re

def encode_text_with_offsets(text, tokenizer):
    all_ids = []
    all_offsets = []

    for match in re.finditer(r"\s+|\S+", text):
        part = match.group()
        symbols = list(part.encode("utf-8"))
        spans = [(i, i + 1) for i in range(len(symbols))]

        # Keep byte symbols and token spans synchronized.
        while len(symbols) >= 2:
            best_rank = None
            best_pair = None
            best_i = -1

            for i in range(len(symbols) - 1):
                pair = (symbols[i], symbols[i + 1])
                rank = tokenizer._merge_ranks.get(pair)
                if rank is not None and (
                    best_rank is None or rank < best_rank
                ):
                    best_rank = rank
                    best_pair = pair
                    best_i = i

            if best_pair is None:
                break

            # Merge spans for the selected adjacent pair.
            spans[best_i:best_i + 2] = [
                (spans[best_i][0], spans[best_i + 1][1])
            ]

            # Apply merge to byte symbols only for determining the next
            # pair; _merge_word's output is token IDs, not bytes.
            symbols = symbols[:best_i] + [
                tokenizer._merge_to_new_id[best_pair]
            ] + symbols[best_i + 2:]

        # The tokenizer's own IDs are authoritative.
        part_ids = tokenizer.encode_word(part)

        if len(part_ids) != len(spans):
            raise RuntimeError(
                f"Alignment mismatch in {part!r}: "
                f"{len(part_ids)} tokens vs {len(spans)} spans."
            )

        all_ids.extend(part_ids)

        # Convert byte offsets to Python character offsets.
        byte_to_char = {}
        byte_pos = 0
        for char_pos, ch in enumerate(part):
            for _ in ch.encode("utf-8"):
                byte_to_char[byte_pos] = char_pos
                byte_pos += 1

        for bs, be in spans:
            all_offsets.append((
                match.start() + byte_to_char[bs],
                match.start() + byte_to_char[be - 1] + 1
            ))

    assert len(all_ids) == len(all_offsets)
    return all_ids, all_offsets

# Quick validation
sample = "PETITION REPORTABLE STATE (NCT OF DELHI)"
ids, offsets = encode_text_with_offsets(sample, tokenizer)
print("IDs:", len(ids), "| offsets:", len(offsets))
print([(sample[s:e], (s, e)) for s, e in offsets])

IDs: 12 | offsets: 12
[('PETITION', (0, 8)), (' ', (8, 9)), ('REPORTABLE', (9, 19)), (' ', (19, 20)), ('STATE', (20, 25)), (' ', (25, 26)), ('(NCT', (26, 30)), (' ', (30, 31)), ('OF', (31, 33)), (' ', (33, 34)), ('DELHI', (34, 39)), (')', (39, 40))]


In [93]:
def make_document_chunks(text, tokenizer=None, chunk_size=1536, overlap=256):
    ids, offsets = encode_text_with_offsets(text, tokenizer if tokenizer is not None else globals()["tokenizer"])
    chunks = []
    step = chunk_size - overlap

    for ts in range(0, len(ids), step):
        te = min(ts + chunk_size, len(ids))
        valid = [(s, e) for s, e in offsets[ts:te] if e > s]
        if not valid:
            continue

        cs = min(s for s, e in valid)
        ce = max(e for s, e in valid)
        chunks.append({
            "input_ids": ids[ts:te],
            "offsets": [(s - cs, e - cs) for s, e in valid],
            "text": text[cs:ce],
            "char_start": cs,
            "char_end": ce
        })

        if te == len(ids):
            break

    return chunks

print("Fixed. Now rerun your existing frozen prediction cell.")

Fixed. Now rerun your existing frozen prediction cell.


In [95]:
import torch

def decode_entities(
    start_logits, end_logits, text_length, labels,
    start_threshold=0.7, end_threshold=0.7,
    max_span_length=200, top_k=100
):
    start_probs = torch.sigmoid(start_logits[0]).detach().cpu()
    end_probs = torch.sigmoid(end_logits[0]).detach().cpu()
    predictions = []

    for label_id, label in enumerate(labels):
        ss = start_probs[:, label_id]
        es = end_probs[:, label_id]

        starts = torch.where(ss >= start_threshold)[0]
        ends = torch.where(es >= end_threshold)[0]

        if len(starts) > top_k:
            starts = starts[torch.topk(ss[starts], top_k).indices]
        if len(ends) > top_k:
            ends = ends[torch.topk(es[ends], top_k).indices]

        candidates = []
        for s in starts.tolist():
            valid_ends = ends[
                (ends >= s) &
                (ends < min(text_length, s + max_span_length))
            ]
            for e in valid_ends.tolist():
                candidates.append((ss[s].item() * es[e].item(), s, e))

        candidates.sort(reverse=True)
        used_starts, used_ends = set(), set()

        for score, s, e in candidates:
            if s in used_starts or e in used_ends:
                continue
            predictions.append((s, e + 1, label))
            used_starts.add(s)
            used_ends.add(e)

    return predictions

print("decode_entities defined. Rerun the frozen prediction cell.")

decode_entities defined. Rerun the frozen prediction cell.


In [97]:
import json
import torch
from tqdm.auto import tqdm

# Load the best head-only checkpoint
ckpt = torch.load(
    "/kaggle/working/m4_character_head_best.pt",
    map_location=device,
    weights_only=False
)
boundary_head.load_state_dict(ckpt["boundary_head"])
boundary_head.to(device).eval()
m4_model.eval()

test_predictions_v1 = {}

with torch.no_grad():
    for doc_id, text in tqdm(fold3_text.items(), desc="M4 frozen predictions", unit="doc"):
        chunks = make_document_chunks(
            text, tokenizer,
            chunk_size=CHUNK_SIZE,
            overlap=CHUNK_OVERLAP
        )
        entities = set()

        for item in chunks:
            ids = torch.tensor(
                [item["input_ids"]], dtype=torch.long, device=device
            )
            hidden = m4_model.get_hidden_states(ids)
            start_logits, end_logits = boundary_head(
                hidden, item["text"], item["offsets"]
            )
            spans = decode_entities(
                start_logits, end_logits, len(item["text"]), labels
            )

            for start, end, label in spans:
                gs = item["char_start"] + start
                ge = item["char_start"] + end
                if 0 <= gs < ge <= len(text):
                    entities.add((gs, ge, label))

        test_predictions_v1[doc_id] = [
            [s, e, lab] for s, e, lab in sorted(entities)
        ]

json_path_v1 = "/kaggle/working/prediction_M4_frozen.json"
with open(json_path_v1, "w", encoding="utf-8") as f:
    json.dump(test_predictions_v1, f, ensure_ascii=False)

print("Documents:", len(test_predictions_v1))
print("Predicted entities:", sum(len(v) for v in test_predictions_v1.values()))
print("Saved:", json_path_v1)

M4 frozen predictions:   0%|          | 0/35 [00:00<?, ?doc/s]

Documents: 35
Predicted entities: 8982
Saved: /kaggle/working/prediction_M4_frozen.json


In [99]:
import os
import zipfile

json_path = "/kaggle/working/prediction_M4_frozen.json"
zip_path = "/kaggle/working/submission_M4.zip"

assert os.path.exists(json_path), f"Missing predictions: {json_path}"

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    z.write(json_path, arcname="prediction.json")

with zipfile.ZipFile(zip_path, "r") as z:
    print("ZIP contents:", z.namelist())
    assert "prediction.json" in z.namelist()

print("Submission ready:", zip_path)
print("Size:", round(os.path.getsize(zip_path) / 1024, 2), "KB")

ZIP contents: ['prediction.json']
Submission ready: /kaggle/working/submission_M4.zip
Size: 50.49 KB


In [ ]:
import os
import zipfile

zip_path_v1 = "/kaggle/working/submission_M4_frozen.zip"

with zipfile.ZipFile(zip_path_v1, "w", zipfile.ZIP_DEFLATED) as z:
    z.write(json_path_v1, arcname="prediction.json")

with zipfile.ZipFile(zip_path_v1, "r") as z:
    print("ZIP contents:", z.namelist())
    assert z.namelist() == ["prediction.json"]

print("Submission ZIP:", zip_path_v1)
print("Size:", round(os.path.getsize(zip_path_v1) / 1024, 2), "KB")

In [101]:
#unfrozen
import torch
import torch.nn as nn
from torch.optim import AdamW
from tqdm.auto import tqdm

# Freeze the full encoder first
for p in m4_model.parameters():
    p.requires_grad = False

# Unfreeze the last two transformer blocks and final normalization
for block in m4_model.blocks[-2:]:
    for p in block.parameters():
        p.requires_grad = True

for p in m4_model.norm.parameters():
    p.requires_grad = True

m4_model.train()

# Reinitialize the M1-style character-aware head
boundary_head = CharacterBoundaryHead(
    hidden_size=768, num_labels=len(labels)
).to(device)

optimizer = AdamW([
    {"params": [p for p in m4_model.parameters() if p.requires_grad],
     "lr": 1e-5},
    {"params": boundary_head.parameters(), "lr": 1e-3}
], weight_decay=1e-4)

pos_weight = torch.full((len(labels),), 20.0, device=device)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

def run_head_epoch(data, training):
    m4_model.train(training)
    boundary_head.train(training)
    total_loss = 0.0
    gold_spans = set()
    pred_spans = set()

    context = torch.enable_grad() if training else torch.no_grad()
    with context:
        for sample in tqdm(data, desc="Training" if training else "Validation"):
            ids = sample["ids"].to(device).long().reshape(1, -1)
            text = sample["text"]
            offsets = sample["offsets"]

            hidden = m4_model.get_hidden_states(ids)
            start_logits, end_logits = boundary_head(hidden, text, offsets)
            start_logits = start_logits.squeeze(0)
            end_logits = end_logits.squeeze(0)

            start_y = sample["start_y"].to(device).float()
            end_y = sample["end_y"].to(device).float()

            loss = criterion(start_logits, start_y) + criterion(end_logits, end_y)

            if training:
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(
                    list(m4_model.parameters()) + list(boundary_head.parameters()), 1.0
                )
                optimizer.step()

            total_loss += loss.item()

            if not training:
                start_probs = torch.sigmoid(start_logits).cpu()
                end_probs = torch.sigmoid(end_logits).cpu()
                doc_id = sample.get("doc_id", "")
                chunk_idx = sample.get("chunk_idx", -1)

                for label_id, label in enumerate(labels):
                    starts = torch.where(start_probs[:, label_id] >= 0.7)[0]
                    ends = torch.where(end_probs[:, label_id] >= 0.7)[0]
                    candidates = []

                    for s in starts.tolist():
                        valid_ends = ends[
                            (ends >= s) &
                            (ends < min(len(text), s + 200))
                        ]
                        for e in valid_ends.tolist():
                            score = (
                                start_probs[s, label_id].item() *
                                end_probs[e, label_id].item()
                            )
                            candidates.append((score, s, e))

                    candidates.sort(reverse=True)
                    used_starts, used_ends = set(), set()

                    for score, s, e in candidates:
                        if s in used_starts or e in used_ends:
                            continue
                        pred_spans.add((doc_id, chunk_idx, s, e + 1, label))
                        used_starts.add(s)
                        used_ends.add(e)

                for s, e, label, *rest in sample["gold"]:
                    gold_spans.add((doc_id, chunk_idx, int(s), int(e), label))

    avg_loss = total_loss / max(len(data), 1)
    if training:
        return avg_loss, None, None, None

    tp = len(pred_spans & gold_spans)
    precision = tp / max(len(pred_spans), 1)
    recall = tp / max(len(gold_spans), 1)
    f1 = 2 * precision * recall / max(precision + recall, 1e-12)
    return avg_loss, precision, recall, f1


best_f1 = -1.0
save_path = "/kaggle/working/m4_character_unforzen.pt"

for epoch in range(1,2):
    train_loss, _, _, _ = run_head_epoch(train_data, True)
    val_loss, precision, recall, f1 = run_head_epoch(val_data, False)

    print(
        f"Epoch {epoch}/5 | train_loss={train_loss:.4f} "
        f"| val_loss={val_loss:.4f} | P={precision:.4f} "
        f"| R={recall:.4f} | F1={f1:.4f}"
    )

    if f1 > best_f1:
        best_f1 = f1
        torch.save({
            "m4_model": m4_model.state_dict(),
            "boundary_head": boundary_head.state_dict(),
            "labels": labels,
            "val_precision": precision,
            "val_recall": recall,
            "val_f1": f1,
            "epoch": epoch
        }, save_path)
        print("Saved best checkpoint:", save_path)

print("Best validation F1:", best_f1)

Training:   0%|          | 0/583 [00:00<?, ?it/s]

Validation:   0%|          | 0/52 [00:00<?, ?it/s]

Epoch 1/5 | train_loss=0.0412 | val_loss=0.0297 | P=0.0966 | R=0.3103 | F1=0.1474
Saved best checkpoint: /kaggle/working/m4_character_unforzen.pt
Best validation F1: 0.1473869641808573


In [104]:
import json
import torch
from tqdm.auto import tqdm

# Load the best head-only checkpoint
ckpt = torch.load(
    "/kaggle/working/m4_character_unforzen.pt",
    map_location=device,
    weights_only=False
)
boundary_head.load_state_dict(ckpt["boundary_head"])
boundary_head.to(device).eval()
m4_model.eval()

test_predictions_v1 = {}

with torch.no_grad():
    for doc_id, text in tqdm(fold3_text.items(), desc="M4 frozen predictions", unit="doc"):
        chunks = make_document_chunks(
            text, tokenizer,
            chunk_size=CHUNK_SIZE,
            overlap=CHUNK_OVERLAP
        )
        entities = set()

        for item in chunks:
            ids = torch.tensor(
                [item["input_ids"]], dtype=torch.long, device=device
            )
            hidden = m4_model.get_hidden_states(ids)
            start_logits, end_logits = boundary_head(
                hidden, item["text"], item["offsets"]
            )
            spans = decode_entities(
                start_logits, end_logits, len(item["text"]), labels
            )

            for start, end, label in spans:
                gs = item["char_start"] + start
                ge = item["char_start"] + end
                if 0 <= gs < ge <= len(text):
                    entities.add((gs, ge, label))

        test_predictions_v1[doc_id] = [
            [s, e, lab] for s, e, lab in sorted(entities)
        ]

json_path_v1 = "/kaggle/working/prediction_M4_unfrozen.json"
with open(json_path_v1, "w", encoding="utf-8") as f:
    json.dump(test_predictions_v1, f, ensure_ascii=False)

print("Documents:", len(test_predictions_v1))
print("Predicted entities:", sum(len(v) for v in test_predictions_v1.values()))
print("Saved:", json_path_v1)

M4 frozen predictions:   0%|          | 0/35 [00:00<?, ?doc/s]

Documents: 35
Predicted entities: 10901
Saved: /kaggle/working/prediction_M4_unfrozen.json


In [105]:
import os
import zipfile

json_path = "/kaggle/working/prediction_M4_unfrozen.json"
zip_path = "/kaggle/working/submission_M4_2.zip"

assert os.path.exists(json_path), f"Missing predictions: {json_path}"

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    z.write(json_path, arcname="prediction.json")

with zipfile.ZipFile(zip_path, "r") as z:
    print("ZIP contents:", z.namelist())
    assert "prediction.json" in z.namelist()

print("Submission ready:", zip_path)
print("Size:", round(os.path.getsize(zip_path) / 1024, 2), "KB")

ZIP contents: ['prediction.json']
Submission ready: /kaggle/working/submission_M4_2.zip
Size: 61.27 KB


Phase 3


In [12]:
from pathlib import Path
from huggingface_hub import snapshot_download

REPO_ID = "Exploration-Lab/CS-781-Capstone"
DATA_DIR = Path("/kaggle/working/phase3_data/Phase_3_RR")

DATA_DIR.mkdir(parents=True, exist_ok=True)

download_path = snapshot_download(
    repo_id=REPO_ID,
    repo_type="dataset",
    allow_patterns=["Phase_3_RR/**"],
    local_dir="/kaggle/working/phase3_data"
)

print("Download completed.")
print("Dataset directory:", DATA_DIR)
print("Directory exists:", DATA_DIR.exists())

if DATA_DIR.exists():
    for path in sorted(DATA_DIR.rglob("*")):
        if path.is_file():
            print(
                path.relative_to(DATA_DIR),
                "|",
                round(path.stat().st_size / 1024, 1),
                "KB"
            )

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 7 files:   0%|          | 0/7 [00:00<?, ?it/s]

Download completed.
Dataset directory: /kaggle/working/phase3_data/Phase_3_RR
Directory exists: True
Test/CL_test.json | 274.0 KB
Test/IT_test.json | 189.8 KB
Train/CL_dev.json | 328.4 KB
Train/CL_train.json | 2402.6 KB
Train/IT_dev.json | 161.5 KB
Train/IT_train.json | 1474.0 KB
Train/label_vocab.json | 0.3 KB


In [13]:
import json
from pathlib import Path
from collections import Counter

DATA_DIR = Path("/kaggle/working/phase3_data/Phase_3_RR")
TRAIN_DIR = DATA_DIR / "Train"
TEST_DIR = DATA_DIR / "Test"

with open(TRAIN_DIR / "label_vocab.json", "r", encoding="utf-8") as f:
    label_vocab = json.load(f)

with open(TRAIN_DIR / "CL_train.json", "r", encoding="utf-8") as f:
    cl_train_docs = json.load(f)

with open(TRAIN_DIR / "CL_dev.json", "r", encoding="utf-8") as f:
    cl_dev_docs = json.load(f)

with open(TRAIN_DIR / "IT_train.json", "r", encoding="utf-8") as f:
    it_train_docs = json.load(f)

with open(TRAIN_DIR / "IT_dev.json", "r", encoding="utf-8") as f:
    it_dev_docs = json.load(f)

with open(TEST_DIR / "CL_test.json", "r", encoding="utf-8") as f:
    cl_test_docs = json.load(f)

with open(TEST_DIR / "IT_test.json", "r", encoding="utf-8") as f:
    it_test_docs = json.load(f)

print("Label vocabulary size:", len(label_vocab))
print("Label vocabulary:", label_vocab)

datasets = {
    "CL train": cl_train_docs,
    "CL dev": cl_dev_docs,
    "IT train": it_train_docs,
    "IT dev": it_dev_docs,
    "CL test": cl_test_docs,
    "IT test": it_test_docs
}

print("\n=== Document counts ===")
for name, docs in datasets.items():
    print(f"{name}: {len(docs)}")

print("\n=== JSON structure ===")
for name, docs in datasets.items():
    if len(docs) > 0:
        print(f"\n{name}:")
        print("First document type:", type(docs[0]).__name__)
        print("First document sample:", repr(docs[0])[:600])
        break

Label vocabulary size: 13
Label vocabulary: {'Fact': 0, 'Issue': 1, 'ArgumentPetitioner': 2, 'ArgumentRespondent': 3, 'Statute': 4, 'Dissent': 5, 'PrecedentReliedUpon': 6, 'PrecedentNotReliedUpon': 7, 'PrecedentOverruled': 8, 'RulingByLowerCourt': 9, 'RatioOfTheDecision': 10, 'RulingByPresentCourt': 11, 'None': 12}

=== Document counts ===
CL train: 40
CL dev: 5
IT train: 40
IT dev: 5
CL test: 5
IT test: 5

=== JSON structure ===

CL train:
First document type: dict
First document sample: {'id': 'CCI_All_India_Distillers_Association_vs_Haldyn_Glass_GCO100028COM855733', 'text': ['Consequent upon the repeal of the MRTP Act this complaint has been received by transfer on 04.03.2010 from the office of DGIR, MRTP Commission under Section 66(6) of the Competition Act, 2002.', 'The facts as stated in the complaint, in brief, are: 2 .1 This complaint was filed by the All India Distillers Association against glass bottle manufacturers in India alleging restrictive and monopolistic trade practic

In [20]:
test_sentences = [
    "The appeal is dismissed.",
    "The court relied upon the precedent.",
    ""
]

print("Vocabulary size:", tokenizer.vocab_size)
print("Target vocabulary size:", tokenizer.target_vocab_size)
print("EOS token ID:", tokenizer.eos_id)

for sentence in test_sentences:
    token_ids = tokenizer.encode_text(sentence)

    print("\nSentence:", repr(sentence))
    print("Encoded type:", type(token_ids).__name__)
    print("Token count:", len(token_ids))
    print("First token IDs:", token_ids[:20])

    if len(token_ids) > 0:
        print("Last token ID:", token_ids[-1])

Vocabulary size: 32000
Target vocabulary size: 32000
EOS token ID: 0

Sentence: 'The appeal is dismissed.'
Encoded type: list
Token count: 8
First token IDs: [323, 33, 627, 33, 268, 33, 2050, 0]
Last token ID: 0

Sentence: 'The court relied upon the precedent.'
Encoded type: list
Token count: 12
First token IDs: [323, 33, 704, 33, 2436, 33, 1045, 33, 258, 33, 31611, 0]
Last token ID: 0

Sentence: ''
Encoded type: list
Token count: 1
First token IDs: [0]
Last token ID: 0


In [28]:
import torch
from train_m4 import M4Model

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

checkpoint_path = "/kaggle/working/m4_checkpoints/best.pt"

checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=True
)

model = M4Model()
load_result = model.load_state_dict(checkpoint["model"], strict=True)
model = model.to(device)

print("Device:", device)
print("Checkpoint loaded successfully.")
print("Load result:", load_result)
print("Training step:", checkpoint.get("step"))
print("Best validation loss:", checkpoint.get("best_val_loss"))
print("Model parameter count:", sum(p.numel() for p in model.parameters()))

Device: cuda
Checkpoint loaded successfully.
Load result: <All keys matched successfully>
Training step: 12213
Best validation loss: 1.5915145874023438
Model parameter count: 135792384


In [31]:
import torch
from torch.nn.utils.rnn import pad_sequence

MAX_SEQ_LEN = 256
PAD_ID = tokenizer.eos_id
NUM_CLASSES = len(label_vocab)
BATCH_SIZE = 2

def encode_classification_example(example):
    ids = example["input_ids"][:MAX_SEQ_LEN]

    if len(ids) == 0:
        ids = [PAD_ID]

    return {
        "input_ids": torch.tensor(ids, dtype=torch.long),
        "label": torch.tensor(example["label"], dtype=torch.long),
    }

def collate_classification_batch(examples):
    encoded = [
        encode_classification_example(example)
        for example in examples
    ]

    input_ids = pad_sequence(
        [item["input_ids"] for item in encoded],
        batch_first=True,
        padding_value=PAD_ID,
    )

    labels = torch.stack([item["label"] for item in encoded])

    # True marks real tokens; False marks padding.
    attention_mask = input_ids.ne(PAD_ID)

    return input_ids, attention_mask, labels

batch_examples = train_examples[:BATCH_SIZE]
input_ids, attention_mask, labels = collate_classification_batch(batch_examples)

print("Input shape:", tuple(input_ids.shape))
print("Attention mask shape:", tuple(attention_mask.shape))
print("Labels shape:", tuple(labels.shape))
print("Labels:", labels.tolist())
print("Real-token counts:", attention_mask.sum(dim=1).tolist())
print("Maximum sequence length:", MAX_SEQ_LEN)

with torch.no_grad():
    logits = model(input_ids.to(device))

print("\nRaw M4 logits shape:", tuple(logits.shape))
print("Expected last dimension:", tokenizer.vocab_size)

Input shape: (2, 146)
Attention mask shape: (2, 146)
Labels shape: (2,)
Labels: [0, 0]
Real-token counts: [69, 145]
Maximum sequence length: 256

Raw M4 logits shape: (2, 146, 32000)
Expected last dimension: 32000


In [33]:
import torch
from train_m4 import M4Model

def forward_hidden(self, x):
    x = self.token_embedding(x)
    x = self.dropout(x)

    for block in self.blocks:
        x = block(x)

    return self.norm(x)

M4Model.forward_hidden = forward_hidden

with torch.no_grad():
    hidden = model.forward_hidden(input_ids.to(device))

print("Hidden-state shape:", tuple(hidden.shape))

Hidden-state shape: (2, 146, 768)


In [34]:
import torch
import torch.nn as nn
from train_m4 import M4Model

class M4Classifier(nn.Module):
    def __init__(self, backbone, num_classes, dropout=0.1):
        super().__init__()

        self.backbone = backbone
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(768, num_classes)

    def forward(self, input_ids, lengths):
        hidden = self.backbone.forward_hidden(input_ids)

        positions = torch.arange(
            input_ids.size(1),
            device=input_ids.device
        ).unsqueeze(0)

        mask = positions < lengths.unsqueeze(1)
        mask = mask.unsqueeze(-1).to(hidden.dtype)

        pooled = (hidden * mask).sum(dim=1)
        pooled = pooled / lengths.unsqueeze(1).clamp(min=1)

        logits = self.classifier(self.dropout(pooled))
        return logits

# Reuse the already-loaded M4 backbone.
classifier_model = M4Classifier(
    backbone=model,
    num_classes=len(label_vocab)
).to(device)

# The collator must retain true lengths because EOS has ID 0.
lengths = torch.tensor(
    [len(example["input_ids"][:MAX_SEQ_LEN]) for example in batch_examples],
    dtype=torch.long,
    device=device
)

with torch.no_grad():
    class_logits = classifier_model(
        input_ids.to(device),
        lengths
    )

print("Classification logits shape:", tuple(class_logits.shape))
print("Expected shape:", (len(batch_examples), len(label_vocab)))
print("Finite logits:", bool(torch.isfinite(class_logits).all()))
print("Trainable classifier parameters:",
      sum(p.numel() for p in classifier_model.classifier.parameters()))

Classification logits shape: (2, 13)
Expected shape: (2, 13)
Finite logits: True
Trainable classifier parameters: 9997


In [38]:
import torch
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

MAX_SEQ_LEN = 256
PAD_ID = tokenizer.eos_id
BATCH_SIZE = 4

class RhetoricalRoleDataset(Dataset):
    def __init__(self, examples, max_seq_len=256):
        self.examples = examples
        self.max_seq_len = max_seq_len

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        example = self.examples[idx]

        token_ids = example["input_ids"][:self.max_seq_len]

        if len(token_ids) == 0:
            token_ids = [PAD_ID]

        return (
            torch.tensor(token_ids, dtype=torch.long),
            torch.tensor(example["label"], dtype=torch.long)
        )

def collate_rhetorical_batch(batch):
    sequences = [item[0] for item in batch]
    labels = torch.stack([item[1] for item in batch])

    lengths = torch.tensor(
        [len(sequence) for sequence in sequences],
        dtype=torch.long
    )

    input_ids = pad_sequence(
        sequences,
        batch_first=True,
        padding_value=PAD_ID
    )

    return input_ids, lengths, labels

train_dataset = RhetoricalRoleDataset(train_examples, MAX_SEQ_LEN)
dev_dataset = RhetoricalRoleDataset(dev_examples, MAX_SEQ_LEN)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_rhetorical_batch,
    num_workers=0
)

dev_loader = DataLoader(
    dev_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_rhetorical_batch,
    num_workers=0
)

print("Training batches:", len(train_loader))
print("Development batches:", len(dev_loader))

batch_input_ids, batch_lengths, batch_labels = next(iter(train_loader))

print("Input shape:", tuple(batch_input_ids.shape))
print("Lengths:", batch_lengths.tolist())
print("Labels:", batch_labels.tolist())
print("Maximum batch length:", int(batch_lengths.max()))

assert batch_input_ids.shape[0] == batch_labels.shape[0]
assert len(batch_lengths) == len(batch_labels)
assert int(batch_lengths.max()) <= MAX_SEQ_LEN


Training batches: 4212
Development batches: 536
Input shape: (4, 221)
Lengths: [137, 13, 221, 77]
Labels: [2, 0, 10, 10]
Maximum batch length: 221
Data loader validation passed.


In [48]:
from pathlib import Path

module_source = r'''
from collections import Counter, defaultdict
import heapq
import re
from tqdm.auto import tqdm

''' + r'''
''' + '''class ByteLevelBPE:
    SPECIALS=["<eos>"]

    def __init__(self,vocab_size=32000):
        self.target_vocab_size=vocab_size
        self._merge_id_pairs=[]
        self.merges=[]
        self._token_bytes=[]
        self._bytes_to_id={}
        self._merge_ranks={}
        self._merge_to_new_id={}
        self.itos=[]
        self.stoi={}
        self.vocab_size=0
        self.eos_id=0

    @staticmethod
    def _get_word_pairs(symbols):
        return Counter(zip(symbols,symbols[1:]))

    @staticmethod
    def _merge_word(symbols,pair,new_id):
        left,right=pair
        merged=[]
        i=0
        while i<len(symbols):
            if i+1<len(symbols) and symbols[i]==left and symbols[i+1]==right:
                merged.append(new_id)
                i+=2
            else:
                merged.append(symbols[i])
                i+=1
        return merged

    def train_from_word_counts(self,word_freqs):
        self._token_bytes=[(byte,) for byte in range(256)]
        words=[]
        freqs=[]
        for word,freq in tqdm(word_freqs.items(),desc="Preparing BPE corpus"):
            symbols=list(word.encode("utf-8"))
            if len(symbols)>=2:
                words.append(symbols)
                freqs.append(freq)

        pair_counts=Counter()
        pair_to_words=defaultdict(set)
        for word_idx,symbols in tqdm(enumerate(words),total=len(words),desc="Building pair statistics"):
            word_freq=freqs[word_idx]
            for pair,occurrences in self._get_word_pairs(symbols).items():
                pair_counts[pair]+=occurrences*word_freq
                pair_to_words[pair].add(word_idx)

        heap=[]
        for pair,count in pair_counts.items():
            left,right=pair
            heap.append((-count,self._token_bytes[left],self._token_bytes[right],pair))
        heapq.heapify(heap)
        max_merges=self.target_vocab_size-256-len(self.SPECIALS)

        for _ in tqdm(range(max_merges),desc="Learning BPE merges"):
            best_pair=None
            while heap:
                neg_count,_,_,pair=heapq.heappop(heap)
                count=pair_counts.get(pair,0)
                if count==-neg_count and count>1:
                    best_pair=pair
                    break
            if best_pair is None:
                break

            left,right=best_pair
            new_id=len(self._token_bytes)
            self._token_bytes.append(self._token_bytes[left]+self._token_bytes[right])
            self._merge_id_pairs.append(best_pair)
            self.merges.append((self._token_bytes[left],self._token_bytes[right]))
            affected_words=list(pair_to_words.get(best_pair,set()))
            changed_pairs=set()

            for word_idx in affected_words:
                symbols=words[word_idx]
                word_freq=freqs[word_idx]
                new_symbols=[]
                i=0
                while i<len(symbols):
                    if i+1<len(symbols) and symbols[i]==left and symbols[i+1]==right:
                        new_symbols.append(new_id)
                        i+=2
                    else:
                        new_symbols.append(symbols[i])
                        i+=1

                if new_symbols==symbols:
                    continue

                old_pairs=self._get_word_pairs(symbols)
                new_pairs=self._get_word_pairs(new_symbols)

                for pair,occurrences in old_pairs.items():
                    pair_counts[pair]-=occurrences*word_freq
                    changed_pairs.add(pair)

                for pair,occurrences in new_pairs.items():
                    pair_counts[pair]+=occurrences*word_freq
                    changed_pairs.add(pair)

                for pair in old_pairs:
                    if pair not in new_pairs:
                        pair_to_words[pair].discard(word_idx)
                for pair in new_pairs:
                    pair_to_words[pair].add(word_idx)

                words[word_idx]=new_symbols

            pair_counts.pop(best_pair,None)
            pair_to_words.pop(best_pair,None)

            for pair in changed_pairs:
                count=pair_counts.get(pair,0)
                if count<=0:
                    pair_counts.pop(pair,None)
                    pair_to_words.pop(pair,None)
                else:
                    left_pair,right_pair=pair
                    heapq.heappush(
                        heap,
                        (-count,self._token_bytes[left_pair],
                         self._token_bytes[right_pair],pair)
                    )

        self._bytes_to_id={token_bytes:idx for idx,token_bytes in enumerate(self._token_bytes)}
        self._merge_ranks={pair:rank for rank,pair in enumerate(self._merge_id_pairs)}
        self._merge_to_new_id={pair:256+rank for rank,pair in enumerate(self._merge_id_pairs)}
        self.itos=self.SPECIALS.copy()
        self.itos.extend(self._token_bytes)
        self.stoi={token:i for i,token in enumerate(self.itos)}
        self.eos_id=0
        self.vocab_size=len(self.itos)
        print("Learned merges:",len(self.merges))
        print("Final vocabulary size:",self.vocab_size)
        return self

    def encode_word(self,word):
        symbols=[byte for byte in word.encode("utf-8")]
        while len(symbols)>=2:
            best_rank=None
            best_pair=None
            for i in range(len(symbols)-1):
                pair=(symbols[i],symbols[i+1])
                rank=self._merge_ranks.get(pair)
                if rank is not None and (best_rank is None or rank<best_rank):
                    best_rank=rank
                    best_pair=pair
            if best_pair is None:
                break
            symbols=self._merge_word(symbols,best_pair,self._merge_to_new_id[best_pair])
        return [token_id+1 for token_id in symbols]

    def encode_text(self,text,add_eos=True):
        tokens=[]
        for part in re.findall(r"\\s+|\\S+",text):
            tokens.extend(self.encode_word(part))
        if add_eos:
            tokens.append(self.eos_id)
        return tokens
'''

module_path = Path("/kaggle/working/legal_tokenizer.py")
module_path.write_text(module_source, encoding="utf-8")

print("Created:", module_path)
print("Size:", module_path.stat().st_size, "bytes")

Created: /kaggle/working/legal_tokenizer.py
Size: 6199 bytes


In [55]:
import json
import os
from collections import Counter

base = "/kaggle/working/phase3_data/Phase_3_RR/Train"

train_counts = Counter()
dev_counts = Counter()
train_sentences = 0
dev_sentences = 0

for name in ["CL_train.json", "IT_train.json"]:
    with open(os.path.join(base, name), "r", encoding="utf-8") as f:
        docs = json.load(f)

    for doc in docs:
        assert len(doc["text"]) == len(doc["labels"]), (
            name, doc["id"], len(doc["text"]), len(doc["labels"])
        )
        train_sentences += len(doc["text"])
        train_counts.update(doc["labels"])

for name in ["CL_dev.json", "IT_dev.json"]:
    with open(os.path.join(base, name), "r", encoding="utf-8") as f:
        docs = json.load(f)

    for doc in docs:
        assert len(doc["text"]) == len(doc["labels"]), (
            name, doc["id"], len(doc["text"]), len(doc["labels"])
        )
        dev_sentences += len(doc["text"])
        dev_counts.update(doc["labels"])

print("Train sentences:", train_sentences)
print("Dev sentences:", dev_sentences)
print("Train label counts:")
for label, count in train_counts.most_common():
    print(f"{label}: {count}")
print("\nDev label counts:")
for label, count in dev_counts.most_common():
    print(f"{label}: {count}")

Train sentences: 16845
Dev sentences: 2142
Train label counts:
RatioOfTheDecision: 4301
Fact: 4098
PrecedentReliedUpon: 1945
ArgumentPetitioner: 1613
ArgumentRespondent: 1348
Statute: 1099
RulingByLowerCourt: 874
PrecedentNotReliedUpon: 569
RulingByPresentCourt: 307
None: 297
Dissent: 284
Issue: 107
PrecedentOverruled: 3

Dev label counts:
Fact: 444
RatioOfTheDecision: 403
ArgumentPetitioner: 400
ArgumentRespondent: 272
PrecedentReliedUpon: 254
Statute: 120
RulingByLowerCourt: 102
PrecedentNotReliedUpon: 50
RulingByPresentCourt: 49
None: 33
Issue: 15


In [56]:
from pathlib import Path

script = r'''
import os
import sys
import json
import math
import pickle
import random
import argparse
from collections import Counter

import numpy as np
import torch
import torch.nn as nn
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import Dataset, DataLoader, DistributedSampler
from torch.nn.utils.rnn import pad_sequence
from sklearn.metrics import accuracy_score, f1_score, classification_report

from legal_tokenizer import ByteLevelBPE
sys.modules["__main__"].ByteLevelBPE = ByteLevelBPE

import train_m4


DATA_DIR = "/kaggle/working/phase3_data/Phase_3_RR"
TOKENIZER_PATH = "/kaggle/input/datasets/spikie/capstone-split/legal_bpe_32k.pkl"
CHECKPOINT_PATH = "/kaggle/working/m4_checkpoints/best.pt"
OUTPUT_DIR = "/kaggle/working/m4_rhetorical_ddp"
MAX_SEQ_LEN = 256
NUM_CLASSES = 13
PAD_ID = 0
BATCH_SIZE = 4
EPOCHS = 5
LR_HEAD = 1e-3
LR_BACKBONE = 1e-5
NUM_WORKERS = 2

LABEL_NAMES = [
    "Fact", "Issue", "ArgumentPetitioner", "ArgumentRespondent",
    "Statute", "Dissent", "PrecedentReliedUpon",
    "PrecedentNotReliedUpon", "PrecedentOverruled",
    "RulingByLowerCourt", "RatioOfTheDecision",
    "RulingByPresentCourt", "None"
]
LABEL_TO_ID = {name: i for i, name in enumerate(LABEL_NAMES)}


def setup():
    local_rank = int(os.environ["LOCAL_RANK"])
    rank = int(os.environ["RANK"])
    world_size = int(os.environ["WORLD_SIZE"])
    torch.cuda.set_device(local_rank)
    dist.init_process_group(backend="nccl")
    return local_rank, rank, world_size


def cleanup():
    dist.destroy_process_group()


def load_docs(filename):
    path = os.path.join(DATA_DIR, "Train", filename)
    with open(path, "r", encoding="utf-8") as f:
        docs = json.load(f)

    examples = []
    for doc in docs:
        if len(doc["text"]) != len(doc["labels"]):
            raise ValueError("Sentence-label mismatch in " + filename + ": " + doc["id"])

        for sentence, label in zip(doc["text"], doc["labels"]):
            if label not in LABEL_TO_ID:
                raise ValueError("Unknown label: " + str(label))
            examples.append((sentence, LABEL_TO_ID[label]))

    return examples


class SentenceDataset(Dataset):
    def __init__(self, examples, tokenizer):
        self.examples = []
        for text, label in examples:
            ids = tokenizer.encode_text(text, add_eos=True)
            ids = ids[:MAX_SEQ_LEN]
            if len(ids) == 0:
                ids = [PAD_ID]
            self.examples.append((ids, label))

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        ids, label = self.examples[idx]
        return torch.tensor(ids, dtype=torch.long), label


def collate_batch(batch):
    sequences = [item[0] for item in batch]
    labels = torch.tensor([item[1] for item in batch], dtype=torch.long)
    lengths = torch.tensor([len(x) for x in sequences], dtype=torch.long)
    input_ids = pad_sequence(sequences, batch_first=True, padding_value=PAD_ID)
    return input_ids, lengths, labels


class M4Classifier(nn.Module):
    def __init__(self, backbone, num_classes):
        super().__init__()
        self.backbone = backbone
        self.dropout = nn.Dropout(0.1)
        self.classifier = nn.Linear(768, num_classes)

    def forward(self, input_ids, lengths):
        x = self.backbone.token_embedding(input_ids)
        x = self.backbone.dropout(x)

        for block in self.backbone.blocks:
            x = block(x)

        x = self.backbone.norm(x)

        positions = torch.arange(x.size(1), device=x.device).unsqueeze(0)
        mask = positions < lengths.unsqueeze(1)
        mask = mask.unsqueeze(-1).to(x.dtype)

        pooled = (x * mask).sum(dim=1)
        pooled = pooled / lengths.unsqueeze(1).clamp(min=1).to(x.dtype)
        return self.classifier(self.dropout(pooled))


def evaluate(model, loader, device):
    model.eval()
    all_preds = []
    all_labels = []
    total_loss = 0.0
    total_count = 0
    criterion = nn.CrossEntropyLoss()

    with torch.no_grad():
        for input_ids, lengths, labels in loader:
            input_ids = input_ids.to(device)
            lengths = lengths.to(device)
            labels = labels.to(device)

            logits = model(input_ids, lengths)
            loss = criterion(logits, labels)

            total_loss += loss.item() * labels.size(0)
            total_count += labels.size(0)
            all_preds.extend(logits.argmax(dim=1).cpu().tolist())
            all_labels.extend(labels.cpu().tolist())

    stats = torch.tensor([total_loss, total_count], dtype=torch.float64, device=device)
    dist.all_reduce(stats, op=dist.ReduceOp.SUM)

    gathered_preds = [None for _ in range(dist.get_world_size())]
    gathered_labels = [None for _ in range(dist.get_world_size())]
    dist.all_gather_object(gathered_preds, all_preds)
    dist.all_gather_object(gathered_labels, all_labels)

    preds = []
    labels = []
    for rank_preds, rank_labels in zip(gathered_preds, gathered_labels):
        preds.extend(rank_preds)
        labels.extend(rank_labels)

    # DistributedSampler may pad validation indices, so remove duplicates by
    # evaluating validation on rank 0 only in the main loop instead.
    return stats[0].item() / max(stats[1].item(), 1), preds, labels


def main():
    local_rank, rank, world_size = setup()
    device = torch.device("cuda", local_rank)

    torch.manual_seed(42 + rank)
    random.seed(42 + rank)
    np.random.seed(42 + rank)

    os.makedirs(OUTPUT_DIR, exist_ok=True)

    if rank == 0:
        print("Loading tokenizer and datasets...", flush=True)

    with open(TOKENIZER_PATH, "rb") as f:
        tokenizer = pickle.load(f)

    train_examples = (
        load_docs("CL_train.json") + load_docs("IT_train.json")
    )
    dev_examples = (
        load_docs("CL_dev.json") + load_docs("IT_dev.json")
    )

    train_dataset = SentenceDataset(train_examples, tokenizer)
    dev_dataset = SentenceDataset(dev_examples, tokenizer)

    train_sampler = DistributedSampler(
        train_dataset, num_replicas=world_size, rank=rank, shuffle=True
    )
    train_loader = DataLoader(
        train_dataset, batch_size=BATCH_SIZE, sampler=train_sampler,
        num_workers=NUM_WORKERS, pin_memory=True, collate_fn=collate_batch
    )

    # Validation only on rank 0 to avoid duplicate examples and metric distortion.
    dev_loader = None
    if rank == 0:
        dev_loader = DataLoader(
            dev_dataset, batch_size=BATCH_SIZE, shuffle=False,
            num_workers=NUM_WORKERS, pin_memory=True, collate_fn=collate_batch
        )

    checkpoint = torch.load(CHECKPOINT_PATH, map_location="cpu", weights_only=False)
    backbone = train_m4.M4Model()
    backbone.load_state_dict(checkpoint["model"], strict=True)

    model = M4Classifier(backbone, NUM_CLASSES).to(device)

    # Fine-tune the backbone and the classification head.
    for param in model.parameters():
        param.requires_grad = True

    model = DDP(model, device_ids=[local_rank], output_device=local_rank,
                find_unused_parameters=False)

    counts = Counter(label for _, label in train_examples)
    weights = []
    for i in range(NUM_CLASSES):
        weights.append(1.0 / max(counts.get(i, 0), 1) ** 0.5)
    class_weights = torch.tensor(weights, dtype=torch.float32, device=device)
    class_weights = class_weights / class_weights.mean()

    criterion = nn.CrossEntropyLoss(weight=class_weights)
    optimizer = torch.optim.AdamW(
        [
            {"params": model.module.backbone.parameters(), "lr": LR_BACKBONE},
            {"params": model.module.classifier.parameters(), "lr": LR_HEAD},
        ],
        weight_decay=0.01
    )

    best_macro_f1 = -1.0

    if rank == 0:
        print("Train examples:", len(train_dataset), flush=True)
        print("Dev examples:", len(dev_dataset), flush=True)
        print("Starting DDP training on", world_size, "GPUs", flush=True)

    for epoch in range(EPOCHS):
        model.train()
        train_sampler.set_epoch(epoch)
        total_loss = 0.0
        total_count = 0

        for input_ids, lengths, labels in train_loader:
            input_ids = input_ids.to(device, non_blocking=True)
            lengths = lengths.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)
            logits = model(input_ids, lengths)
            loss = criterion(logits, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            total_loss += loss.item() * labels.size(0)
            total_count += labels.size(0)

        stats = torch.tensor([total_loss, total_count], dtype=torch.float64, device=device)
        dist.all_reduce(stats, op=dist.ReduceOp.SUM)
        train_loss = stats[0].item() / max(stats[1].item(), 1)

        # All ranks must reach this barrier before rank 0 evaluates.
        dist.barrier()

        if rank == 0:
            val_loss, preds, labels = evaluate_rank0(model.module, dev_loader, device)
            macro_f1 = f1_score(labels, preds, average="macro", zero_division=0)
            accuracy = accuracy_score(labels, preds)

            print(
                f"Epoch {epoch + 1}/{EPOCHS} | "
                f"train_loss={train_loss:.4f} | val_loss={val_loss:.4f} | "
                f"val_accuracy={accuracy:.4f} | val_macro_f1={macro_f1:.4f}",
                flush=True
            )

            if macro_f1 > best_macro_f1:
                best_macro_f1 = macro_f1
                torch.save(
                    {
                        "model": model.module.state_dict(),
                        "epoch": epoch + 1,
                        "macro_f1": macro_f1,
                        "accuracy": accuracy,
                        "label_names": LABEL_NAMES,
                    },
                    os.path.join(OUTPUT_DIR, "best.pt")
                )
                print("Saved best checkpoint.", flush=True)

            with open(os.path.join(OUTPUT_DIR, "history.txt"), "a") as f:
                f.write(
                    f"{epoch + 1},{train_loss:.6f},{val_loss:.6f},"
                    f"{accuracy:.6f},{macro_f1:.6f}\n"
                )

        dist.barrier()

    if rank == 0:
        print("Training finished. Best macro-F1:", best_macro_f1, flush=True)

    cleanup()


def evaluate_rank0(model, loader, device):
    model.eval()
    criterion = nn.CrossEntropyLoss()
    all_preds = []
    all_labels = []
    total_loss = 0.0
    total_count = 0

    with torch.no_grad():
        for input_ids, lengths, labels in loader:
            input_ids = input_ids.to(device)
            lengths = lengths.to(device)
            labels = labels.to(device)

            logits = model(input_ids, lengths)
            loss = criterion(logits, labels)
            total_loss += loss.item() * labels.size(0)
            total_count += labels.size(0)
            all_preds.extend(logits.argmax(dim=1).cpu().tolist())
            all_labels.extend(labels.cpu().tolist())

    return total_loss / max(total_count, 1), all_preds, all_labels


if __name__ == "__main__":
    main()
'''

path = Path("/kaggle/working/train_m4_rhetorical_ddp.py")
path.write_text(script, encoding="utf-8")
print("Created:", path)
print("Size:", path.stat().st_size, "bytes")

Created: /kaggle/working/train_m4_rhetorical_ddp.py
Size: 11240 bytes


In [59]:
from pathlib import Path

path = Path("/kaggle/working/train_m4_rhetorical_ddp.py")
script = path.read_text(encoding="utf-8")

# Add tqdm import
script = script.replace(
    "from torch.nn.utils.rnn import pad_sequence\n",
    "from torch.nn.utils.rnn import pad_sequence\nfrom tqdm.auto import tqdm\n"
)

# Replace the training batch loop with a tqdm progress bar.
old = '''        for input_ids, lengths, labels in train_loader:
            input_ids = input_ids.to(device, non_blocking=True)
            lengths = lengths.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)
            logits = model(input_ids, lengths)
            loss = criterion(logits, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            total_loss += loss.item() * labels.size(0)
            total_count += labels.size(0)
'''

new = '''        progress = tqdm(
            train_loader,
            desc=f"Epoch {epoch + 1}/{EPOCHS}",
            disable=(rank != 0),
            dynamic_ncols=True
        )

        for input_ids, lengths, labels in progress:
            input_ids = input_ids.to(device, non_blocking=True)
            lengths = lengths.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)
            logits = model(input_ids, lengths)
            loss = criterion(logits, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            total_loss += loss.item() * labels.size(0)
            total_count += labels.size(0)

            progress.set_postfix(
                loss=f"{loss.item():.4f}",
                avg_loss=f"{total_loss / max(total_count, 1):.4f}"
            )
'''

if old not in script:
    raise RuntimeError("Training loop did not match; script was not modified.")

script = script.replace(old, new, 1)
path.write_text(script, encoding="utf-8")

print("Updated:", path)
print("tqdm import added:", "from tqdm.auto import tqdm" in script)
print("Progress bar added:", "progress.set_postfix(" in script)

Updated: /kaggle/working/train_m4_rhetorical_ddp.py
tqdm import added: True
Progress bar added: True


In [58]:
import subprocess
import sys

script = r'''
import torch
from train_m4 import M4Model
from train_m4_rhetorical_ddp import M4Classifier

device = torch.device("cuda:0")

ckpt = torch.load(
    "/kaggle/working/m4_checkpoints/best.pt",
    map_location="cpu",
    weights_only=False
)

backbone = M4Model()
backbone.load_state_dict(ckpt["model"], strict=True)

model = M4Classifier(backbone, 13).to(device)
model.train()

x = torch.randint(1, 32000, (2, 32), device=device)
lengths = torch.tensor([32, 25], device=device)
labels = torch.tensor([0, 10], device=device)

logits = model(x, lengths)
loss = torch.nn.functional.cross_entropy(logits, labels)
loss.backward()

print("Logits shape:", tuple(logits.shape))
print("Loss:", float(loss.item()))
print("Backward pass: OK")
print("Trainable parameters:", sum(p.numel() for p in model.parameters() if p.requires_grad))
'''

result = subprocess.run(
    [sys.executable, "-c", script],
    cwd="/kaggle/working",
    capture_output=True,
    text=True
)

print(result.stdout)
if result.returncode != 0:
    print("ERROR:\n", result.stderr)
else:
    print("Smoke test passed.")

Logits shape: (2, 13)
Loss: 2.1933395862579346
Backward pass: OK
Trainable parameters: 135802381

Smoke test passed.


In [61]:
import subprocess
import sys

cmd = [
    "torchrun",
    "--standalone",
    "--nproc_per_node=2",
    "/kaggle/working/train_m4_rhetorical_ddp.py"
]

print("Launching M4 rhetorical-role training on 2 GPUs...")
subprocess.run(cmd, cwd="/kaggle/working", check=True)

Launching M4 rhetorical-role training on 2 GPUs...


W1016 16:56:04.492000 263 torch/distributed/run.py:851] 
W1016 16:56:04.492000 263 torch/distributed/run.py:851] *****************************************
W1016 16:56:04.492000 263 torch/distributed/run.py:851] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W1016 16:56:04.492000 263 torch/distributed/run.py:851] *****************************************


Loading tokenizer and datasets...
Train examples: 16845
Dev examples: 2142
Starting DDP training on 2 GPUs


Epoch 1/5: 100%|██████████| 2106/2106 [09:34<00:00,  3.66it/s, avg_loss=1.6234, loss=0.4419]
/usr/local/lib/python3.13/dist-packages/torch/distributed/c10d_logger.py:83: UserWarning: barrier(): using the device under current context. You can specify `device_id` in `init_process_group` to mute this warning.
  return func(*args, **kwargs)


Epoch 1/5 | train_loss=1.6199 | val_loss=1.7977 | val_accuracy=0.4076 | val_macro_f1=0.3211
Saved best checkpoint.


/usr/local/lib/python3.13/dist-packages/torch/distributed/c10d_logger.py:83: UserWarning: barrier(): using the device under current context. You can specify `device_id` in `init_process_group` to mute this warning.
  return func(*args, **kwargs)
Epoch 2/5: 100%|██████████| 2106/2106 [09:37<00:00,  3.65it/s, avg_loss=1.0128, loss=1.5919]


... (5 outputs left out)

/usr/local/lib/python3.13/dist-packages/torch/distributed/c10d_logger.py:83: UserWarning: barrier(): using the device under current context. You can specify `device_id` in `init_process_group` to mute this warning.
  return func(*args, **kwargs)
Epoch 5/5: 100%|██████████| 2106/2106 [09:38<00:00,  3.64it/s, avg_loss=0.2208, loss=0.6396]


Epoch 5/5 | train_loss=0.2237 | val_loss=5.9977 | val_accuracy=0.4029 | val_macro_f1=0.3389
Saved best checkpoint.
Training finished. Best macro-F1: 0.33891934463214896


/usr/local/lib/python3.13/dist-packages/torch/distributed/c10d_logger.py:83: UserWarning: barrier(): using the device under current context. You can specify `device_id` in `init_process_group` to mute this warning.
  return func(*args, **kwargs)


CompletedProcess(args=['torchrun', '--standalone', '--nproc_per_node=2', '/kaggle/working/train_m4_rhetorical_ddp.py'], returncode=0)

In [67]:
import json
from pathlib import Path
from collections import Counter
from tqdm.auto import tqdm
import torch
from torch.nn.utils.rnn import pad_sequence

TEST_DIR = Path("/kaggle/working/phase3_data/Phase_3_RR/Test")
OUT_DIR = Path("/kaggle/working/m4_rhetorical_ddp")
OUT_PATH = OUT_DIR / "prediction.json"

MAX_SEQ_LEN = 256
PAD_ID = 0
BATCH_SIZE = 4

def encode_test_sentence(sentence):
    # Use the exact custom BPE encoding method used during training.
    ids = tokenizer.encode_text(sentence, add_eos=True)
    ids = list(ids)[:MAX_SEQ_LEN]

    if len(ids) == 0:
        ids = [PAD_ID]

    return ids

predictions = {}
model.eval()

for filename in ["CL_test.json", "IT_test.json"]:
    with open(TEST_DIR / filename, "r", encoding="utf-8") as f:
        test_docs = json.load(f)

    print(f"\nPredicting {filename}: {len(test_docs)} documents")

    for doc_id, sentences in tqdm(test_docs.items(), desc=filename):
        encoded_sentences = [
            encode_test_sentence(sentence) for sentence in sentences
        ]
        doc_predictions = []

        # Batch sentences, padding only to the longest sequence in each batch.
        for start in range(0, len(encoded_sentences), BATCH_SIZE):
            batch_ids = encoded_sentences[start:start + BATCH_SIZE]

            lengths = torch.tensor(
                [len(ids) for ids in batch_ids], dtype=torch.long
            )

            sequences = [
                torch.tensor(ids, dtype=torch.long) for ids in batch_ids
            ]

            input_ids = pad_sequence(
                sequences,
                batch_first=True,
                padding_value=PAD_ID
            ).to(DEVICE)

            lengths_device = lengths.to(DEVICE)

            with torch.inference_mode():
                logits = model(input_ids, lengths_device)
                predicted_ids = logits.argmax(dim=-1).cpu().tolist()

            doc_predictions.extend(
                ID_TO_LABEL[predicted_id] for predicted_id in predicted_ids
            )

        assert len(doc_predictions) == len(sentences), (
            f"Sentence count mismatch for {doc_id}"
        )
        predictions[doc_id] = doc_predictions

# Validate the output against both original test files.
assert len(predictions) == 10, (
    f"Expected 10 documents, found {len(predictions)}"
)

for filename in ["CL_test.json", "IT_test.json"]:
    with open(TEST_DIR / filename, "r", encoding="utf-8") as f:
        source_docs = json.load(f)

    for doc_id, sentences in source_docs.items():
        assert doc_id in predictions, f"Missing document: {doc_id}"
        assert len(predictions[doc_id]) == len(sentences)
        assert all(label in LABEL_NAMES for label in predictions[doc_id])

with open(OUT_PATH, "w", encoding="utf-8") as f:
    json.dump(predictions, f, ensure_ascii=False, indent=2)

print("\nPrediction file:", OUT_PATH)
print("Documents:", len(predictions))
print("Total sentences:", sum(len(v) for v in predictions.values()))
print("Label distribution:")
print(Counter(
    label for labels in predictions.values() for label in labels
))
print("\nSample document:", next(iter(predictions)))
print("Sample predictions:", next(iter(predictions.values()))[:10])
print("Validation passed.")


Predicting CL_test.json: 5 documents


CL_test.json:   0%|          | 0/5 [00:00<?, ?it/s]


Predicting IT_test.json: 5 documents


IT_test.json:   0%|          | 0/5 [00:00<?, ?it/s]


Prediction file: /kaggle/working/m4_rhetorical_ddp/prediction.json
Documents: 10
Total sentences: 2197
Label distribution:
Counter({'RatioOfTheDecision': 784, 'Fact': 433, 'PrecedentReliedUpon': 236, 'ArgumentPetitioner': 176, 'Statute': 167, 'RulingByLowerCourt': 164, 'ArgumentRespondent': 86, 'PrecedentNotReliedUpon': 55, 'RulingByPresentCourt': 43, 'None': 29, 'Dissent': 13, 'Issue': 11})

Sample document: HC_Raghupati_Singhania__vs__Competition_Commission_ofDE201616051616063761COM513489
Sample predictions: ['None', 'Fact', 'RulingByLowerCourt', 'Fact', 'ArgumentPetitioner', 'ArgumentPetitioner', 'ArgumentPetitioner', 'ArgumentPetitioner', 'ArgumentPetitioner', 'RatioOfTheDecision']
Validation passed.


In [69]:
from pathlib import Path
import json
import zipfile

prediction_path = Path(
    "/kaggle/working/m4_rhetorical_ddp/prediction.json"
)
zip_path = Path(
    "/kaggle/working/m4_rhetorical_ddp/submission_M4_P3.zip"
)

with open(prediction_path, "r", encoding="utf-8") as f:
    predictions = json.load(f)

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    zf.write(prediction_path, arcname="prediction.json")

with zipfile.ZipFile(zip_path, "r") as zf:
    print("ZIP contents:", zf.namelist())
    print("ZIP integrity:", zf.testzip())

print("Documents:", len(predictions))
print("Total sentences:", sum(len(v) for v in predictions.values()))
print("Submission ZIP:", zip_path)
print("ZIP size (MB):", round(zip_path.stat().st_size / (1024 ** 2), 3))

ZIP contents: ['prediction.json']
ZIP integrity: None
Documents: 10
Total sentences: 2197
Submission ZIP: /kaggle/working/m4_rhetorical_ddp/submission_M4_P3.zip
ZIP size (MB): 0.002


PHASE 4

In [2]:
import os
import json
from huggingface_hub import hf_hub_download

HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
PHASE4_FILES = {
    "train": "Phase_4_CJPE/Train/train.json",
    "dev": "Phase_4_CJPE/Train/dev.json",
    "test": "Phase_4_CJPE/Test/test.json",
}
phase4_paths = {}
for split, filename in PHASE4_FILES.items():
    print(f"Accessing {split}: {filename}")
    path = hf_hub_download(
        repo_id=HF_DATASET_REPO,
        filename=filename,
        repo_type="dataset",
        token=HF_TOKEN,
    )
    phase4_paths[split] = path
    print("Cached path:", path)

with open(phase4_paths["train"], "r", encoding="utf-8") as f:
    train_data = json.load(f)

with open(phase4_paths["dev"], "r", encoding="utf-8") as f:
    dev_data = json.load(f)

with open(phase4_paths["test"], "r", encoding="utf-8") as f:
    test_data = json.load(f)

print("\nDataset sizes:")
print("Train:", len(train_data))
print("Dev:", len(dev_data))
print("Test:", len(test_data))
print("\nFirst training example keys:", train_data[0].keys())
print("First test example keys:", test_data[0].keys())

Accessing train: Phase_4_CJPE/Train/train.json


Phase_4_CJPE/Train/train.json: reconstructing file:   0%|          |  0.00B /  737MB            

Phase_4_CJPE/Train/train.json: downloading bytes:           |  0.00B            

Cached path: /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/93b0059be756c6c461f3a225de25aabda7d5f84a/Phase_4_CJPE/Train/train.json
Accessing dev: Phase_4_CJPE/Train/dev.json


Phase_4_CJPE/Train/dev.json: reconstructing file:   0%|          |  0.00B / 45.0MB            

Phase_4_CJPE/Train/dev.json: downloading bytes:           |  0.00B            

Cached path: /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/93b0059be756c6c461f3a225de25aabda7d5f84a/Phase_4_CJPE/Train/dev.json
Accessing test: Phase_4_CJPE/Test/test.json


Phase_4_CJPE/Test/test.json: reconstructing file:   0%|          |  0.00B / 35.3MB            

Phase_4_CJPE/Test/test.json: downloading bytes:           |  0.00B            

Cached path: /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/93b0059be756c6c461f3a225de25aabda7d5f84a/Phase_4_CJPE/Test/test.json

Dataset sizes:
Train: 37387
Dev: 1988
Test: 1517

First training example keys: dict_keys(['id', 'text', 'label', 'expert_1', 'expert_2', 'expert_3', 'expert_4', 'expert_5'])
First test example keys: dict_keys(['id', 'text', 'label', 'expert_1', 'expert_2', 'expert_3', 'expert_4', 'expert_5'])


In [3]:
import os
import pickle
import torch
from tqdm.auto import tqdm

OUT = "/kaggle/working/cjpe_m4"
os.makedirs(OUT, exist_ok=True)

with open(
    "/kaggle/input/datasets/spikie/capstone-split/legal_bpe_32k.pkl",
    "rb"
) as f:
    tokenizer = pickle.load(f)

MAX_LEN = 1024

def encode_rows(rows, name, deduplicate=False):
    seen = set()
    ids_list, labels, case_ids = [], [], []

    for row in tqdm(rows, desc=name):
        if deduplicate and row["id"] in seen:
            continue
        seen.add(row["id"])

        ids = tokenizer.encode_text(row["text"])[:MAX_LEN]
        if not ids:
            continue

        ids_list.append(torch.tensor(ids, dtype=torch.long))
        labels.append(int(row["label"]))
        case_ids.append(row["id"])

    return {
        "ids": ids_list,
        "labels": torch.tensor(labels, dtype=torch.long),
        "case_ids": case_ids
    }

train_enc = encode_rows(train_data, "M4 train", True)
dev_enc = encode_rows(dev_data, "M4 dev")

torch.save(train_enc, OUT + "/train.pt")
torch.save(dev_enc, OUT + "/dev.pt")

print("Train:", len(train_enc["ids"]))
print("Dev:", len(dev_enc["ids"]))
print("Saved to:", OUT)

M4 train:   0%|          | 0/37387 [00:00<?, ?it/s]

M4 dev:   0%|          | 0/1988 [00:00<?, ?it/s]

Train: 32305
Dev: 1988
Saved to: /kaggle/working/cjpe_m4


In [15]:

%%writefile /kaggle/working/train_m4_cjpe.py
import os
import random
import numpy as np
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import Dataset, DataLoader, DistributedSampler
from sklearn.metrics import f1_score, precision_score, recall_score

# Import the exact Phase 1 model implementation.
from train_m4 import M4Model


SEED = 42
MAX_LEN = 256
BATCH_SIZE = 2
EPOCHS = 3
LR = 1e-3
NUM_WORKERS = 2

PRETRAINED = "/kaggle/working/m4_checkpoints/best.pt"
TRAIN_FILE = "/kaggle/working/cjpe_m4/train.pt"
DEV_FILE = "/kaggle/working/cjpe_m4/dev.pt"
OUT_DIR = "/kaggle/working/cjpe_m4_results"

os.makedirs(OUT_DIR, exist_ok=True)


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


class CJDataset(Dataset):
    def __init__(self, path):
        self.data = torch.load(
            path, map_location="cpu", weights_only=False
        )

    def __len__(self):
        return len(self.data["labels"])

    def __getitem__(self, idx):
        ids = self.data["ids"][idx][:MAX_LEN]
        label = int(self.data["labels"][idx])
        return torch.as_tensor(ids, dtype=torch.long), label


def collate_fn(batch):
    ids = [item[0] for item in batch]
    labels = torch.tensor(
        [item[1] for item in batch], dtype=torch.long
    )
    lengths = torch.tensor([len(x) for x in ids])

    input_ids = nn.utils.rnn.pad_sequence(
        ids, batch_first=True, padding_value=0
    )

    # Use lengths because token ID 0 is also EOS.
    positions = torch.arange(input_ids.size(1)).unsqueeze(0)
    mask = positions < lengths.unsqueeze(1)

    return input_ids, mask, labels


class M4Classifier(nn.Module):
    def __init__(self, backbone):
        super().__init__()
        self.backbone = backbone

        # Frozen Phase 1 backbone.
        for p in self.backbone.parameters():
            p.requires_grad = False

        self.classifier = nn.Linear(768, 2)

    def forward(self, input_ids, mask):
        # Extract hidden states before the vocabulary LM head.
        x = self.backbone.token_embedding(input_ids)
        x = self.backbone.dropout(x)

        for block in self.backbone.blocks:
            x = block(x)

        hidden = self.backbone.norm(x)
        # hidden shape: [batch, sequence, 768]

        mask_float = mask.unsqueeze(-1).to(hidden.dtype)
        pooled = (hidden * mask_float).sum(dim=1)
        pooled = pooled / mask_float.sum(dim=1).clamp(min=1.0)

        return self.classifier(pooled)


def main():
    set_seed(SEED)

    local_rank = int(os.environ["LOCAL_RANK"])
    rank = int(os.environ["RANK"])
    world_size = int(os.environ["WORLD_SIZE"])

    torch.cuda.set_device(local_rank)
    dist.init_process_group(backend="nccl")
    device = torch.device("cuda", local_rank)

    try:
        train_ds = CJDataset(TRAIN_FILE)
        dev_ds = CJDataset(DEV_FILE)

        train_sampler = DistributedSampler(
            train_ds,
            num_replicas=world_size,
            rank=rank,
            shuffle=True,
            seed=SEED
        )

        train_loader = DataLoader(
            train_ds,
            batch_size=BATCH_SIZE,
            sampler=train_sampler,
            num_workers=NUM_WORKERS,
            pin_memory=True,
            collate_fn=collate_fn,
            persistent_workers=(NUM_WORKERS > 0)
        )

        dev_loader = None
        if rank == 0:
            dev_loader = DataLoader(
                dev_ds,
                batch_size=4,
                shuffle=False,
                num_workers=NUM_WORKERS,
                pin_memory=True,
                collate_fn=collate_fn,
                persistent_workers=(NUM_WORKERS > 0)
            )

        # Load Phase 1 weights strictly.
        backbone = M4Model()
        ckpt = torch.load(
            PRETRAINED, map_location="cpu", weights_only=False
        )
        backbone.load_state_dict(ckpt["model"], strict=True)

        model = M4Classifier(backbone).to(device)

        trainable = sum(
            p.numel() for p in model.parameters() if p.requires_grad
        )

        if rank == 0:
            print("Train/dev:", len(train_ds), len(dev_ds))
            print("GPUs:", world_size)
            print("Max sequence length:", MAX_LEN)
            print("Trainable parameters:", trainable)
            print("Phase 1 checkpoint step:", ckpt["step"])

        model = DDP(
            model,
            device_ids=[local_rank],
            output_device=local_rank
        )

        optimizer = torch.optim.AdamW(
            model.module.classifier.parameters(),
            lr=LR,
            weight_decay=0.01
        )
        criterion = nn.CrossEntropyLoss()
        scaler = torch.amp.GradScaler("cuda")

        # -------- Smoke test --------
        model.train()
        batch = next(iter(train_loader))
        x, mask, y = [t.to(device) for t in batch]

        optimizer.zero_grad(set_to_none=True)

        with torch.autocast("cuda", dtype=torch.float16):
            logits = model(x, mask)
            loss = criterion(logits, y)

        if logits.shape != (len(y), 2):
            raise RuntimeError("Unexpected classifier output shape.")
        if not torch.isfinite(loss).item():
            raise RuntimeError("Non-finite smoke-test loss.")

        scaler.scale(loss).backward()

        if model.module.classifier.weight.grad is None:
            raise RuntimeError("Classifier gradients are missing.")

        if not torch.isfinite(
            model.module.classifier.weight.grad
        ).all().item():
            raise RuntimeError("Non-finite classifier gradients.")

        optimizer.zero_grad(set_to_none=True)
        del x, mask, y, logits, loss, batch
        torch.cuda.empty_cache()
        dist.barrier()

        if rank == 0:
            print("SMOKE TEST PASSED.", flush=True)

        best_f1 = -1.0

        for epoch in range(EPOCHS):
            train_sampler.set_epoch(epoch)
            model.train()
            total_loss = 0.0
            steps = 0

            progress = tqdm(
                train_loader,
                desc="Epoch {}/{}".format(epoch + 1, EPOCHS),
                disable=(rank != 0)
            )

            for x, mask, y in progress:
                x = x.to(device, non_blocking=True)
                mask = mask.to(device, non_blocking=True)
                y = y.to(device, non_blocking=True)

                optimizer.zero_grad(set_to_none=True)

                with torch.autocast("cuda", dtype=torch.float16):
                    logits = model(x, mask)
                    loss = criterion(logits, y)

                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()

                total_loss += loss.item()
                steps += 1

                if rank == 0:
                    progress.set_postfix(loss=round(loss.item(), 4))

            dist.barrier()

            if rank == 0:
                model.eval()
                all_preds = []
                all_labels = []
                dev_loss = 0.0
                dev_steps = 0

                with torch.inference_mode():
                    for x, mask, y in tqdm(
                        dev_loader, desc="Validation"
                    ):
                        x = x.to(device, non_blocking=True)
                        mask = mask.to(device, non_blocking=True)
                        y = y.to(device, non_blocking=True)

                        with torch.autocast(
                            "cuda", dtype=torch.float16
                        ):
                            logits = model.module(x, mask)
                            loss = criterion(logits, y)

                        dev_loss += loss.item()
                        dev_steps += 1
                        all_preds.extend(
                            logits.argmax(1).cpu().tolist()
                        )
                        all_labels.extend(y.cpu().tolist())

                macro_f1 = f1_score(
                    all_labels, all_preds,
                    average="macro", zero_division=0
                )
                precision = precision_score(
                    all_labels, all_preds, zero_division=0
                )
                recall = recall_score(
                    all_labels, all_preds, zero_division=0
                )

                print(
                    "Epoch {} | train_loss={:.4f} | dev_loss={:.4f} "
                    "| macro_f1={:.4f} | precision={:.4f} | recall={:.4f}"
                    .format(
                        epoch + 1,
                        total_loss / max(steps, 1),
                        dev_loss / max(dev_steps, 1),
                        macro_f1, precision, recall
                    ),
                    flush=True
                )

                saved = {
                    "epoch": epoch + 1,
                    "model": model.module.state_dict(),
                    "optimizer": optimizer.state_dict(),
                    "macro_f1": macro_f1,
                    "max_len": MAX_LEN
                }

                torch.save(saved, OUT_DIR + "/latest.pt")

                if macro_f1 > best_f1:
                    best_f1 = macro_f1
                    torch.save(saved, OUT_DIR + "/best.pt")
                    print("Saved best checkpoint.", flush=True)

            dist.barrier()

    finally:
        dist.destroy_process_group()


if __name__ == "__main__":
    main()


Writing /kaggle/working/train_m4_cjpe.py


In [22]:
import sys
import torch

common_dir = "/kaggle/input/datasets/spikie/capstone-split"
sys.path.insert(0, common_dir)

from legal_m1 import GPTModel

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

ckpt_path = f"{common_dir}/latest.pt"
ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)

cfg = ckpt["config"]

teacher = GPTModel(
    vocab_size=cfg["vocab_size"],
    d_model=cfg["d_model"],
    n_layers=cfg["n_layers"],
    n_heads=cfg["n_heads"],
    ffn_dim=cfg["ffn_dim"],
    dropout=cfg["dropout"]
)

result = teacher.load_state_dict(ckpt["model"], strict=True)
teacher = teacher.to(device)
teacher.eval()

for param in teacher.parameters():
    param.requires_grad = False

print("M1 Phase 1 teacher loaded successfully.")
print("Device:", device)
print("Parameters:", sum(p.numel() for p in teacher.parameters()))
print("Strict loading:", result)
print("Training tokens seen:", ckpt["tokens_seen"])

M1 Phase 1 teacher loaded successfully.
Device: cuda
Parameters: 109612800
Strict loading: <All keys matched successfully>
Training tokens seen: 750000330


In [27]:

import torch
import torch.nn as nn
import torch.nn.functional as F

# Freeze pretrained M1 backbone
teacher.eval()
for p in teacher.parameters():
    p.requires_grad = False

class M1CJPEClassifier(nn.Module):
    def __init__(self, backbone, hidden_size=768, num_classes=2):
        super().__init__()
        self.backbone = backbone
        self.dropout = nn.Dropout(0.1)
        self.classifier = nn.Linear(hidden_size, num_classes)

    def forward(self, input_ids, lengths):
        # M1 is frozen; avoid building a backbone autograd graph.
        with torch.no_grad():
            x = self.backbone.token_embedding(input_ids)
            for block in self.backbone.blocks:
                x = block(x)
            x = self.backbone.norm_f(x) if hasattr(self.backbone, "norm_f") else x

        positions = torch.arange(input_ids.size(1), device=input_ids.device)
        mask = (positions.unsqueeze(0) < lengths.unsqueeze(1)).unsqueeze(-1)
        mask = mask.to(x.dtype)

        pooled = (x * mask).sum(dim=1)
        pooled = pooled / lengths.unsqueeze(1).clamp(min=1).to(x.dtype)

        return self.classifier(self.dropout(pooled))

m1_cjpe = M1CJPEClassifier(teacher).to(device)
print("Teacher classifier created.")
print("Trainable parameters:", sum(p.numel() for p in m1_cjpe.parameters() if p.requires_grad))


Teacher classifier created.
Trainable parameters: 1538


In [35]:

import torch
import torch.nn as nn
from tqdm.auto import tqdm
from sklearn.metrics import f1_score, confusion_matrix
import numpy as np

# Reset classifier head
m1_cjpe.classifier.reset_parameters()

# Class-weighted loss based on the full CJPE training distribution
counts = torch.bincount(train_cjpe["labels"])
weights = (counts.sum() / (2.0 * counts.float())).to(device)
criterion_fast = nn.CrossEntropyLoss(weight=weights)

optimizer_fast = torch.optim.AdamW(
    m1_cjpe.classifier.parameters(),
    lr=5e-4,
    weight_decay=0.01
)

for epoch in range(3):
    m1_cjpe.train()
    m1_cjpe.backbone.eval()

    for ids, lengths, labels in tqdm(
        train_loader_fast, desc=f"Balanced teacher {epoch+1}/3"
    ):
        ids = ids.to(device)
        lengths = lengths.to(device)
        labels = labels.to(device)

        optimizer_fast.zero_grad(set_to_none=True)
        logits = m1_cjpe(ids, lengths)
        loss = criterion_fast(logits.float(), labels)
        loss.backward()
        optimizer_fast.step()

    m1_cjpe.eval()
    y_true, y_pred = [], []
    with torch.no_grad():
        for ids, lengths, labels in dev_loader_fast:
            logits = m1_cjpe(ids.to(device), lengths.to(device))
            y_pred.extend(logits.argmax(1).cpu().tolist())
            y_true.extend(labels.tolist())

    print(f"Epoch {epoch+1} Macro-F1: {f1_score(y_true, y_pred, average='macro'):.4f}")
    print("Confusion matrix:\n", confusion_matrix(y_true, y_pred))


Balanced teacher 1/3:   0%|          | 0/250 [00:00<?, ?it/s]

Epoch 1 Macro-F1: 0.5209
Confusion matrix:
 [[167  62]
 [173  98]]


Balanced teacher 2/3:   0%|          | 0/250 [00:00<?, ?it/s]

Epoch 2 Macro-F1: 0.4822
Confusion matrix:
 [[202  27]
 [211  60]]


Balanced teacher 3/3:   0%|          | 0/250 [00:00<?, ?it/s]

Epoch 3 Macro-F1: 0.4475
Confusion matrix:
 [[204  25]
 [224  47]]


In [36]:

import os
import torch
import torch.nn as nn
from tqdm.auto import tqdm

# Recreate the epoch-1 teacher state deterministically as far as possible.
# Save immediately after this one epoch.
m1_cjpe.classifier.reset_parameters()
optimizer_fast = torch.optim.AdamW(
    m1_cjpe.classifier.parameters(), lr=5e-4, weight_decay=0.01
)
criterion_fast = nn.CrossEntropyLoss(weight=weights)

m1_cjpe.train()
m1_cjpe.backbone.eval()

for ids, lengths, labels in tqdm(train_loader_fast, desc="Save best teacher"):
    ids = ids.to(device)
    lengths = lengths.to(device)
    labels = labels.to(device)

    optimizer_fast.zero_grad(set_to_none=True)
    logits = m1_cjpe(ids, lengths)
    loss = criterion_fast(logits.float(), labels)
    loss.backward()
    optimizer_fast.step()

save_path = "/kaggle/working/cjpe_m1_teacher/teacher_head_best.pt"
os.makedirs(os.path.dirname(save_path), exist_ok=True)

torch.save({
    "classifier": m1_cjpe.classifier.state_dict(),
    "val_macro_f1": 0.5209,
    "max_len": MAX_LEN_FAST,
    "train_subset": TRAIN_N
}, save_path)

print("Saved teacher head:", save_path)


Save best teacher:   0%|          | 0/250 [00:00<?, ?it/s]

Saved teacher head: /kaggle/working/cjpe_m1_teacher/teacher_head_best.pt


In [39]:

import torch
import torch.nn as nn
import os

# Freeze M1 teacher
m1_cjpe.eval()
for p in m1_cjpe.parameters():
    p.requires_grad = False

# Freeze M4 backbone initially; train only the binary head.
classifier_model.backbone.eval()
for p in classifier_model.backbone.parameters():
    p.requires_grad = False

for p in classifier_model.classifier.parameters():
    p.requires_grad = True

print("Teacher parameters trainable:",
      sum(p.numel() for p in m1_cjpe.parameters() if p.requires_grad))
print("Student backbone trainable:",
      sum(p.numel() for p in classifier_model.backbone.parameters() if p.requires_grad))
print("Student classifier trainable:",
      sum(p.numel() for p in classifier_model.classifier.parameters() if p.requires_grad))

# Ensure outputs go to a separate location; preserve Phase 1 checkpoints.
KD_DIR = "/kaggle/working/cjpe_m4_kd"
os.makedirs(KD_DIR, exist_ok=True)
print("KD output directory:", KD_DIR)


Teacher parameters trainable: 0
Student backbone trainable: 0
Student classifier trainable: 9997
KD output directory: /kaggle/working/cjpe_m4_kd


In [47]:

import torch
import types

def m4_forward_hidden(self, input_ids):
    x = self.token_embedding(input_ids)
    x = self.dropout(x)

    for block in self.blocks:
        x = block(x)

    x = self.norm(x)
    return x

# Attach to the exact backbone used by the classifier
classifier_model.backbone.forward_hidden = types.MethodType(
    m4_forward_hidden, classifier_model.backbone
)

print("Attached to classifier_model.backbone:",
      hasattr(classifier_model.backbone, "forward_hidden"))


Attached to classifier_model.backbone: True


In [49]:
import torch
import torch.nn as nn

# Replace the incorrect 13-class head with a binary CJPE head.
classifier_model.classifier = nn.Linear(768, 2).to(device)

# Freeze the M4 backbone and train only this new head initially.
for p in classifier_model.backbone.parameters():
    p.requires_grad = False

for p in classifier_model.classifier.parameters():
    p.requires_grad = True

print("Student output classes:", classifier_model.classifier.out_features)
print("Trainable parameters:",
      sum(p.numel() for p in classifier_model.parameters() if p.requires_grad))

Student output classes: 2
Trainable parameters: 1538


In [50]:

import torch
import torch.nn.functional as F
import torch.nn as nn
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
from sklearn.metrics import f1_score, accuracy_score
import os

# Fast settings
KD_TRAIN_N = 2000
KD_DEV_N = 500
KD_MAX_LEN = 256
KD_BATCH = 8
TEMPERATURE = 2.0
ALPHA = 0.5
KD_EPOCHS = 2

# Reuse the quick subset, but truncate to the KD context length.
kd_train = make_subset(train_cjpe, KD_TRAIN_N, KD_MAX_LEN, shuffle=True)
kd_dev = make_subset(dev_cjpe, KD_DEV_N, KD_MAX_LEN)

kd_train_ds = CJPEBinaryDataset(kd_train, KD_MAX_LEN)
kd_dev_ds = CJPEBinaryDataset(kd_dev, KD_MAX_LEN)

kd_train_loader = DataLoader(
    kd_train_ds, batch_size=KD_BATCH, shuffle=True,
    num_workers=0, collate_fn=collate_cjpe
)
kd_dev_loader = DataLoader(
    kd_dev_ds, batch_size=KD_BATCH, shuffle=False,
    num_workers=0, collate_fn=collate_cjpe
)

# Train only the binary M4 classifier head for this quick experiment.
optimizer_kd = torch.optim.AdamW(
    classifier_model.classifier.parameters(), lr=1e-3, weight_decay=0.01
)
best_kd_f1 = -1.0
best_path = os.path.join(KD_DIR, "m4_kd_best.pt")

for epoch in range(KD_EPOCHS):
    classifier_model.train()
    classifier_model.backbone.eval()
    m1_cjpe.eval()
    total_loss = 0.0

    for ids, lengths, labels in tqdm(kd_train_loader, desc=f"KD {epoch+1}/{KD_EPOCHS}"):
        ids = ids.to(device)
        lengths = lengths.to(device)
        labels = labels.to(device)

        with torch.no_grad():
            teacher_logits = m1_cjpe(ids, lengths)
            teacher_soft = F.softmax(teacher_logits.float() / TEMPERATURE, dim=-1)

        optimizer_kd.zero_grad(set_to_none=True)
        student_logits = classifier_model(ids, lengths)

        hard_loss = F.cross_entropy(student_logits.float(), labels)
        soft_loss = F.kl_div(
            F.log_softmax(student_logits.float() / TEMPERATURE, dim=-1),
            teacher_soft,
            reduction="batchmean"
        ) * (TEMPERATURE ** 2)

        loss = (1 - ALPHA) * hard_loss + ALPHA * soft_loss
        if not torch.isfinite(loss):
            raise RuntimeError("Non-finite KD loss; stopping.")

        loss.backward()
        torch.nn.utils.clip_grad_norm_(
            classifier_model.classifier.parameters(), max_norm=1.0
        )
        optimizer_kd.step()
        total_loss += loss.item()

    classifier_model.eval()
    y_true, y_pred = [], []
    with torch.no_grad():
        for ids, lengths, labels in tqdm(kd_dev_loader, desc="KD validation"):
            logits = classifier_model(ids.to(device), lengths.to(device))
            y_pred.extend(logits.argmax(1).cpu().tolist())
            y_true.extend(labels.tolist())

    val_f1 = f1_score(y_true, y_pred, average="macro")
    val_acc = accuracy_score(y_true, y_pred)
    print(f"Epoch {epoch+1}: loss={total_loss/len(kd_train_loader):.4f}, "
          f"Macro-F1={val_f1:.4f}, accuracy={val_acc:.4f}")

    if val_f1 > best_kd_f1:
        best_kd_f1 = val_f1
        torch.save({
            "student_state_dict": classifier_model.state_dict(),
            "epoch": epoch + 1,
            "val_macro_f1": val_f1,
            "val_accuracy": val_acc,
            "method": "M1-to-M4 knowledge distillation",
            "temperature": TEMPERATURE,
            "alpha": ALPHA,
            "train_subset": KD_TRAIN_N,
            "max_len": KD_MAX_LEN
        }, best_path)
        print("Saved best KD checkpoint:", best_path)

print("Best KD Macro-F1:", best_kd_f1)


KD 1/2:   0%|          | 0/250 [00:00<?, ?it/s]

KD validation:   0%|          | 0/63 [00:00<?, ?it/s]

Epoch 1: loss=0.3768, Macro-F1=0.5666, accuracy=0.5680
Saved best KD checkpoint: /kaggle/working/cjpe_m4_kd/m4_kd_best.pt


KD 2/2:   0%|          | 0/250 [00:00<?, ?it/s]

KD validation:   0%|          | 0/63 [00:00<?, ?it/s]

Epoch 2: loss=0.3623, Macro-F1=0.3834, accuracy=0.5400
Best KD Macro-F1: 0.5666409861325116


In [51]:
import os
import torch
from tqdm.auto import tqdm

OUT = "/kaggle/working/cjpe_m4_long"
os.makedirs(OUT, exist_ok=True)

MAX_DOC_TOKENS = 2048

def encode_long_documents(rows, name, deduplicate=False):
    seen = set()
    ids_list = []
    labels = []
    case_ids = []

    for row in tqdm(rows, desc=name):
        case_id = row["id"]

        if deduplicate and case_id in seen:
            continue
        seen.add(case_id)

        ids = list(tokenizer.encode_text(row["text"]))
        ids = ids[:MAX_DOC_TOKENS]

        if len(ids) == 0:
            continue

        ids_list.append(torch.tensor(ids, dtype=torch.long))
        labels.append(int(row["label"]))
        case_ids.append(case_id)

    return {
        "ids": ids_list,
        "labels": torch.tensor(labels, dtype=torch.long),
        "case_ids": case_ids
    }

train_long = encode_long_documents(
    train_data, "Long-document train", deduplicate=True
)
dev_long = encode_long_documents(
    dev_data, "Long-document dev"
)

torch.save(train_long, OUT + "/train.pt")
torch.save(dev_long, OUT + "/dev.pt")

print("Train:", len(train_long["ids"]))
print("Dev:", len(dev_long["ids"]))
print("Max tokens:", max(len(x) for x in train_long["ids"]))
print("Saved to:", OUT)

Long-document train:   0%|          | 0/37387 [00:00<?, ?it/s]

Long-document dev:   0%|          | 0/1988 [00:00<?, ?it/s]

Train: 32305
Dev: 1988
Max tokens: 2048
Saved to: /kaggle/working/cjpe_m4_long


In [59]:
%%writefile /kaggle/working/train_m4_long_cjpe.py
import os
import random
import math
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import f1_score, precision_score, recall_score, confusion_matrix
from tqdm.auto import tqdm

from train_m4 import M4Model

SEED = 42
CHUNK_LEN = 256
STRIDE = 192
MAX_DOC_TOKENS = 2048
MAX_CHUNKS = math.ceil((MAX_DOC_TOKENS - CHUNK_LEN) / STRIDE) + 1

BATCH_SIZE = 1
EPOCHS = 3
BACKBONE_LR = 1e-5
HEAD_LR = 3e-4
WEIGHT_DECAY = 0.01
NUM_WORKERS = 2

PRETRAINED = "/kaggle/working/m4_checkpoints/best.pt"
TRAIN_FILE = "/kaggle/working/cjpe_m4_long/train.pt"
DEV_FILE = "/kaggle/working/cjpe_m4_long/dev.pt"
OUT_DIR = "/kaggle/working/cjpe_m4_long_results"
os.makedirs(OUT_DIR, exist_ok=True)

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

class DocumentDataset(Dataset):
    def __init__(self, path):
        self.data = torch.load(path, map_location="cpu", weights_only=False)

    def __len__(self):
        return len(self.data["labels"])

    def __getitem__(self, idx):
        ids = torch.as_tensor(
            self.data["ids"][idx], dtype=torch.long
        )[:MAX_DOC_TOKENS]
        label = int(self.data["labels"][idx])

        if ids.numel() == 0:
            ids = torch.tensor([0], dtype=torch.long)

        chunks = []
        start = 0
        while start < ids.numel() and len(chunks) < MAX_CHUNKS:
            chunk = ids[start:start + CHUNK_LEN]
            if chunk.numel() > 0:
                chunks.append(chunk)

            if start + CHUNK_LEN >= ids.numel():
                break
            start += STRIDE

        return chunks, label

def collate_documents(batch):
    all_docs = [item[0] for item in batch]
    labels = [item[1] for item in batch]

    flat_chunks = []
    doc_indices = []
    for doc_i, chunks in enumerate(all_docs):
        for chunk in chunks:
            flat_chunks.append(chunk)
            doc_indices.append(doc_i)

    max_len = max(len(x) for x in flat_chunks)
    input_ids = torch.zeros(
        (len(flat_chunks), max_len), dtype=torch.long
    )
    token_mask = torch.zeros(
        (len(flat_chunks), max_len), dtype=torch.bool
    )

    for i, chunk in enumerate(flat_chunks):
        input_ids[i, :len(chunk)] = chunk
        token_mask[i, :len(chunk)] = True

    chunk_mask = torch.zeros(
        (len(batch), max(len(x) for x in all_docs)), dtype=torch.bool
    )
    for doc_i, chunks in enumerate(all_docs):
        chunk_mask[doc_i, :len(chunks)] = True

    return (
        input_ids,
        token_mask,
        torch.tensor(doc_indices, dtype=torch.long),
        chunk_mask,
        torch.tensor(labels, dtype=torch.long)
    )

class M4LongDocumentClassifier(nn.Module):
    def __init__(self, backbone, unfreeze_last_n=2):
        super().__init__()
        self.backbone = backbone

        for p in self.backbone.parameters():
            p.requires_grad = False

        for block in self.backbone.blocks[-unfreeze_last_n:]:
            for p in block.parameters():
                p.requires_grad = True

        for p in self.backbone.norm.parameters():
            p.requires_grad = True

        self.chunk_score = nn.Sequential(
            nn.Linear(768, 256),
            nn.Tanh(),
            nn.Linear(256, 1, bias=False)
        )
        self.classifier = nn.Sequential(
            nn.Dropout(0.2),
            nn.Linear(768, 2)
        )

    def forward(self, input_ids, token_mask, doc_indices, chunk_mask):
        x = self.backbone.token_embedding(input_ids)
        x = self.backbone.dropout(x)

        for block in self.backbone.blocks:
            x = block(x)

        x = self.backbone.norm(x)

        mask = token_mask.unsqueeze(-1).to(x.dtype)
        chunk_vectors = (x * mask).sum(dim=1)
        chunk_vectors = chunk_vectors / mask.sum(dim=1).clamp(min=1.0)

        num_docs = chunk_mask.size(0)
        max_chunks = chunk_mask.size(1)
        doc_chunk_vectors = x.new_zeros((num_docs, max_chunks, 768))

        # Restore each chunk vector to its document/chunk position.
        offsets = torch.zeros(num_docs, dtype=torch.long, device=x.device)
        for i in range(chunk_vectors.size(0)):
            doc_i = int(doc_indices[i].item())
            slot = int(offsets[doc_i].item())
            doc_chunk_vectors[doc_i, slot] = chunk_vectors[i]
            offsets[doc_i] += 1

        scores = self.chunk_score(doc_chunk_vectors).squeeze(-1)
        scores = scores.masked_fill(~chunk_mask.to(x.device), -1e4)
        weights = torch.softmax(scores, dim=1)

        document_vectors = (
            doc_chunk_vectors * weights.unsqueeze(-1)
        ).sum(dim=1)

        return self.classifier(document_vectors)

def evaluate(model, loader, device, criterion):
    model.eval()
    total_loss = 0.0
    all_labels = []
    all_probs = []

    with torch.no_grad():
        for input_ids, token_mask, doc_indices, chunk_mask, labels in tqdm(
            loader, desc="Evaluating", leave=False
        ):
            input_ids = input_ids.to(device)
            token_mask = token_mask.to(device)
            doc_indices = doc_indices.to(device)
            chunk_mask = chunk_mask.to(device)
            labels = labels.to(device)

            with torch.autocast(
                device_type="cuda",
                dtype=torch.bfloat16,
                enabled=(device.type == "cuda" and torch.cuda.is_bf16_supported())
            ):
                logits = model(input_ids, token_mask, doc_indices, chunk_mask)
                loss = criterion(logits, labels)

            total_loss += loss.item() * labels.size(0)
            probs = torch.softmax(logits.float(), dim=-1)[:, 1]
            all_probs.extend(probs.cpu().tolist())
            all_labels.extend(labels.cpu().tolist())

    preds = [int(p >= 0.5) for p in all_probs]
    return (
        total_loss / max(1, len(all_labels)),
        f1_score(all_labels, preds, average="macro", zero_division=0),
        precision_score(all_labels, preds, zero_division=0),
        recall_score(all_labels, preds, zero_division=0),
        confusion_matrix(all_labels, preds, labels=[0, 1]),
        np.array(all_labels),
        np.array(all_probs)
    )

def main():
    set_seed(SEED)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Device:", device)
    if device.type == "cuda":
        print("GPU:", torch.cuda.get_device_name(0))

    for path in [PRETRAINED, TRAIN_FILE, DEV_FILE]:
        if not os.path.exists(path):
            raise FileNotFoundError(path)

    train_ds = DocumentDataset(TRAIN_FILE)
    dev_ds = DocumentDataset(DEV_FILE)
    train_loader = DataLoader(
        train_ds, batch_size=BATCH_SIZE, shuffle=True,
        num_workers=NUM_WORKERS, collate_fn=collate_documents,
        pin_memory=(device.type == "cuda")
    )
    dev_loader = DataLoader(
        dev_ds, batch_size=BATCH_SIZE, shuffle=False,
        num_workers=NUM_WORKERS, collate_fn=collate_documents,
        pin_memory=(device.type == "cuda")
    )

    backbone = M4Model()
    checkpoint = torch.load(
        PRETRAINED, map_location="cpu", weights_only=False
    )
    backbone.load_state_dict(checkpoint["model"], strict=True)

    model = M4LongDocumentClassifier(backbone).to(device)

    backbone_params = [
        p for p in model.backbone.parameters() if p.requires_grad
    ]
    head_params = list(model.chunk_score.parameters()) + list(
        model.classifier.parameters()
    )

    optimizer = torch.optim.AdamW(
        [
            {"params": backbone_params, "lr": BACKBONE_LR},
            {"params": head_params, "lr": HEAD_LR}
        ],
        weight_decay=WEIGHT_DECAY
    )
    criterion = nn.CrossEntropyLoss()

    best_f1 = -1.0
    best_path = os.path.join(OUT_DIR, "best.pt")

    print("Train documents:", len(train_ds))
    print("Dev documents:", len(dev_ds))
    print("Trainable parameters:",
          sum(p.numel() for p in model.parameters() if p.requires_grad))

    # One-batch smoke test before the full run.
    batch = next(iter(train_loader))
    with torch.no_grad():
        test_logits = model(
            batch[0].to(device),
            batch[1].to(device),
            batch[2].to(device),
            batch[3].to(device)
        )
    print("Smoke test logits shape:", tuple(test_logits.shape))
    assert test_logits.shape == (BATCH_SIZE, 2)

    for epoch in range(1, EPOCHS + 1):
        model.train()
        running_loss = 0.0
        seen = 0
        progress = tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS}")

        for input_ids, token_mask, doc_indices, chunk_mask, labels in progress:
            input_ids = input_ids.to(device)
            token_mask = token_mask.to(device)
            doc_indices = doc_indices.to(device)
            chunk_mask = chunk_mask.to(device)
            labels = labels.to(device)

            optimizer.zero_grad(set_to_none=True)

            with torch.autocast(
                device_type="cuda",
                dtype=torch.bfloat16,
                enabled=(device.type == "cuda" and torch.cuda.is_bf16_supported())
            ):
                logits = model(input_ids, token_mask, doc_indices, chunk_mask)
                loss = criterion(logits, labels)

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            running_loss += loss.item() * labels.size(0)
            seen += labels.size(0)
            progress.set_postfix(loss=running_loss / max(1, seen))

        dev_loss, dev_f1, dev_p, dev_r, cm, y_true, y_prob = evaluate(
            model, dev_loader, device, criterion
        )
        train_loss = running_loss / max(1, seen)

        print(
            f"Epoch {epoch}: train_loss={train_loss:.4f}, "
            f"dev_loss={dev_loss:.4f}, dev_macro_f1={dev_f1:.4f}, "
            f"precision={dev_p:.4f}, recall={dev_r:.4f}"
        )
        print("Confusion matrix:\n", cm)

        torch.save(
            {
                "epoch": epoch,
                "model": model.state_dict(),
                "optimizer": optimizer.state_dict(),
                "dev_macro_f1": dev_f1,
                "dev_loss": dev_loss
            },
            os.path.join(OUT_DIR, "latest.pt")
        )

        if dev_f1 > best_f1:
            best_f1 = dev_f1
            torch.save(
                {
                    "epoch": epoch,
                    "model": model.state_dict(),
                    "optimizer": optimizer.state_dict(),
                    "dev_macro_f1": dev_f1,
                    "dev_loss": dev_loss
                },
                best_path
            )
            np.savez(
                os.path.join(OUT_DIR, "best_dev_predictions.npz"),
                labels=y_true, probabilities=y_prob
            )
            print("Saved new best checkpoint:", best_path)

    print("\nTraining finished. Best dev Macro-F1:", best_f1)
    print("Best checkpoint:", best_path)

if __name__ == "__main__":
    main()

Overwriting /kaggle/working/train_m4_long_cjpe.py


In [64]:
import os
import sys
import torch
import importlib.util

M1_DIR = "/kaggle/input/datasets/spikie/capstone-split"
M1_CODE = os.path.join(M1_DIR, "legal_m1.py")
M1_CHECKPOINT = os.path.join(M1_DIR, "latest.pt")
TOKENIZER_PATH = os.path.join(M1_DIR, "legal_bpe_32k.pkl")

for path in [M1_CODE, M1_CHECKPOINT, TOKENIZER_PATH]:
    assert os.path.exists(path), f"Missing file: {path}"
    print("Found:", path)

# Import the M1 architecture from the mounted dataset.
spec = importlib.util.spec_from_file_location("legal_m1", M1_CODE)
legal_m1 = importlib.util.module_from_spec(spec)
sys.modules["legal_m1"] = legal_m1
spec.loader.exec_module(legal_m1)

print("\nModel classes available:")
print([name for name in dir(legal_m1)
       if isinstance(getattr(legal_m1, name), type)])

# Inspect checkpoint metadata.
ckpt = torch.load(
    M1_CHECKPOINT,
    map_location="cpu",
    weights_only=False
)

print("\nCheckpoint keys:", list(ckpt.keys()))
if "step" in ckpt:
    print("Pretraining step:", ckpt["step"])
if "tokens_seen" in ckpt:
    print("Tokens seen:", ckpt["tokens_seen"])
if "best_val_loss" in ckpt:
    print("Best pretraining validation loss:", ckpt["best_val_loss"])
if "config" in ckpt:
    print("Config:", ckpt["config"])

Found: /kaggle/input/datasets/spikie/capstone-split/legal_m1.py
Found: /kaggle/input/datasets/spikie/capstone-split/latest.pt
Found: /kaggle/input/datasets/spikie/capstone-split/legal_bpe_32k.pkl

Model classes available:
['CausalSelfAttention', 'FeedForward', 'GPTModel', 'RMSNorm', 'RotaryEmbedding', 'TransformerBlock']

Checkpoint keys: ['model', 'optimizer', 'scaler', 'step', 'tokens_seen', 'best_val_loss', 'config']
Pretraining step: 36639
Tokens seen: 750000330
Best pretraining validation loss: 1.3824894428253174
Config: {'vocab_size': 32000, 'block_size': 2048, 'd_model': 768, 'n_layers': 12, 'n_heads': 12, 'ffn_dim': 3072, 'dropout': 0.2, 'start_tokens': 500000220, 'target_tokens': 750000000, 'start_lr': 3e-05, 'min_lr': 3e-06}


In [67]:
import json
from collections import Counter

from huggingface_hub import hf_hub_download

# Download/read the official Phase 4 split files.
train_path = hf_hub_download(
    repo_id="Exploration-Lab/CS-781-Capstone",
    repo_type="dataset",
    filename="Phase_4_CJPE/Train/train.json",
    token=True
)

dev_path = hf_hub_download(
    repo_id="Exploration-Lab/CS-781-Capstone",
    repo_type="dataset",
    filename="Phase_4_CJPE/Train/dev.json",
    token=True
)

with open(train_path, "r", encoding="utf-8") as f:
    raw_train = json.load(f)

with open(dev_path, "r", encoding="utf-8") as f:
    raw_dev = json.load(f)

# Accommodate either a list or a wrapped JSON object.
if isinstance(raw_train, dict):
    raw_train = raw_train.get("data", raw_train.get("train", raw_train))
if isinstance(raw_dev, dict):
    raw_dev = raw_dev.get("data", raw_dev.get("dev", raw_dev))

assert isinstance(raw_train, list)
assert isinstance(raw_dev, list)

# Deduplicate training records by case ID, preserving the first occurrence.
seen_ids = set()
train_cjpe_full = []

for row in raw_train:
    case_id = str(row["id"])
    if case_id not in seen_ids:
        seen_ids.add(case_id)
        train_cjpe_full.append(row)

dev_cjpe_full = raw_dev

train_ids = {str(row["id"]) for row in train_cjpe_full}
dev_ids = {str(row["id"]) for row in dev_cjpe_full}

print("Raw train:", len(raw_train))
print("Deduplicated train:", len(train_cjpe_full))
print("Dev:", len(dev_cjpe_full))
print("Train label counts:", Counter(row["label"] for row in train_cjpe_full))
print("Dev label counts:", Counter(row["label"] for row in dev_cjpe_full))
print("Train/dev ID overlap:", len(train_ids & dev_ids))

assert len(train_cjpe_full) == 32305, "Unexpected train size; inspect the split before training."
assert len(dev_cjpe_full) == 1988, "Unexpected dev size; inspect the split before training."
assert not (train_ids & dev_ids), "Train/dev overlap detected."

print("Dataset checks passed.")

Raw train: 37387
Deduplicated train: 32305
Dev: 1988
Train label counts: Counter({0: 18920, 1: 13385})
Dev label counts: Counter({0: 994, 1: 994})
Train/dev ID overlap: 0
Dataset checks passed.


In [68]:
import pickle
import torch
from tqdm.auto import tqdm

TOKENIZER_PATH = "/kaggle/input/datasets/spikie/capstone-split/legal_bpe_32k.pkl"
TEACHER_DATA_DIR = "/kaggle/working/cjpe_m1_teacher_full"
MAX_LEN_TEACHER = 1024

os.makedirs(TEACHER_DATA_DIR, exist_ok=True)

with open(TOKENIZER_PATH, "rb") as f:
    tokenizer = pickle.load(f)

print("Tokenizer vocabulary:", tokenizer.vocab_size)
print("EOS ID:", tokenizer.eos_id)

def tokenize_cjpe_split(rows, name):
    all_ids = []
    all_labels = []
    all_case_ids = []

    for row in tqdm(rows, desc=f"Tokenizing {name}"):
        ids = tokenizer.encode_text(row["text"], add_eos=True)
        ids = ids[:MAX_LEN_TEACHER]

        # Ensure every example has at least one token.
        if not ids:
            ids = [tokenizer.eos_id]

        all_ids.append(torch.tensor(ids, dtype=torch.long))
        all_labels.append(int(row["label"]))
        all_case_ids.append(str(row["id"]))

    result = {
        "ids": all_ids,
        "labels": torch.tensor(all_labels, dtype=torch.long),
        "case_ids": all_case_ids
    }

    out_path = os.path.join(TEACHER_DATA_DIR, f"{name}.pt")
    torch.save(result, out_path)

    lengths = torch.tensor([len(x) for x in all_ids])
    print(
        f"{name}: {len(all_ids)} examples | "
        f"mean tokens={lengths.float().mean():.1f} | "
        f"max tokens={lengths.max().item()} | saved={out_path}"
    )
    return result

teacher_train_data = tokenize_cjpe_split(train_cjpe_full, "train")
teacher_dev_data = tokenize_cjpe_split(dev_cjpe_full, "dev")

assert len(teacher_train_data["labels"]) == 32305
assert len(teacher_dev_data["labels"]) == 1988
assert set(teacher_train_data["case_ids"]).isdisjoint(
    set(teacher_dev_data["case_ids"])
)

print("Full teacher tokenization complete.")

Tokenizer vocabulary: 32000
EOS ID: 0


Tokenizing train:   0%|          | 0/32305 [00:00<?, ?it/s]

train: 32305 examples | mean tokens=958.8 | max tokens=1024 | saved=/kaggle/working/cjpe_m1_teacher_full/train.pt


Tokenizing dev:   0%|          | 0/1988 [00:00<?, ?it/s]

dev: 1988 examples | mean tokens=1020.4 | max tokens=1024 | saved=/kaggle/working/cjpe_m1_teacher_full/dev.pt
Full teacher tokenization complete.


In [7]:
import torch
import torch.nn as nn

class M1CJPEClassifier(nn.Module):
    def __init__(self, backbone):
        super().__init__()
        self.backbone = backbone

        # Freeze the pretrained language model initially
        for param in self.backbone.parameters():
            param.requires_grad = False

        self.classifier = nn.Sequential(nn.Dropout(0.2),nn.Linear(768, 2))

    def forward(self, input_ids, lengths):
        # input_ids: [batch_size, sequence_length]
        x = self.backbone.token_embedding(input_ids)
        x = self.backbone.dropout(x)

        for block in self.backbone.blocks:
            x = block(x)

        x = self.backbone.norm(x)

        # Mask padding positions before mean pooling
        positions = torch.arange(input_ids.size(1), device=input_ids.device).unsqueeze(0)

        mask = positions < lengths.unsqueeze(1)
        mask = mask.unsqueeze(-1).to(x.dtype)

        x = (x * mask).sum(dim=1)
        x = x / lengths.clamp(min=1).unsqueeze(1).to(x.dtype)

        return self.classifier(x)


# Instantiate and check trainable parameters
m1_cjpe = M1CJPEClassifier(m1_backbone)
trainable = sum(p.numel() for p in m1_cjpe.parameters() if p.requires_grad)
total = sum(p.numel() for p in m1_cjpe.parameters())
print("Total parameters:", f"{total:,}")
print("Trainable parameters:", f"{trainable:,}")
print("Frozen backbone:", all(not p.requires_grad for p in m1_cjpe.backbone.parameters()))

Total parameters: 109,614,338
Trainable parameters: 1,538
Frozen backbone: True


In [12]:
import torch
import torch.nn as nn

# Unwrap DataParallel if it was already applied in an earlier cell
if isinstance(m1_cjpe, nn.DataParallel):
    m1_cjpe = m1_cjpe.module

model = m1_cjpe
backbone = model.backbone

# Freeze the entire pretrained backbone first
for param in backbone.parameters():
    param.requires_grad = False

# Unfreeze the final two transformer blocks
for block in backbone.blocks[-2:]:
    for param in block.parameters():
        param.requires_grad = True

# Unfreeze final normalization and classification head
for param in backbone.norm.parameters():
    param.requires_grad = True

for param in model.classifier.parameters():
    param.requires_grad = True

# Mixed precision: FP16 is appropriate for Tesla T4
device = torch.device("cuda:0")
model = model.to(device)

print("Trainable blocks:", len(backbone.blocks) - 2, "frozen blocks,",
      "last 2 unfrozen")
print("Trainable parameters:",
      f"{sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
print("Total parameters:",
      f"{sum(p.numel() for p in model.parameters()):,}")
print("FP16 enabled for training")

Trainable blocks: 10 frozen blocks, last 2 unfrozen
Trainable parameters: 14,174,978
Total parameters: 109,614,338
FP16 enabled for training


In [13]:
%%writefile /kaggle/working/train_m1_teacher_ddp.py
import os
import random
import numpy as np
import torch
import torch.nn as nn
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import Dataset, DataLoader, DistributedSampler
from torch.nn.utils.rnn import pad_sequence
from sklearn.metrics import f1_score, accuracy_score
from sklearn.metrics import confusion_matrix
from tqdm import tqdm
import importlib.util
import sys

SEED = 42
BATCH_SIZE = 4
ACCUM_STEPS = 4
EPOCHS = 3
LR = 2e-5
WEIGHT_DECAY = 0.01
SAVE_DIR = "/kaggle/working/cjpe_m1_teacher_full"
BEST_PATH = os.path.join(SAVE_DIR, "best_teacher.pt")
os.makedirs(SAVE_DIR, exist_ok=True)

def setup():
    dist.init_process_group(backend="nccl")
    local_rank = int(os.environ["LOCAL_RANK"])
    torch.cuda.set_device(local_rank)
    return local_rank, dist.get_rank(), dist.get_world_size()

def cleanup():
    dist.destroy_process_group()

class CJPEClassificationDataset(Dataset):
    def __init__(self, data):
        self.ids = data["ids"]
        self.labels = data["labels"]

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, idx):
        return self.ids[idx], int(self.labels[idx])

def collate_cjpe(batch):
    seqs = [x[0] for x in batch]
    labels = torch.tensor([x[1] for x in batch], dtype=torch.long)
    lengths = torch.tensor([len(x) for x in seqs], dtype=torch.long)
    ids = pad_sequence(seqs, batch_first=True, padding_value=0)
    return ids, lengths, labels

class M1CJPEClassifier(nn.Module):
    def __init__(self, backbone):
        super().__init__()
        self.backbone = backbone
        for p in self.backbone.parameters():
            p.requires_grad = False
        for block in self.backbone.blocks[-2:]:
            for p in block.parameters():
                p.requires_grad = True
        for p in self.backbone.norm.parameters():
            p.requires_grad = True
        self.classifier = nn.Sequential(nn.Dropout(0.2), nn.Linear(768, 2))

    def forward(self, input_ids, lengths):
        x = self.backbone.token_embedding(input_ids)
        x = self.backbone.dropout(x)
        for block in self.backbone.blocks:
            x = block(x)
        x = self.backbone.norm(x)

        positions = torch.arange(input_ids.size(1), device=input_ids.device).unsqueeze(0)
        mask = positions < lengths.unsqueeze(1)
        mask = mask.unsqueeze(-1).to(x.dtype)
        pooled = (x * mask).sum(dim=1)
        pooled = pooled / lengths.clamp(min=1).unsqueeze(1).to(x.dtype)
        return self.classifier(pooled)

def evaluate(model, loader, device, rank, world_size):
    model.eval()
    local_probs, local_labels = [], []
    with torch.no_grad():
        for ids, lengths, labels in loader:
            ids = ids.to(device, non_blocking=True)
            lengths = lengths.to(device, non_blocking=True)
            with torch.autocast("cuda", dtype=torch.float16):
                logits = model(ids, lengths)
            probs = torch.softmax(logits.float(), dim=1)[:, 1]
            local_probs.extend(probs.cpu().tolist())
            local_labels.extend(labels.tolist())

    # Gather uneven-length evaluation outputs safely as Python objects.
    gathered = [None for _ in range(world_size)]
    dist.all_gather_object(gathered, (local_labels, local_probs))
    if rank != 0:
        return None

    labels, probs = [], []
    for labs, ps in gathered:
        labels.extend(labs)
        probs.extend(ps)
    preds = [int(p >= 0.5) for p in probs]
    return {
        "macro_f1": f1_score(labels, preds, average="macro"),
        "accuracy": accuracy_score(labels, preds),
        "confusion_matrix": confusion_matrix(labels, preds).tolist()
    }

def main():
    local_rank, rank, world_size = setup()
    device = torch.device(f"cuda:{local_rank}")
    random.seed(SEED + rank)
    np.random.seed(SEED + rank)
    torch.manual_seed(SEED + rank)

    data_dir = "/kaggle/working/cjpe_m1_teacher_full"
    train_data = torch.load(
        os.path.join(data_dir, "train.pt"),
        map_location="cpu",
        weights_only=False
    )
    dev_data = torch.load(
        os.path.join(data_dir, "dev.pt"),
        map_location="cpu",
        weights_only=False
    )

    m1_dir = "/kaggle/input/datasets/spikie/capstone-split"
    spec = importlib.util.spec_from_file_location(
        "legal_m1", os.path.join(m1_dir, "legal_m1.py")
    )
    legal_m1 = importlib.util.module_from_spec(spec)
    sys.modules["legal_m1"] = legal_m1
    spec.loader.exec_module(legal_m1)

    ckpt = torch.load(
        os.path.join(m1_dir, "latest.pt"),
        map_location="cpu",
        weights_only=False
    )
    cfg = ckpt["config"]
    backbone = legal_m1.GPTModel(
        vocab_size=cfg["vocab_size"],
        d_model=cfg["d_model"],
        n_layers=cfg["n_layers"],
        n_heads=cfg["n_heads"],
        ffn_dim=cfg["ffn_dim"],
        dropout=cfg["dropout"]
    )
    backbone.load_state_dict(ckpt["model"], strict=True)
    del ckpt

    model = M1CJPEClassifier(backbone).to(device)
    model = DDP(model, device_ids=[local_rank], output_device=local_rank)

    train_ds = CJPEClassificationDataset(train_data)
    dev_ds = CJPEClassificationDataset(dev_data)
    train_sampler=DistributedSampler(train_ds,num_replicas=world_size,rank=rank,shuffle=True,seed=SEED)
    dev_sampler=DistributedSampler(dev_ds,num_replicas=world_size,rank=rank,shuffle=False,drop_last=False)

    train_loader = DataLoader(
        train_ds, batch_size=BATCH_SIZE, sampler=train_sampler,
        collate_fn=collate_cjpe, num_workers=2, pin_memory=True,
        persistent_workers=True
    )
    dev_loader = DataLoader(
        dev_ds, batch_size=BATCH_SIZE, sampler=dev_sampler,
        collate_fn=collate_cjpe, num_workers=2, pin_memory=True,
        persistent_workers=True
    )

    counts = torch.bincount(train_data["labels"], minlength=2).float()
    weights = counts.sum() / (2.0 * counts)
    criterion = nn.CrossEntropyLoss(weight=weights.to(device))

    optimizer = torch.optim.AdamW(
        [p for p in model.parameters() if p.requires_grad],
        lr=LR,weight_decay=WEIGHT_DECAY
    )
    scaler = torch.amp.GradScaler("cuda")
    best_f1 = -1.0

    if rank == 0:
        print("World size:", world_size)
        print("Training examples:", len(train_ds))
        print("Development examples:", len(dev_ds))
        print("Trainable parameters:",
              sum(p.numel() for p in model.parameters() if p.requires_grad))

    for epoch in range(1, EPOCHS + 1):
        model.train()
        train_sampler.set_epoch(epoch)
        optimizer.zero_grad(set_to_none=True)
        loss_sum, seen = 0.0, 0
        iterator = tqdm(train_loader, desc=f"Epoch {epoch} train") if rank == 0 else train_loader

        for step, (ids, lengths, labels) in enumerate(iterator):
            ids = ids.to(device, non_blocking=True)
            lengths = lengths.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            with torch.autocast("cuda", dtype=torch.float16):
                logits = model(ids, lengths)
                loss = criterion(logits, labels) / ACCUM_STEPS

            scaler.scale(loss).backward()

            if (step + 1) % ACCUM_STEPS == 0 or (step + 1) == len(train_loader):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(
                    [p for p in model.parameters() if p.requires_grad],1.0)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)

            loss_sum += loss.item()*ACCUM_STEPS*len(labels)
            seen += len(labels)

        stats = evaluate(model, dev_loader, device, rank, world_size)
        if rank == 0:
            print(f"\nEpoch {epoch}: train_loss={loss_sum / max(seen, 1):.4f}")
            print("Development metrics:", stats)

            if stats["macro_f1"] > best_f1:
                best_f1 = stats["macro_f1"]
                torch.save({
                    "model": model.module.state_dict(),
                    "epoch": epoch,
                    "dev_macro_f1": best_f1,
                    "dev_accuracy": stats["accuracy"],
                    "confusion_matrix": stats["confusion_matrix"],
                    "config": cfg
                }, BEST_PATH)
                print("Saved best checkpoint:", BEST_PATH)

        dist.barrier()

    if rank == 0:
        print("Training complete. Best macro-F1:", best_f1)
    cleanup()

if __name__ == "__main__":
    main()

Writing /kaggle/working/train_m1_teacher_ddp.py


In [14]:
!torchrun --standalone --nproc_per_node=2 /kaggle/working/train_m1_teacher_ddp.py

W1026 16:33:08.518000 182 torch/distributed/run.py:851] 
W1026 16:33:08.518000 182 torch/distributed/run.py:851] *****************************************
W1026 16:33:08.518000 182 torch/distributed/run.py:851] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W1026 16:33:08.518000 182 torch/distributed/run.py:851] *****************************************
World size: 2
Training examples: 32305
Development examples: 1988
Trainable parameters: 14174978
Epoch 1 train: 100%|████████████████████████| 4039/4039 [08:21<00:00,  8.05it/s]

Epoch 1: train_loss=0.6900
Development metrics: {'macro_f1': 0.5256675365097654, 'accuracy': 0.5533199195171026, 'confusion_matrix': [[790, 204], [684, 310]]}
Saved best checkpoint: /kaggle/working/cjpe_m1_teacher_full/best_teacher.pt
/usr/local/lib/python3.13/dist-packages/torch/distributed/c10d_logg

In [15]:
import os
import torch
import torch.nn as nn
import numpy as np
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix

BEST_PATH = "/kaggle/working/cjpe_m1_teacher_full/best_teacher.pt"
DEVICE = torch.device("cuda:0")

assert os.path.exists(BEST_PATH), "Best teacher checkpoint not found."

# Recreate the same architecture used in training.
# `m1_cjpe` and `dev_loader` were defined earlier.
if isinstance(m1_cjpe, nn.DataParallel):
    m1_cjpe = m1_cjpe.module

checkpoint = torch.load(BEST_PATH, map_location="cpu",weights_only=False)
m1_cjpe.load_state_dict(checkpoint["model"], strict=True)
m1_cjpe = m1_cjpe.to(DEVICE)
m1_cjpe.eval()

all_labels = []
all_probs = []

with torch.no_grad():
    for ids, lengths, labels in dev_loader:
        ids = ids.to(DEVICE)
        lengths = lengths.to(DEVICE)

        with torch.autocast("cuda", dtype=torch.float16):
            logits = m1_cjpe(ids, lengths)

        probs = torch.softmax(logits.float(), dim=1)[:, 1]
        all_probs.extend(probs.cpu().tolist())
        all_labels.extend(labels.tolist())

all_labels = np.array(all_labels)
all_probs = np.array(all_probs)
best_threshold = 0.5
best_f1 = -1.0
for threshold in np.arange(0.20, 0.801, 0.01):
    preds = (all_probs >= threshold).astype(int)
    score = f1_score(all_labels, preds, average="macro")

    if score > best_f1:
        best_f1 = score
        best_threshold = float(threshold)

default_preds = (all_probs >= 0.5).astype(int)
tuned_preds = (all_probs >= best_threshold).astype(int)

print("Best checkpoint epoch:", checkpoint["epoch"])
print("Saved dev macro-F1:", checkpoint["dev_macro_f1"])
print("\nAt threshold 0.5:")
print("Macro-F1:", f1_score(all_labels, default_preds, average="macro"))
print("Accuracy:", accuracy_score(all_labels, default_preds))
print("Confusion matrix:\n", confusion_matrix(all_labels, default_preds))

print("\nBest threshold:", round(best_threshold, 2))
print("Tuned dev macro-F1:", round(best_f1, 4))
print("Tuned accuracy:", round(accuracy_score(all_labels, tuned_preds), 4))
print("Tuned confusion matrix:\n", confusion_matrix(all_labels, tuned_preds))

Best checkpoint epoch: 2
Saved dev macro-F1: 0.5436087262148073

At threshold 0.5:
Macro-F1: 0.5436087262148073
Accuracy: 0.545271629778672
Confusion matrix:
 [[602 392]
 [512 482]]

Best threshold: 0.5
Tuned dev macro-F1: 0.5436
Tuned accuracy: 0.5453
Tuned confusion matrix:
 [[602 392]
 [512 482]]


In [16]:
import os
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

DEVICE = torch.device("cuda:0")
SAVE_DIR = "/kaggle/working/cjpe_m1_teacher_full"
BEST_PATH = os.path.join(SAVE_DIR, "best_teacher.pt")

# Restore the best model
if isinstance(m1_cjpe, nn.DataParallel):
    m1_cjpe = m1_cjpe.module

checkpoint = torch.load(
    BEST_PATH, map_location="cpu", weights_only=False
)
m1_cjpe.load_state_dict(checkpoint["model"], strict=True)
m1_cjpe = m1_cjpe.to(DEVICE)
m1_cjpe.eval()

def generate_teacher_logits(dataset, split_name, batch_size=8):
    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False,
        collate_fn=collate_cjpe,
        num_workers=2,
        pin_memory=True
    )

    logits_all = []
    labels_all = []

    with torch.no_grad():
        for ids, lengths, labels in tqdm(loader, desc=f"Teacher {split_name}"):
            ids = ids.to(DEVICE, non_blocking=True)
            lengths = lengths.to(DEVICE, non_blocking=True)

            with torch.autocast("cuda", dtype=torch.float16):
                logits = m1_cjpe(ids, lengths)

            logits_all.append(logits.float().cpu())
            labels_all.append(labels.cpu())

    result = {
        "logits": torch.cat(logits_all, dim=0),
        "labels": torch.cat(labels_all, dim=0)
    }

    out_path = os.path.join(SAVE_DIR, f"{split_name}_teacher_logits.pt")
    torch.save(result, out_path)

    print(f"\nSaved: {out_path}")
    print("Logits shape:", tuple(result["logits"].shape))
    print("Labels shape:", tuple(result["labels"].shape))
    return result

teacher_train = generate_teacher_logits(
    train_dataset, "train", batch_size=8
teacher_dev = generate_teacher_logits(
    dev_dataset, "dev", batch_size=8)

assert teacher_train["logits"].shape == (32305, 2)
assert teacher_dev["logits"].shape == (1988, 2)
assert torch.equal(teacher_train["labels"], train_data["labels"])
assert torch.equal(teacher_dev["labels"], dev_data["labels"])

print("\nTeacher-logit generation and alignment checks passed.")

Teacher train:   0%|          | 0/4039 [00:00<?, ?it/s]


Saved: /kaggle/working/cjpe_m1_teacher_full/train_teacher_logits.pt
Logits shape: (32305, 2)
Labels shape: (32305,)


Teacher dev:   0%|          | 0/249 [00:00<?, ?it/s]


Saved: /kaggle/working/cjpe_m1_teacher_full/dev_teacher_logits.pt
Logits shape: (1988, 2)
Labels shape: (1988,)

Teacher-logit generation and alignment checks passed.


In [17]:
import os
import torch

paths = {
    "M4 model source": "/kaggle/working/train_m4.py",
    "M4 checkpoint": "/kaggle/working/m4_checkpoints/best.pt",
    "M4 tokenized train": "/kaggle/working/cjpe_m4_long/train.pt",
    "M4 tokenized dev": "/kaggle/working/cjpe_m4_long/dev.pt",
    "Teacher train logits": "/kaggle/working/cjpe_m1_teacher_full/train_teacher_logits.pt",
    "Teacher dev logits": "/kaggle/working/cjpe_m1_teacher_full/dev_teacher_logits.pt",
}

for name, path in paths.items():
    exists = os.path.exists(path)
    print(f"{name}: {'FOUND' if exists else 'MISSING'}")
    if exists:
        print(f"  Size: {os.path.getsize(path) / (1024**3):.3f} GB")

# Inspect M4 checkpoint metadata without loading onto GPU
m4_path = paths["M4 checkpoint"]
if os.path.exists(m4_path):
    ckpt = torch.load(m4_path, map_location="cpu", weights_only=False)
    print("\nM4 checkpoint keys:", list(ckpt.keys())[:20])
    if isinstance(ckpt, dict):
        for key in ["epoch", "step", "best_val_loss", "config"]:
            if key in ckpt:
                print(f"{key}:", ckpt[key])

M4 model source: FOUND
  Size: 0.000 GB
M4 checkpoint: FOUND
  Size: 1.518 GB
M4 tokenized train: FOUND
  Size: 0.436 GB
M4 tokenized dev: FOUND
  Size: 0.030 GB
Teacher train logits: FOUND
  Size: 0.000 GB
Teacher dev logits: FOUND
  Size: 0.000 GB

M4 checkpoint keys: ['model', 'optimizer', 'scaler', 'step', 'tokens_seen', 'best_val_loss', 'config']
step: 12213
best_val_loss: 1.5915145874023438
config: {'vocab_size': 32000, 'block_size': 2048, 'd_model': 768, 'n_layers': 16, 'n_heads': 12, 'q_lora_rank': 384, 'kv_lora_rank': 192, 'qk_nope_head_dim': 48, 'qk_rope_head_dim': 24, 'v_head_dim': 64, 'ffn_dim': 2304, 'dropout': 0.05}


In [6]:
%%writefile /kaggle/working/train_m4_cjpe_distill.py
import os
import sys
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.distributed as dist
import importlib.util

from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import Dataset, DataLoader, DistributedSampler
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix
from tqdm import tqdm

SEED = 42
BATCH_SIZE = 1
ACCUM_STEPS = 8
EPOCHS = 1
LR = 1e-4
WEIGHT_DECAY = 0.01
TEMPERATURE = 2.0
KD_WEIGHT = 0.30

M4_DIR = "/kaggle/working/cjpe_m4_long"
TEACHER_DIR = "/kaggle/working/cjpe_m1_teacher_full"
M4_SOURCE = "/kaggle/working/train_m4.py"
M4_CHECKPOINT = "/kaggle/working/m4_checkpoints/best.pt"

OUT_DIR = "/kaggle/working/cjpe_m4_distill"
BEST_PATH = os.path.join(OUT_DIR, "best_m4_cjpe.pt")
LAST_PATH = os.path.join(OUT_DIR, "last_m4_cjpe.pt")
os.makedirs(OUT_DIR, exist_ok=True)


def setup():
    dist.init_process_group(backend="nccl")
    local_rank = int(os.environ["LOCAL_RANK"])
    rank = dist.get_rank()
    world_size = dist.get_world_size()
    torch.cuda.set_device(local_rank)
    return local_rank, rank, world_size


def cleanup():
    if dist.is_initialized():
        dist.destroy_process_group()


class CJPEData(Dataset):
    def __init__(self, data, teacher):
        self.ids = data["ids"]
        self.labels = data["labels"]
        self.teacher_logits = teacher["logits"]

        assert len(self.ids) == len(self.labels)
        assert len(self.labels) == len(self.teacher_logits)
        assert torch.equal(self.labels, teacher["labels"])

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return idx, self.ids[idx], self.labels[idx], self.teacher_logits[idx]


class M4CJPEClassifier(nn.Module):
    def __init__(self, backbone):
        super().__init__()
        self.backbone = backbone

        for p in self.backbone.parameters():
            p.requires_grad = False

        for block in self.backbone.blocks[-2:]:
            for p in block.parameters():
                p.requires_grad = True

        for p in self.backbone.norm.parameters():
            p.requires_grad = True

        self.classifier = nn.Sequential(
            nn.Dropout(0.2),
            nn.Linear(768, 2)
        )

    def forward(self, input_ids):
        x = self.backbone.token_embedding(input_ids)
        x = self.backbone.dropout(x)

        for block in self.backbone.blocks:
            x = block(x)

        x = self.backbone.norm(x)

        # The stored M4 examples are fixed-length 2048-token sequences.
        pooled = x[:, -1, :]
        return self.classifier(pooled)


def evaluate(model, loader, device, rank, world_size):
    model.eval()
    local_results = []

    with torch.no_grad():
        for indices, ids, labels, teacher_logits in loader:
            ids = ids.to(device, non_blocking=True)

            with torch.autocast("cuda", dtype=torch.float16):
                logits = model(ids)

            probs = torch.softmax(logits.float(), dim=1)[:, 1].cpu()

            for j in range(len(indices)):
                local_results.append((
                    int(indices[j]),
                    int(labels[j]),
                    float(probs[j])
                ))

    gathered = [None for _ in range(world_size)]
    dist.all_gather_object(gathered, local_results)

    if rank != 0:
        return None

    by_index = {}
    for shard in gathered:
        for idx, label, prob in shard:
            if idx not in by_index:
                by_index[idx] = (label, prob)

    expected_n = len(loader.dataset)
    if len(by_index) != expected_n:
        raise RuntimeError(
            f"Validation found {len(by_index)} unique examples; "
            f"expected {expected_n}."
        )

    y_true = [by_index[i][0] for i in range(expected_n)]
    y_prob = [by_index[i][1] for i in range(expected_n)]
    y_pred = [int(p >= 0.5) for p in y_prob]

    return {
        "macro_f1": f1_score(y_true, y_pred, average="macro"),
        "accuracy": accuracy_score(y_true, y_pred),
        "confusion_matrix": confusion_matrix(
            y_true, y_pred, labels=[0, 1]
        ).tolist()
    }


def main():
    local_rank, rank, world_size = setup()
    device = torch.device(f"cuda:{local_rank}")

    try:
        random.seed(SEED + rank)
        np.random.seed(SEED + rank)
        torch.manual_seed(SEED + rank)

        if rank == 0:
            print("Loading M4 train data...", flush=True)
        train_data = torch.load(
            os.path.join(M4_DIR, "train.pt"),
            map_location="cpu",
            weights_only=False
        )

        if rank == 0:
            print("Loading M4 dev data...", flush=True)
        dev_data = torch.load(
            os.path.join(M4_DIR, "dev.pt"),
            map_location="cpu",
            weights_only=False
        )

        if rank == 0:
            print("Loading teacher logits...", flush=True)
        teacher_train = torch.load(
            os.path.join(TEACHER_DIR, "train_teacher_logits.pt"),
            map_location="cpu",
            weights_only=False
        )
        teacher_dev = torch.load(
            os.path.join(TEACHER_DIR, "dev_teacher_logits.pt"),
            map_location="cpu",
            weights_only=False
        )

        train_ds = CJPEData(train_data, teacher_train)
        dev_ds = CJPEData(dev_data, teacher_dev)

        del train_data, dev_data, teacher_train, teacher_dev

        train_sampler = DistributedSampler(
            train_ds,
            num_replicas=world_size,
            rank=rank,
            shuffle=True,
            seed=SEED,
            drop_last=False
        )
        dev_sampler = DistributedSampler(
            dev_ds,
            num_replicas=world_size,
            rank=rank,
            shuffle=False,
            drop_last=False
        )

        train_loader = DataLoader(
            train_ds,
            batch_size=BATCH_SIZE,
            sampler=train_sampler,
            num_workers=0,
            pin_memory=True
        )
        dev_loader = DataLoader(
            dev_ds,
            batch_size=BATCH_SIZE,
            sampler=dev_sampler,
            num_workers=0,
            pin_memory=True
        )

        if rank == 0:
            print("Loading M4 architecture...", flush=True)
        spec = importlib.util.spec_from_file_location("m4_source", M4_SOURCE)
        m4_source = importlib.util.module_from_spec(spec)
        sys.modules["m4_source"] = m4_source
        spec.loader.exec_module(m4_source)

        if rank == 0:
            print("Loading pretrained M4 checkpoint...", flush=True)
        checkpoint = torch.load(
            M4_CHECKPOINT,
            map_location="cpu",
            weights_only=False
        )
        backbone = m4_source.M4Model()
        backbone.load_state_dict(checkpoint["model"], strict=True)
        del checkpoint
        import gc
        gc.collect()

        model = M4CJPEClassifier(backbone).to(device)
        model = DDP(
            model,
            device_ids=[local_rank],
            output_device=local_rank
        )

        counts = torch.bincount(
            train_ds.labels, minlength=2
        ).float()
        class_weights = counts.sum() / (2.0 * counts)

        ce_criterion = nn.CrossEntropyLoss(
            weight=class_weights.to(device)
        )

        optimizer = torch.optim.AdamW(
            [p for p in model.parameters() if p.requires_grad],
            lr=LR,
            weight_decay=WEIGHT_DECAY
        )
        scaler = torch.amp.GradScaler("cuda")
        best_f1 = -1.0

        if rank == 0:
            trainable = sum(
                p.numel() for p in model.parameters()
                if p.requires_grad
            )
            print("World size:", world_size, flush=True)
            print("Train examples:", len(train_ds), flush=True)
            print("Dev examples:", len(dev_ds), flush=True)
            print("Trainable parameters:", f"{trainable:,}", flush=True)
            print("Starting training.", flush=True)

        for epoch in range(1, EPOCHS + 1):
            model.train()
            train_sampler.set_epoch(epoch)
            optimizer.zero_grad(set_to_none=True)

            loss_sum = 0.0
            seen = 0

            iterator = train_loader
            if rank == 0:
                iterator = tqdm(
                    train_loader,
                    desc=f"Epoch {epoch}/{EPOCHS}"
                )

            for step, (indices, ids, labels, teacher_logits) in enumerate(iterator):
                ids = ids.to(device, non_blocking=True)
                labels = labels.to(device, non_blocking=True)
                teacher_logits = teacher_logits.to(
                    device, non_blocking=True
                )

                with torch.autocast("cuda", dtype=torch.float16):
                    student_logits = model(ids)
                    ce_loss = ce_criterion(student_logits, labels)

                    student_log_probs = F.log_softmax(
                        student_logits.float() / TEMPERATURE,
                        dim=1
                    )
                    teacher_probs = F.softmax(
                        teacher_logits.float() / TEMPERATURE,
                        dim=1
                    )

                    kd_loss = F.kl_div(
                        student_log_probs,
                        teacher_probs,
                        reduction="batchmean"
                    ) * (TEMPERATURE ** 2)

                    combined_loss = (
                        (1.0 - KD_WEIGHT) * ce_loss
                        + KD_WEIGHT * kd_loss
                    )
                    loss = combined_loss / ACCUM_STEPS

                scaler.scale(loss).backward()

                should_step = (
                    (step + 1) % ACCUM_STEPS == 0
                    or (step + 1) == len(train_loader)
                )

                if should_step:
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(
                        [p for p in model.parameters() if p.requires_grad],
                        1.0
                    )
                    scaler.step(optimizer)
                    scaler.update()
                    optimizer.zero_grad(set_to_none=True)

                loss_sum += combined_loss.detach().item() * len(labels)
                seen += len(labels)

            stats = torch.tensor(
                [loss_sum, seen], dtype=torch.float64, device=device
            )
            dist.all_reduce(stats, op=dist.ReduceOp.SUM)
            train_loss = (stats[0] / stats[1].clamp(min=1)).item()

            metrics = evaluate(
                model, dev_loader, device, rank, world_size
            )

            if rank == 0:
                print(f"\nEpoch {epoch}: train_loss={train_loss:.4f}")
                print("Dev macro-F1:", round(metrics["macro_f1"], 4))
                print("Dev accuracy:", round(metrics["accuracy"], 4))
                print("Confusion matrix:", metrics["confusion_matrix"])

                state = {
                    "model": model.module.state_dict(),
                    "epoch": epoch,
                    "dev_macro_f1": metrics["macro_f1"],
                    "dev_accuracy": metrics["accuracy"],
                    "confusion_matrix": metrics["confusion_matrix"],
                    "temperature": TEMPERATURE,
                    "kd_weight": KD_WEIGHT,
                    "train_loss": train_loss
                }

                torch.save(state, LAST_PATH)

                if metrics["macro_f1"] > best_f1:
                    best_f1 = metrics["macro_f1"]
                    torch.save(state, BEST_PATH)
                    print("Saved best checkpoint:", BEST_PATH)

            dist.barrier()

        if rank == 0:
            print("\nTraining complete.")
            print("Best dev macro-F1:", round(best_f1, 4))
            print("Best checkpoint:", BEST_PATH)

    finally:
        cleanup()


if __name__ == "__main__":
    main()

Overwriting /kaggle/working/train_m4_cjpe_distill.py


In [7]:
!PYTHONUNBUFFERED=1 torchrun --standalone --nproc_per_node=2 /kaggle/working/train_m4_cjpe_distill.py

W1026 17:32:07.275000 471 torch/distributed/run.py:851] 
W1026 17:32:07.275000 471 torch/distributed/run.py:851] *****************************************
W1026 17:32:07.275000 471 torch/distributed/run.py:851] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W1026 17:32:07.275000 471 torch/distributed/run.py:851] *****************************************
Loading M4 train data...
Loading M4 dev data...
Loading teacher logits...
Loading M4 architecture...
Loading pretrained M4 checkpoint...
World size: 2
Train examples: 32305
Dev examples: 1988
Trainable parameters: 13,904,258
Starting training.
Epoch 1/1: 100%|██████████████████████████| 16153/16153 [26:42<00:00, 10.08it/s]

Epoch 1: train_loss=0.5149
Dev macro-F1: 0.3949
Dev accuracy: 0.5211
Confusion matrix: [[972, 22], [930, 64]]
Saved best checkpoint: /kaggle/working/cjpe_m

In [8]:
import os
import torch

path = "/kaggle/working/cjpe_m4_distill/best_m4_cjpe.pt"

print("Checkpoint exists:", os.path.exists(path))

if os.path.exists(path):
    ckpt = torch.load(path, map_location="cpu", weights_only=False)
    print("Epoch:", ckpt["epoch"])
    print("Dev macro-F1:", ckpt["dev_macro_f1"])
    print("Dev accuracy:", ckpt["dev_accuracy"])
    print("Confusion matrix:", ckpt["confusion_matrix"])
    print("Training loss:", ckpt["train_loss"])

Checkpoint exists: True
Epoch: 1
Dev macro-F1: 0.3948946183752814
Dev accuracy: 0.5211267605633803
Confusion matrix: [[972, 22], [930, 64]]
Training loss: 0.5149353479002854


In [11]:

import os
import sys
import json
import pickle
import zipfile
import torch
import torch.nn as nn
import importlib.util
from tqdm.auto import tqdm
from huggingface_hub import hf_hub_download

# --------------------------------------------------
# 1. Paths and configuration
# --------------------------------------------------
TEST_FILE = None
TOKENIZER_PATH = "/kaggle/input/datasets/spikie/capstone-split/legal_bpe_32k.pkl"
M4_SOURCE = "/kaggle/working/train_m4.py"
CHECKPOINT_PATH = "/kaggle/working/cjpe_m4_distill/best_m4_cjpe.pt"

OUT_DIR = "/kaggle/working/cjpe_m4_submission"
os.makedirs(OUT_DIR, exist_ok=True)

PREDICTION_PATH = os.path.join(OUT_DIR, "prediction.json")
ZIP_PATH = "/kaggle/working/submission.zip"

MAX_LEN = 2048
BATCH_SIZE = 1
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

print("Device:", DEVICE, flush=True)

# --------------------------------------------------
# 2. Download official Phase 4 test set
# --------------------------------------------------
if TEST_FILE is None:
    print("Downloading official test.json...", flush=True)
    TEST_FILE = hf_hub_download(
        repo_id="Exploration-Lab/CS-781-Capstone",
        repo_type="dataset",
        filename="Phase_4_CJPE/Test/test.json",
        token=os.environ.get("HF_TOKEN")
    )

with open(TEST_FILE, "r", encoding="utf-8") as f:
    test_data = json.load(f)

if isinstance(test_data, dict):
    if "data" in test_data:
        test_data = test_data["data"]
    else:
        raise ValueError(
            "Unexpected test JSON structure. Top-level keys: "
            + str(list(test_data.keys())[:20])
        )

assert isinstance(test_data, list), "Expected a list of test records."
assert all("id" in r and "text" in r for r in test_data), (
    "Test records must contain id and text fields."
)

print("Test examples:", len(test_data), flush=True)
print("Example fields:", list(test_data[0].keys()), flush=True)

# --------------------------------------------------
# 3. Load the same tokenizer used for M4 preprocessing
# --------------------------------------------------
print("Loading tokenizer...", flush=True)

with open(TOKENIZER_PATH, "rb") as f:
    tokenizer = pickle.load(f)

def encode_text(text):
    if hasattr(tokenizer, "encode_text"):
        ids = tokenizer.encode_text(text, add_eos=True)
    elif hasattr(tokenizer, "encode"):
        try:
            ids = tokenizer.encode(text, add_eos=True)
        except TypeError:
            ids = tokenizer.encode(text)
    else:
        raise AttributeError(
            "Tokenizer has no encode_text() or encode() method. "
            "Inspect the tokenizer object and use the exact M4 preprocessing method."
        )

    if isinstance(ids, dict):
        ids = ids["input_ids"]

    if isinstance(ids, torch.Tensor):
        ids = ids.tolist()

    # Handle tokenizers that return a single nested sequence.
    if ids and isinstance(ids[0], list):
        ids = ids[0]

    ids = [int(x) for x in ids]
    ids = ids[:MAX_LEN]

    # M4 sequences use token ID 0 for EOS/padding.
    if len(ids) < MAX_LEN:
        ids.extend([0] * (MAX_LEN - len(ids)))

    return ids

# --------------------------------------------------
# 4. Rebuild the exact saved classifier architecture
# --------------------------------------------------
print("Loading M4 architecture...", flush=True)

spec = importlib.util.spec_from_file_location("m4_source", M4_SOURCE)
m4_source = importlib.util.module_from_spec(spec)
sys.modules["m4_source"] = m4_source
spec.loader.exec_module(m4_source)

class M4CJPEClassifier(nn.Module):
    def __init__(self, backbone):
        super().__init__()
        self.backbone = backbone
        self.classifier = nn.Sequential(
            nn.Dropout(0.2),
            nn.Linear(768, 2)
        )

    def forward(self, input_ids):
        x = self.backbone.token_embedding(input_ids)
        x = self.backbone.dropout(x)

        for block in self.backbone.blocks:
            x = block(x)

        x = self.backbone.norm(x)
        pooled = x[:, -1, :]
        return self.classifier(pooled)

print("Loading trained M4 classifier checkpoint...", flush=True)

checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=False
)

backbone = m4_source.M4Model()
model = M4CJPEClassifier(backbone)
model.load_state_dict(checkpoint["model"], strict=True)
model.to(DEVICE)
model.eval()

print("Loaded checkpoint from epoch:", checkpoint["epoch"], flush=True)
print("Development macro-F1:", checkpoint["dev_macro_f1"], flush=True)

del checkpoint

# --------------------------------------------------
# 5. Generate predictions
# --------------------------------------------------
predictions = []

with torch.inference_mode():
    for start in tqdm(range(0, len(test_data), BATCH_SIZE),
                      desc="Predicting test cases"):
        batch_records = test_data[start:start + BATCH_SIZE]

        batch_ids = [
            encode_text(record["text"])
            for record in batch_records
        ]

        input_ids = torch.tensor(
            batch_ids, dtype=torch.long, device=DEVICE
        )

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=(DEVICE.type == "cuda")
        ):
            logits = model(input_ids)

        predicted_labels = logits.argmax(dim=1).cpu().tolist()

        for record, pred in zip(batch_records, predicted_labels):
            predictions.append({
                "id": record["id"],
                "pred": int(pred)
            })

assert len(predictions) == len(test_data)
assert len({p["id"] for p in predictions}) == len(predictions)
assert all(p["pred"] in (0, 1) for p in predictions)

# --------------------------------------------------
# 6. Write prediction.json
# --------------------------------------------------
with open(PREDICTION_PATH, "w", encoding="utf-8") as f:
    json.dump(predictions, f, ensure_ascii=False, indent=2)

print("\nSaved:", PREDICTION_PATH)
print("Total predictions:", len(predictions))
print("Predicted label counts:", {
    label: sum(p["pred"] == label for p in predictions)
    for label in (0, 1)
})
print("First three predictions:", predictions[:3])

# --------------------------------------------------
# 7. Create submission ZIP
# --------------------------------------------------
with zipfile.ZipFile(
    ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED
) as zf:
    zf.write(PREDICTION_PATH, arcname="prediction.json")

print("\nSubmission ZIP created:", ZIP_PATH)
print("ZIP contents:", zipfile.ZipFile(ZIP_PATH).namelist())
print("ZIP size: {:.2f} MB".format(os.path.getsize(ZIP_PATH) / (1024**2)))


Device: cuda:0
Test examples: 1517
Example fields: ['id', 'text', 'label', 'expert_1', 'expert_2', 'expert_3', 'expert_4', 'expert_5']
Loading tokenizer...
Loading M4 architecture...
Loading trained M4 classifier checkpoint...
Loaded checkpoint from epoch: 1
Development macro-F1: 0.3948946183752814


Predicting test cases:   0%|          | 0/1517 [00:00<?, ?it/s]


Saved: /kaggle/working/cjpe_m4_submission/prediction.json
Total predictions: 1517
Predicted label counts: {0: 1468, 1: 49}
First three predictions: [{'id': '1986_397', 'pred': 0}, {'id': '1977_183', 'pred': 0}, {'id': '1993_98', 'pred': 0}]

Submission ZIP created: /kaggle/working/submission.zip
ZIP contents: ['prediction.json']
ZIP size: 0.01 MB


Phase 5

In [7]:

import os
import json
from huggingface_hub import hf_hub_download

REPO_ID = "Exploration-Lab/CS-781-Capstone"
DATA_DIR = "/kaggle/working/lsi_m4"
os.makedirs(DATA_DIR, exist_ok=True)

def load_lsi_split(filename):
    path = hf_hub_download(
        repo_id=REPO_ID,
        repo_type="dataset",
        filename="Phase_5_LSI/" + filename,
        token=os.environ.get("HF_TOKEN"),
        local_dir=DATA_DIR
    )
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

train_data = load_lsi_split("Train/train.json")
dev_data = load_lsi_split("Train/dev.json")
test_data = load_lsi_split("Test/test.json")

print("Train:", len(train_data))
print("Dev:", len(dev_data))
print("Test:", len(test_data))
print("Example fields:", train_data[0].keys())
print("Example label count:", len(train_data[0]["labels"]))

Phase_5_LSI/Train/train.json: reconstructing file:   0%|          |  0.00B /  299MB            

Phase_5_LSI/Train/train.json: downloading bytes:           |  0.00B            

Phase_5_LSI/Train/dev.json: reconstructing file:   0%|          |  0.00B / 79.4MB            

Phase_5_LSI/Train/dev.json: downloading bytes:           |  0.00B            

Phase_5_LSI/Test/test.json: reconstructing file:   0%|          |  0.00B / 98.1MB            

Phase_5_LSI/Test/test.json: downloading bytes:           |  0.00B            

Train: 42750
Dev: 10181
Test: 13019
Example fields: dict_keys(['id', 'text', 'labels'])
Example label count: 3


In [5]:

import os
import pickle

TOKENIZER_PATH = (
    "/kaggle/input/datasets/spikie/capstone-split/"
    "legal_bpe_32k.pkl"
)

assert os.path.isfile(TOKENIZER_PATH), (
    "Shared tokenizer file not found. Check the attached Kaggle dataset."
)

with open(TOKENIZER_PATH, "rb") as f:
    tok = pickle.load(f)

print("Loaded shared tokenizer:", type(tok).__name__)
print("Vocabulary size:", tok.vocab_size)
print("Test encoding:", tok.encode_text("A legal case", add_eos=True))


Loaded shared tokenizer: ByteLevelBPE
Vocabulary size: 32000
Test encoding: [66, 33, 1320, 33, 422, 0]


In [8]:

import torch
from tqdm.auto import tqdm

WORK = "/kaggle/working/lsi_m4"
CHUNK_SIZE = 1024
OVERLAP = 128
STRIDE = CHUNK_SIZE - OVERLAP
LABEL_IDS = sorted(set(
    int(label)
    for row in train_data
    for label in row["labels"]
))
label_to_idx = {label: i for i, label in enumerate(LABEL_IDS)}

def encode_document(text):
    token_ids = tok.encode_text(text, add_eos=True)
    if len(token_ids) == 0:
        token_ids = [0]

    chunks = []
    for start in range(0, len(token_ids), STRIDE):
        chunk = token_ids[start:start + CHUNK_SIZE]
        if len(chunk) > 0:
            chunks.append(torch.tensor(chunk, dtype=torch.int32))
        if start + CHUNK_SIZE >= len(token_ids):
            break

    return chunks

def encode_split(rows, has_labels, name):
    encoded = []
    for row in tqdm(rows, desc="Tokenizing " + name):
        item = {
            "id": str(row["id"]),
            "chunks": encode_document(row["text"])
        }
        if has_labels:
            item["labels"] = [
                int(label) for label in row["labels"]
            ]
        encoded.append(item)

    output_path = os.path.join(WORK, name + ".pt")
    torch.save(encoded, output_path)
    print("Saved:", output_path)
    return encoded

train_encoded = encode_split(train_data, True, "train")
dev_encoded = encode_split(dev_data, True, "dev")
test_encoded = encode_split(test_data, False, "test")

torch.save(LABEL_IDS, os.path.join(WORK, "label_ids.pt"))

print("Number of labels:", len(LABEL_IDS))
print("Chunks in first training document:", len(train_encoded[0]["chunks"]))
print("Maximum chunks in a training document:",
      max(len(row["chunks"]) for row in train_encoded))


Tokenizing train:   0%|          | 0/42750 [00:00<?, ?it/s]

Saved: /kaggle/working/lsi_m4/train.pt


Tokenizing dev:   0%|          | 0/10181 [00:00<?, ?it/s]

Saved: /kaggle/working/lsi_m4/dev.pt


Tokenizing test:   0%|          | 0/13019 [00:00<?, ?it/s]

Saved: /kaggle/working/lsi_m4/test.pt
Number of labels: 100
Chunks in first training document: 4
Maximum chunks in a training document: 232


Phase 6

In [ ]:
import os
import json
import random
from huggingface_hub import hf_hub_download

HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"

TRAIN_FILE = "Phase_6_Summ/Train/train.json"
TEST_FILE = "Phase_6_Summ/Test/test.json"

SEED = 42
random.seed(SEED)


train_path = hf_hub_download(
    repo_id=HF_DATASET_REPO,
    filename=TRAIN_FILE,
    repo_type="dataset",
    token=HF_TOKEN
)

print("Training dataset:",train_path)

test_path = hf_hub_download(
    repo_id=HF_DATASET_REPO,
    filename=TEST_FILE,
    repo_type="dataset",
    token=HF_TOKEN
)